# 01 Data Wrangling
**Capstone:** Pengelolaan Persediaan dan Pemantauan Penjualan Bulanan pada Toko Pertanian Multikanal (Asah by Dicoding)

**Periode data:** Nov 2023 sampai Okt 2026 (offline), Feb 2025 sampai Okt 2026 (Shopee, Tokopedia), Jun 2025 sampai Okt 2026 (TikTok Shop)

### Alur notebook
1. **Gathering Data**: membaca semua file mentah apa adanya (8 tabel: master, offline, online, stok masuk/keluar, pemecahan, rakit, opname, metrik marketplace).
2. **Assessing Data**: memeriksa **setiap tabel** dan mencatat masalah yang ditemukan. Belum ada data yang diubah.
3. **Cleaning Data**: memperbaiki masalah yang sudah ditemukan, **tabel demi tabel**. Setiap langkah cleaning menyebut masalah mana yang sedang ditangani.
4. **Membangun tabel akhir** dan **menyimpan** hasilnya.

### Aturan kerja
1. File di `data/raw/` **tidak diubah**. Hasil tiap tabel disimpan ke `data/interim/`, tabel akhir ke `data/processed/`.
2. **Pemetaan SKU strict**: teks yang cocok dengan master atau dengan mapping tersimpan → masuk; selain itu → **ditolak dan dilaporkan** (berkas, baris, teks). Tidak pernah dibuat SKU baru. Aturan yang sama dipakai saat admin meng-import file.
3. **Tidak ada data yang hilang diam-diam.** Setiap perubahan dicatat di log (berkas, baris Excel, masalah, tindakan, nilai lama, nilai baru). Baris yang tidak bisa dipakai ditandai di kolom `alasan_ditolak` dan disimpan di laporan `*_ditolak.csv`.
4. Pendapatan **online = NETTO** (dihitung ulang), **offline = Subtotal**.
5. Stok dua tingkat: SKU jual dan SKU induk. Empat SKU induk 50 kg (`UREA50KG`, `KCL50KG`, `DLMT50KG`, `FRTPHS50KG`) tidak dijual dan stoknya dicatat dalam kg. Pemecahan tanpa susut. Paket dirakit di muka dan punya stok sendiri (`ASSEMBLE_OUT` dan `ASSEMBLE_IN`).
6. Penjualan online baru mengurangi stok saat laporan bulanan diimpor. `data_complete_until` dicatat per kanal.

## 1. Import Library dan Pengaturan Folder

In [1]:
import re
import difflib
from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import load_workbook

ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
print("pandas", pd.__version__)

pandas 3.0.6


## 2. Fungsi Bantu Umum
Di sini hanya dibuat fungsi yang pasti dibutuhkan sejak awal: memeriksa sel kosong dan mencatat log.

### 2.1 Memeriksa sel kosong
Sel Excel yang kosong bisa terbaca sebagai `None`, `NaN`, `NaT`, atau teks berisi spasi. Fungsi ini menyamakan semuanya.

In [2]:
def kosong(nilai):
    # True jika sel dianggap kosong: None, NaN, NaT, atau teks berisi spasi saja.
    if nilai is None or nilai is pd.NaT:
        return True
    if isinstance(nilai, float) and np.isnan(nilai):
        return True
    return isinstance(nilai, str) and nilai.strip() == ""


print(kosong(None), kosong(np.nan), kosong("   "), kosong("KCL"), kosong(0))

True True True False False


### 2.2 Log perubahan dan penandaan baris yang ditolak
- `catat(...)` menyimpan setiap perubahan: berkas, baris Excel, kolom, masalah, tindakan, nilai lama, nilai baru.
- `tolak(...)` menandai baris yang **tidak bisa dipakai** di kolom `alasan_ditolak`. Di akhir cleaning tiap tabel, baris ini dipisahkan ke laporan `*_ditolak.csv` (berkas, baris, teks, alasan) supaya bisa diperbaiki di file sumber.

In [3]:
daftar_log = []      # setiap tabel menambahkan catatannya ke list ini


def catat(tahap, baris, masalah, tindakan, kolom=None, lama=None, baru=None):
    # Catat satu baris log untuk setiap baris di DataFrame `baris`.
    # `lama` dan `baru` adalah NAMA KOLOM yang nilainya ingin disimpan sebagai nilai lama/baru.
    if len(baris) == 0:
        return
    daftar_log.append(pd.DataFrame({
        "tahap": tahap,
        "berkas": baris["berkas"].values,
        "baris_excel": baris["baris_excel"].values,
        "kolom": kolom,
        "masalah": masalah,
        "tindakan": tindakan,
        "nilai_lama": baris[lama].astype(str).values if lama else None,
        "nilai_baru": baris[baru].astype(str).values if baru else None,
    }))


def tolak(df, kondisi, alasan):
    # Tandai baris yang memenuhi `kondisi` sebagai ditolak. Alasan pertama yang tercatat tidak ditimpa.
    baru = kondisi & df["alasan_ditolak"].isna()
    df.loc[baru, "alasan_ditolak"] = alasan
    return baru


def log_tahap(tahap):
    # Gabungkan semua catatan log milik satu tahap menjadi satu DataFrame.
    semua = pd.concat(daftar_log, ignore_index=True)
    return semua[semua["tahap"] == tahap]

## 3. Gathering Data

### 3.1 Daftar berkas mentah

In [4]:
berkas_offline = sorted((RAW / "rekap_offline").glob("*.xlsx"))
berkas_online = sorted((RAW / "rekap_online").glob("rekap_online_*.xlsx"))
berkas_opname = sorted((RAW / "stok").glob("stok_opname_*.xlsx"))
print("Master produk :", sorted(p.name for p in (RAW / "master_produk").iterdir()))
print("Rekap offline :", len(berkas_offline), "berkas,", berkas_offline[0].name, "s.d.", berkas_offline[-1].name)
print("Rekap online  :", len(berkas_online), "berkas")
print("Stok          :", sorted(p.name for p in (RAW / "stok").glob("stok_[!o]*.xlsx")))
print("Stok opname   :", len(berkas_opname), "berkas")
print("Metrik marketplace :", [p.name for p in (RAW / "rekap_online").glob("*.xlsx") if p not in berkas_online])

Master produk : ['master_induk_konversi.csv', 'master_kategori.csv', 'master_paket_komposisi.csv', 'master_supplier.csv', 'penetapan_harga_dasar.xlsx']
Rekap offline : 36 berkas, rekap_penjualan_offline_2023-11.xlsx s.d. rekap_penjualan_offline_2026-10.xlsx
Rekap online  : 59 berkas
Stok          : ['stok_masuk_keluar.xlsx', 'stok_pemecahan.xlsx', 'stok_rakit_paket.xlsx']
Stok opname   : 13 berkas
Metrik marketplace : ['metrik_marketplace_OPSIONAL.xlsx']


### 3.2 Master produk (5 file)

In [5]:
MASTER = RAW / "master_produk"
master_kategori = pd.read_csv(MASTER / "master_kategori.csv")
master_konversi = pd.read_csv(MASTER / "master_induk_konversi.csv")
master_harga = pd.read_excel(MASTER / "penetapan_harga_dasar.xlsx")
master_komposisi = pd.read_csv(MASTER / "master_paket_komposisi.csv")
master_supplier = pd.read_csv(MASTER / "master_supplier.csv")
for nama, df in [("master_kategori", master_kategori), ("master_induk_konversi", master_konversi),
                 ("penetapan_harga_dasar", master_harga), ("master_paket_komposisi", master_komposisi),
                 ("master_supplier", master_supplier)]:
    print(f"{nama:24s}: {df.shape[0]:4d} baris, kolom = {df.columns.tolist()}")

master_kategori         :  186 baris, kolom = ['sku', 'nama', 'kategori', 'label']
master_induk_konversi   :  190 baris, kolom = ['sku', 'product_id', 'parent_id', 'parent_name', 'base_unit', 'faktor_per_unit_jual', 'isi_per_pembelian', 'satuan_beli', 'is_bundle', 'label']
penetapan_harga_dasar   :  186 baris, kolom = ['KODE SKU', 'NO.', 'NAMA', 'HARGA PRODUKSI/KULAKAN', 'BIAYA PENGEMASAN', 'BIAYA PENYIMPANAN', 'HPP', 'PROFIT KOTOR %', 'PROFIT KOTOR Rp', 'HARGA JUAL BRUTO', 'DISKON PEMBELI %', 'DISKON PEMBELI Rp', 'HARGA JUAL NETTO', 'BIAYA ADMIN MP %', 'BIAYA ADMIN MP Rp', 'BIAYA PROMOSI %', 'BIAYA PROMOSI Rp', 'PROFIT BERSIH', 'PERSEN PROFIT THD HPP', 'PERSEN PROFIT THD HARGA JUAL', 'HARGA JUAL ONLINE', 'HARGA JUAL OFFLINE', 'SUMBER DATA']
master_paket_komposisi  :   13 baris, kolom = ['sku_paket', 'sku_komponen', 'jumlah_per_paket_dalam_satuan_jual_komponen', 'label']
master_supplier         :    7 baris, kolom = ['supplier_code', 'nama_dummy', 'kelompok_produk', 'lead_time_hari_asu

### 3.3 Rekap penjualan offline (36 berkas bulanan)
Isi dari setiap berkas **dilihat dulu** tanpa menganggap baris mana pun sebagai header. Yang dilihat: isi 3 baris pertama setiap berkas.

In [6]:
intip = []
for path in berkas_offline:
    mentah = pd.read_excel(path, header=None, nrows=3, dtype=object)
    baris = {f"baris_{i + 1}": " | ".join(str(v) for v in mentah.iloc[i] if not kosong(v)) for i in range(len(mentah))}
    intip.append({"berkas": path.name, **baris})
intip = pd.DataFrame(intip)
print("Isi baris 1 (judul laporan dikelompokkan):")
print(intip["baris_1"].mask(intip["baris_1"].str.startswith("REKAP"), "REKAP PENJUALAN TOKO <bulan>").value_counts().to_string())
intip.head(4)

Isi baris 1 (judul laporan dikelompokkan):
baris_1
REKAP PENJUALAN TOKO <bulan>                                                                   17
Tanggal | No. Nota | Nama Produk di Nota | Qty | Harga Satuan | Diskon | Subtotal | Catatan    10
Tgl | No Nota | Nama Barang | Jml | Harga | Disc | Total | Ket                                  9


,berkas,baris_1,baris_2,baris_3
0,rekap_penjualan_offline_2023-11.xlsx,Tgl | No Nota | Nama Barang | Jml | Harga | Disc | Total...,2023-11-01 00:00:00 | 2311-0001 | pupuk amazing grow per...,2023-11-01 00:00:00 | 2311-0002 | furadan 3gr | 1 | 2900...
1,rekap_penjualan_offline_2023-12.xlsx,REKAP PENJUALAN TOKO DES 2023,,Tanggal | No. Nota | Nama Produk di Nota | Qty | Harga S...
2,rekap_penjualan_offline_2024-01.xlsx,REKAP PENJUALAN TOKO JAN 2024,,Tanggal | No. Nota | Nama Produk di Nota | Qty | Harga S...
3,rekap_penjualan_offline_2024-02.xlsx,REKAP PENJUALAN TOKO FEB 2024,,Tanggal | No. Nota | Nama Produk di Nota | Qty | Harga S...


**Hasil intip berkas offline:**
- **17 dari 36 berkas** diawali judul laporan (`REKAP PENJUALAN TOKO DES 2023`), sehingga header tabelnya turun ke **baris 3**. Di 19 berkas lain, header ada di baris 1.
- Header-nya juga **tidak sama**: ada versi `Tgl | No Nota | Nama Barang | Jml | Harga | Disc | Total | Ket` dan versi `Tanggal | No. Nota | Nama Produk di Nota | Qty | Harga Satuan | ...`.
- Karena itu file tidak bisa dibaca dengan `pd.read_excel(path)` biasa: baris header harus dicari dulu. Kata yang selalu ada di kedua versi header adalah **"Nota"** (`No Nota` / `No. Nota`), jadi kata itu dipakai sebagai penanda baris header.

Fungsi `baca_excel` membaca satu sheet **apa adanya** (`dtype=object`, supaya tidak ada nilai yang diubah pandas), mencari baris header dari sebuah kata kunci yang pasti ada di header, lalu menyimpan nomor baris Excel setiap baris supaya semua perubahan nanti bisa dilacak ke file aslinya.

In [7]:
def baca_excel(path, kata_kunci, sheet=0):
    # Baca satu sheet Excel apa adanya (dtype=object) dan deteksi baris header dari kata kunci.
    mentah = pd.read_excel(path, sheet_name=sheet, header=None, dtype=object)
    for i in range(len(mentah)):
        teks_baris = " ".join(str(v) for v in mentah.iloc[i] if not kosong(v)).lower()
        if kata_kunci.lower() in teks_baris:
            break
    df = mentah.iloc[i + 1:].copy()
    df.columns = [str(c).strip() for c in mentah.iloc[i]]
    df["baris_excel"] = df.index + 1       # nomor baris seperti yang terlihat di Excel
    df["baris_header"] = i + 1
    return df.reset_index(drop=True)

In [8]:
# Header tiap berkas: apakah nama kolomnya sama semua?
header_offline = pd.Series({p.name: " | ".join(c for c in baca_excel(p, "nota").columns if c not in ("baris_excel", "baris_header"))
                            for p in berkas_offline})
header_offline.value_counts()

Tanggal | No. Nota | Nama Produk di Nota | Qty | Harga Satuan | Diskon | Subtotal | Catatan    27
Tgl | No Nota | Nama Barang | Jml | Harga | Disc | Total | Ket                                  9
Name: count, dtype: int64

**Hasil:** ada **dua versi header** (27 dan 9 berkas). Supaya 36 berkas bisa digabung menjadi satu tabel, kedua versi dipetakan ke nama kolom yang sama (`KOLOM_OFFLINE`), misalnya `Tgl` dan `Tanggal` → `tanggal`, `Jml` dan `Qty` → `qty`, `Total` dan `Subtotal` → `subtotal`. Nama kolom asli tiap berkas tetap disimpan di `info_header_offline` untuk dicek ulang di Assessing. **Isi sel belum diubah.**

In [9]:
KOLOM_OFFLINE = {
    "tgl": "tanggal", "tanggal": "tanggal",
    "no nota": "no_nota", "no. nota": "no_nota",
    "nama barang": "nama_produk", "nama produk di nota": "nama_produk",
    "jml": "qty", "qty": "qty",
    "harga": "harga_satuan", "harga satuan": "harga_satuan",
    "disc": "diskon", "diskon": "diskon",
    "total": "subtotal", "subtotal": "subtotal",
    "ket": "catatan", "catatan": "catatan",
}

potongan, info_header_offline = [], []
for path in berkas_offline:
    df = baca_excel(path, "nota")
    info_header_offline.append({"berkas": path.name, "baris_header": df["baris_header"].iloc[0],
                                "kolom_asli": " | ".join(c for c in df.columns if c not in ("baris_excel", "baris_header"))})
    df = df.rename(columns=lambda c: KOLOM_OFFLINE.get(c.lower(), c))
    df["berkas"] = path.name
    df["bulan_berkas"] = re.search(r"\d{4}-\d{2}", path.name)[0]      # mis. '2023-11'
    potongan.append(df)
offline_raw = pd.concat(potongan, ignore_index=True)
info_header_offline = pd.DataFrame(info_header_offline)
print("Jumlah baris mentah:", len(offline_raw))
offline_raw.head()

Jumlah baris mentah: 55776


,tanggal,no_nota,nama_produk,qty,harga_satuan,diskon,subtotal,catatan,baris_excel,baris_header,berkas,bulan_berkas
0,2023-11-01 00:00:00,2311-0001,pupuk amazing grow pertumbuhan,1,19000,NaN,19000,NaN,2,1,rekap_penjualan_offline_2023-11.xlsx,2023-11
1,2023-11-01 00:00:00,2311-0002,furadan 3gr,1,29000,NaN,29000,NaN,3,1,rekap_penjualan_offline_2023-11.xlsx,2023-11
2,2023/11/01,2311-0002,Emkape Maxxi Agri,1,51000,NaN,51000,tempo,4,1,rekap_penjualan_offline_2023-11.xlsx,2023-11
3,2023-11-01 00:00:00,0002,UREA CURAH,3,12500,NaN,37500,NaN,5,1,rekap_penjualan_offline_2023-11.xlsx,2023-11
4,1-11-23,2311-0003,MKP PAK TANI,4,64500,NaN,258000,NaN,6,1,rekap_penjualan_offline_2023-11.xlsx,2023-11


### 3.4 Rekap online (59 berkas: Shopee, Tokopedia, TikTok Shop)
Berkas online juga dilihat terlebih dahulu dengan cara yang sama.

In [10]:
intip_on = []
for path in berkas_online:
    mentah = pd.read_excel(path, header=None, nrows=3, dtype=object)
    intip_on.append({"berkas": path.name, "baris_1": " | ".join(str(v) for v in mentah.iloc[0] if not kosong(v))})
intip_on = pd.DataFrame(intip_on)
print(intip_on["baris_1"].mask(intip_on["baris_1"].str.startswith("REKAP"), "REKAP ORDER <kanal> <bulan>")
      .str.slice(0, 90).value_counts().to_string())
header_online = pd.Series({p.name: len([c for c in baca_excel(p, "no order").columns if c not in ("baris_excel", "baris_header")])
                           for p in berkas_online})
print("\nJumlah kolom header per berkas:", header_online.value_counts().to_dict())

baris_1
NO | TANGGAL PESANAN MASUK | NO ORDER | DAERAH | NO HP | SKU BARANG | NAMA PRODUK | HARGA     45
REKAP ORDER <kanal> <bulan>                                                                   14

Jumlah kolom header per berkas: {27: 59}


**Hasil intip berkas online:**
- **14 dari 59 berkas** diawali judul laporan (`REKAP ORDER SHOPEE APR 2025`), sama seperti offline, sehingga posisi header perlu dicari. Kata penandanya **"NO ORDER"**.
- Berbeda dengan offline, nama kolom di semua berkas **sama** (27 kolom). Nama kolom dipetakan ke nama `snake_case` supaya mudah dipanggil, dan kanal diambil dari nama file apa adanya (mis. `TikTokShop`).

In [11]:
KOLOM_ONLINE = {
    "NO": "no", "TANGGAL PESANAN MASUK": "tanggal_pesanan", "NO ORDER": "no_order",
    "DAERAH": "daerah", "NO HP": "no_hp", "SKU BARANG": "sku_barang",
    "NAMA PRODUK": "nama_produk", "HARGA": "harga", "QTY": "qty", "BRUTO": "bruto",
    "DISKON": "diskon_persen", "RP DISKON PLATFORM": "rp_diskon_platform",
    "RP DISKON": "rp_diskon", "NETTO": "netto", "TOTAL": "total", "ONGKIR": "ongkir",
    "PROMO ONGKIR": "promo_ongkir", "BIAYA PROGRAM ONGKIR": "biaya_program_ongkir",
    "SISA ONGKIR (TANGGUNGAN PEMBELI)": "sisa_ongkir_pembeli",
    "KOMISI PLATFORM": "komisi_platform",
    "KOMISI AFFILIATOR / BIAYA CASHBACK": "komisi_affiliator",
    "LABA KOTOR (TIKTOK)": "laba_kotor", "HARGA KULAKAN": "harga_kulakan",
    "BIAYA PACKING": "biaya_packing", "LABA BERSIH": "laba_bersih",
    "BIAYA BALIK KULAKAN": "biaya_balik_kulakan", "LABA BERSIH TOTAL": "laba_bersih_total",
}

potongan, info_header_online = [], []
for path in berkas_online:
    df = baca_excel(path, "NO ORDER")
    info_header_online.append({"berkas": path.name, "baris_header": df["baris_header"].iloc[0]})
    df = df.rename(columns=lambda c: KOLOM_ONLINE.get(c.strip().upper(), c))
    m = re.search(r"rekap_online_(\w+?)_(\d{4}-\d{2})", path.name)
    df["platform"], df["bulan_laporan"] = m[1], m[2]
    df["berkas"] = path.name
    potongan.append(df)
online_raw = pd.concat(potongan, ignore_index=True)
info_header_online = pd.DataFrame(info_header_online)
KOLOM_ISI_ONLINE = list(KOLOM_ONLINE.values())
print("Jumlah baris mentah:", len(online_raw))
print(online_raw.groupby("platform")["berkas"].nunique().rename("jumlah_berkas"))
online_raw.head(3)

Jumlah baris mentah: 5220
platform
Shopee        21
TikTokShop    17
Tokopedia     21
Name: jumlah_berkas, dtype: int64


,no,tanggal_pesanan,no_order,daerah,no_hp,sku_barang,nama_produk,harga,qty,bruto,diskon_persen,rp_diskon_platform,rp_diskon,netto,total,ongkir,promo_ongkir,biaya_program_ongkir,sisa_ongkir_pembeli,komisi_platform,komisi_affiliator,laba_kotor,harga_kulakan,biaya_packing,laba_bersih,biaya_balik_kulakan,laba_bersih_total,baris_excel,baris_header,platform,bulan_laporan,berkas
0,1,2025-02-01 00:00:00,250201YQ1NZ13A,Kota Malang,NaN,DANGKE100,Dngke 40 WP,38798,1,38798,0.08,NaN,3103.8,35694.2,35694.2,0,0,980,0,2856,0,31858.2,23000,1000,7858.2,23000,8858.2,2,1,Shopee,2025-02,rekap_online_Shopee_2025-02.xlsx
1,2,2-2-25,250202B0EEV657,"KOTA MALANG, Jawa Timur",NaN,METAZEB1,Metazeb 80 WP - Murah,191468,1,191468,0.05,NaN,9573.4,181894.6,181894.6,0,0,0,0,14552,0,167342.6,113000,1000,53342.6,113000,63571.9,3,1,Shopee,2025-02,rekap_online_Shopee_2025-02.xlsx
2,3,2025-02-02 00:00:00,250202L7Q7BEQN,"LUMAJANG, Jawa Timur",08XX-XXXX-7144,FRD1,Furadan 3GR Ready Stock,36497,1,36497,0.1,NaN,3649.7,32847.3,32847.3,0,0,890,0,2628,0,29329.3,20100,1000,8229.3,20100,NaN,4,1,Shopee,2025-02,rekap_online_Shopee_2025-02.xlsx


### 3.5 Pencatatan stok: masuk/keluar, pemecahan, rakit paket
Workbook stok, opname, dan metrik juga dilihat dulu: daftar sheet dan isi 3 baris pertama setiap sheet.

In [12]:
intip_stok = []
workbook_lain = [RAW / "stok" / n for n in ["stok_masuk_keluar.xlsx", "stok_pemecahan.xlsx", "stok_rakit_paket.xlsx"]]
workbook_lain += [berkas_opname[0], berkas_opname[-1], RAW / "rekap_online" / "metrik_marketplace_OPSIONAL.xlsx"]
for path in workbook_lain:
    for sheet in pd.ExcelFile(path).sheet_names:
        mentah = pd.read_excel(path, sheet_name=sheet, header=None, nrows=3, dtype=object)
        intip_stok.append({"berkas": path.name, "sheet": sheet,
                           **{f"baris_{i + 1}": " | ".join(str(v) for v in mentah.iloc[i] if not kosong(v))[:80] for i in range(len(mentah))}})
pd.DataFrame(intip_stok)[["berkas", "sheet", "baris_1", "baris_2"]]

,berkas,sheet,baris_1,baris_2
0,stok_masuk_keluar.xlsx,2023-2024,Tanggal | SKU | Jenis | Qty | Harga Beli Satuan | Suppli...,2023-11-04 00:00:00 | UREA50KG | RESTOK | 1 karung | 651...
1,stok_masuk_keluar.xlsx,2025,Tgl | Kode | Jenis | Jumlah | Harga Beli | Supplier | Ket,2025-08-27 00:00:00 | RDPB | RESTOK | 50 pcs | 79600 | S...
2,stok_masuk_keluar.xlsx,2026,Tanggal | SKU | Jenis | Qty | Harga Beli Satuan | Suppli...,2026-01-01 00:00:00 | PLENUM25 | Barang masuk | 10 | SUP...
3,stok_pemecahan.xlsx,2023-2024,Tanggal | Barang Induk | Bahan Dipakai (kg) | SKU Hasil ...,1-11-23 | Kanitrat Maxxi Agri | 1 | KANITRAT500 | 2
4,stok_pemecahan.xlsx,2025,Tanggal | Barang Induk | Bahan Dipakai (kg) | SKU Hasil ...,2025-01-02 00:00:00 | Kapur Pertanian Kebomas | 50 | DLM...
5,stok_pemecahan.xlsx,2026,Tanggal | Barang Induk | Bahan Dipakai (kg) | SKU Hasil ...,2026-01-02 00:00:00 | Emkape Maxxi Agri | 1 | EMKAPE500 | 2
6,stok_rakit_paket.xlsx,2023-2024,Tanggal | SKU Paket | Jumlah Paket Jadi | Catatan,22-11-23 | PAKETTUMBUH250 | 4 pcs
7,stok_rakit_paket.xlsx,2025,Tanggal | SKU Paket | Jumlah Paket Jadi | Catatan,2025-04-17 00:00:00 | PAKETTUMBUH250 | 4
8,stok_rakit_paket.xlsx,2026,Tanggal | SKU Paket | Jumlah Paket Jadi | Catatan,07/01/2026 | PAKETTUMBUH250 | 4
9,stok_opname_2023-11-01.xlsx,Sheet1,Tanggal Opname | SKU | Stok Fisik | Catatan,01/11/2023 | AGT | 20


**Hasil intip workbook lain:**
- `stok_masuk_keluar.xlsx`, `stok_pemecahan.xlsx`, dan `stok_rakit_paket.xlsx` masing-masing punya **3 sheet** (2023-2024, 2025, 2026), dengan header di baris 1.
- Header `stok_masuk_keluar` **berbeda di sheet 2025**: `Tgl | Kode | Jenis | Jumlah | Harga Beli | Supplier | Ket`, sedangkan dua sheet lain memakai `Tanggal | SKU | Jenis | Qty | Harga Beli Satuan | Supplier | Keterangan`. Keduanya dipetakan ke nama yang sama (`KOLOM_STOK`); kata penanda header yang ada di kedua versi adalah **"Jenis"**.
- Header pemecahan dan rakit paket sama di semua sheet (penanda: **"SKU Hasil"** dan **"SKU Paket"**). Opname (penanda **"SKU"**) dan metrik marketplace (3 sheet per kanal, penanda **"Kanal"**) juga punya header yang konsisten.
- Fungsi `baca_semua_sheet` menggabungkan semua sheet dalam satu workbook memakai `baca_excel`, dan mencatat nama kolom asli per sheet.

In [13]:
KOLOM_STOK = {
    "tanggal": "tanggal", "tgl": "tanggal", "sku": "sku_teks", "kode": "sku_teks", "jenis": "jenis",
    "qty": "qty_teks", "jumlah": "qty_teks", "harga beli satuan": "harga_beli", "harga beli": "harga_beli",
    "supplier": "supplier", "keterangan": "keterangan", "ket": "keterangan",
}
KOLOM_PEMECAHAN = {"Tanggal": "tanggal", "Barang Induk": "barang_induk", "Bahan Dipakai (kg)": "bahan_teks",
                   "SKU Hasil": "sku_hasil_teks", "Jumlah Kemasan": "kemasan_teks", "Catatan": "catatan"}
KOLOM_RAKIT = {"Tanggal": "tanggal", "SKU Paket": "sku_paket_teks", "Jumlah Paket Jadi": "paket_teks",
               "Catatan": "catatan"}


def baca_semua_sheet(path, kata_kunci, ganti_nama):
    # Gabungkan semua sheet dalam satu workbook dan catat nama kolom asli tiap sheet.
    potongan, kolom_asli = [], {}
    for sheet in pd.ExcelFile(path).sheet_names:
        df = baca_excel(path, kata_kunci, sheet=sheet)
        kolom_asli[sheet] = [c for c in df.columns if c not in ("baris_excel", "baris_header")]
        df = df.rename(columns=lambda c: ganti_nama.get(c.lower(), ganti_nama.get(c, c)))
        df["sheet"] = sheet
        df["berkas"] = f"{path.name}[{sheet}]"
        potongan.append(df)
    return pd.concat(potongan, ignore_index=True), kolom_asli


stok_mk_raw, kolom_asli_stok_mk = baca_semua_sheet(RAW / "stok" / "stok_masuk_keluar.xlsx", "jenis", KOLOM_STOK)
pemecahan_raw, _ = baca_semua_sheet(RAW / "stok" / "stok_pemecahan.xlsx", "SKU Hasil", KOLOM_PEMECAHAN)
rakit_raw, _ = baca_semua_sheet(RAW / "stok" / "stok_rakit_paket.xlsx", "SKU Paket", KOLOM_RAKIT)
for nama, df in [("stok_masuk_keluar", stok_mk_raw), ("stok_pemecahan", pemecahan_raw), ("stok_rakit_paket", rakit_raw)]:
    print(f"{nama:18s}: {len(df):5d} baris")
stok_mk_raw.head()

stok_masuk_keluar :  4234 baris
stok_pemecahan    :   708 baris
stok_rakit_paket  :    41 baris


,tanggal,sku_teks,jenis,qty_teks,harga_beli,supplier,keterangan,baris_excel,baris_header,sheet,berkas
0,2023-11-04 00:00:00,UREA50KG,RESTOK,1 karung,651300,SUP-01,NaN,2,1,2023-2024,stok_masuk_keluar.xlsx[2023-2024]
1,05/11/2023,RGNT,Retur,1,NaN,NaN,NaN,3,1,2023-2024,stok_masuk_keluar.xlsx[2023-2024]
2,2023-11-07 00:00:00,KCL50KG,restock,3 karung,512500,SUP-01,NaN,4,1,2023-2024,stok_masuk_keluar.xlsx[2023-2024]
3,2023-11-09 00:00:00,FARMQUAT250,RESTOK,30,16400,SUP-06,NaN,5,1,2023-2024,stok_masuk_keluar.xlsx[2023-2024]
4,2023-11-09 00:00:00,GDSLD500,RESTOK,10,NaN,SUP-03,nota supplier,6,1,2023-2024,stok_masuk_keluar.xlsx[2023-2024]


### 3.6 Stok opname (13 kali opname)

In [14]:
potongan = []
for path in berkas_opname:
    df = baca_excel(path, "SKU")
    df = df.rename(columns={"Tanggal Opname": "tanggal", "SKU": "sku_teks", "Stok Fisik": "stok_fisik", "Catatan": "catatan"})
    df["berkas"] = path.name
    df["tanggal_berkas"] = re.search(r"\d{4}-\d{2}-\d{2}", path.name)[0]
    potongan.append(df)
opname_raw = pd.concat(potongan, ignore_index=True)
opname_raw.groupby("tanggal_berkas").size().rename("jumlah_baris")

tanggal_berkas
2023-11-01    189
2024-02-29     94
2024-05-31     83
2024-08-31     54
2024-11-30     56
2025-02-28     74
2025-05-31    107
2025-08-31     43
2025-11-30     74
2026-02-28     39
2026-05-31     45
2026-08-31    108
2026-10-31     97
Name: jumlah_baris, dtype: int64

### 3.7 Metrik marketplace (1 file, 3 sheet: Shopee, Tokopedia, TikTok Shop)
Metrik bulanan per produk per kanal: impresi, klik, pesanan, rating, jumlah ulasan, harga tayang. Dipakai untuk Q12.

In [15]:
KOLOM_METRIK = {"Periode Mulai": "periode_mulai", "Periode Akhir": "periode_akhir", "Kanal": "kanal",
                "SKU / Nama Produk": "produk", "Impresi": "impresi", "Klik / Pengunjung": "klik", "Pesanan": "pesanan",
                "Rating (1-5)": "rating", "Jumlah Ulasan": "jumlah_ulasan", "Harga Tayang (Rp)": "harga_tayang"}
path_metrik = RAW / "rekap_online" / "metrik_marketplace_OPSIONAL.xlsx"
metrik_raw, kolom_asli_metrik = baca_semua_sheet(path_metrik, "Kanal", KOLOM_METRIK)
for sheet, kolom in kolom_asli_metrik.items():
    print(f"{sheet:12s}: {len(metrik_raw[metrik_raw['sheet'] == sheet])} baris")
metrik_raw.head()

Shopee      : 1470 baris
Tokopedia   : 1470 baris
TikTok Shop : 1190 baris


,periode_mulai,periode_akhir,kanal,produk,impresi,klik,pesanan,rating,jumlah_ulasan,harga_tayang,baris_excel,baris_header,sheet,berkas
0,2025-02-01 00:00:00,2025-02-28 00:00:00,Shopee,Trobos 200 SL,441,19,1,4.6,0,20261,2,1,Shopee,metrik_marketplace_OPSIONAL.xlsx[Shopee]
1,2025-02-01 00:00:00,2025-02-28 00:00:00,Shopee,ETREKKCL,1231,13,0,4.8,0,14544,3,1,Shopee,metrik_marketplace_OPSIONAL.xlsx[Shopee]
2,2025-02-01 00:00:00,2025-02-28 00:00:00,Shopee,Superel 480 SL,6557,86,2,NaN,0,29610,4,1,Shopee,metrik_marketplace_OPSIONAL.xlsx[Shopee]
3,2025-02-01 00:00:00,2025-02-28 00:00:00,Shopee,MKP1,1113,53,2,4.9,0,78168,5,1,Shopee,metrik_marketplace_OPSIONAL.xlsx[Shopee]
4,2025-02-01 00:00:00,2025-02-28 00:00:00,Shopee,Emkape Maxxi Agri,874,26,2,4.6,0,67890,6,1,Shopee,metrik_marketplace_OPSIONAL.xlsx[Shopee]


**Insight (Gathering Data):**
- Ada **5 file master**, **36 rekap offline** (Nov 2023 s.d. Okt 2026), **59 rekap online** (Shopee 21, Tokopedia 21, TikTok Shop 17) + 1 file metrik marketplace (3 sheet), serta **3 workbook pencatatan stok** (masing-masing 3 sheet: 2023-2024, 2025, 2026) dan **13 file opname**.
- `products` berisi **190 SKU**: 186 SKU dijual + 4 SKU induk 50 kg. Ada 13 baris komposisi paket dan 7 supplier dummy.
- Data mentah: offline **55.776 baris**, online **5.220 baris**, stok masuk/keluar **4.234**, pemecahan **708**, rakit paket **41**, opname **1.063**, metrik marketplace **4.130** (Shopee 1.470, Tokopedia 1.470, TikTok Shop 1.190).
- Rekap offline **tidak punya kolom SKU**, hanya nama produk di nota. Rekap online punya kolom `SKU BARANG` dan `NAMA PRODUK`. Semua tabel lain merujuk SKU, jadi pemetaan ke master adalah langkah terpenting.
- Opname pertama (1 Nov 2023) mencakup semua SKU (189 baris). Opname berikutnya bergilir 39 sampai 108 SKU.

## 4. Assessing Data
Setiap tabel diperiksa satu per satu. Pada tahap ini **belum ada data yang diubah**. Pemeriksaan setiap tabel dilakukan dalam dua langkah:

1. **Review awal (umum).** Pemeriksaan yang sama untuk semua tabel, tanpa menebak masalahnya: info dan tipe data, missing values, duplikat, ringkasan statistik, jumlah nilai unik, dan contoh isi. Karena file Excel dibaca apa adanya (`dtype=object`, supaya nomor pesanan panjang tidak rusak), ditambah satu pemeriksaan: kolom yang seharusnya angka dicoba diubah ke angka, lalu dihitung berapa yang gagal. Hasilnya dicatat sebagai **gejala**.
2. **Pengecekan lanjutan (terarah).** Setiap gejala diselidiki lebih dalam sampai jelas masalahnya dan jumlahnya. Seberapa dalam sebuah kolom diperiksa juga ditentukan oleh **pertanyaan bisnis nya**.

Fungsi bantu untuk review awal. `review_awal` menjalankan pemeriksaan umum yang sama untuk setiap tabel; `coba_jadi_angka` mencoba mengubah kolom ke angka dengan `pd.to_numeric` dan menghitung isi yang gagal; `tipe_isi` menghitung jenis isi sel per kolom (satu kolom Excel bisa berisi angka, teks, dan tanggal sekaligus).

In [16]:
KOLOM_BANTU = ["berkas", "sheet", "baris_excel", "baris_header", "tanggal_berkas", "bulan_berkas", "bulan_laporan", "platform"]


def review_awal(df, nama):
    # Pemeriksaan umum: info, missing values, duplikat, ringkasan statistik, nilai unik.
    isi = [k for k in df.columns if k not in KOLOM_BANTU]
    letak = [k for k in ["berkas", "sheet"] if k in df.columns]
    print(f"===== {nama}: {len(df)} baris, {len(isi)} kolom isi =====")
    print("1. Info & tipe data")
    df[isi].info()
    print("\n2. Missing values per kolom")
    print(df[isi].isna().sum().to_string())
    print("\n3. Jumlah duplikat (semua kolom isi sama, dalam berkas/sheet yang sama):",
          df.duplicated(subset=letak + isi).sum())
    print("\n4. Jumlah nilai unik per kolom")
    print(df[isi].nunique().to_string())
    print("\n5. Ringkasan statistik")
    return df[isi].describe(include="all").T


def coba_jadi_angka(df, kolom):
    # Coba ubah kolom ke angka. Hitung isi yang terisi tetapi gagal diubah, beserta contohnya.
    baris = []
    for k in kolom:
        terisi = df[k].notna()
        angka = pd.to_numeric(df[k], errors="coerce")
        gagal = terisi & angka.isna()
        baris.append({"kolom": k, "terisi": terisi.sum(), "gagal_jadi_angka": gagal.sum(),
                      "contoh_gagal": df.loc[gagal, k].astype(str).unique()[:4].tolist(),
                      "min": angka.min(), "max": angka.max(), "negatif": (angka < 0).sum()})
    return pd.DataFrame(baris).set_index("kolom")


def tipe_isi(df, kolom):
    # Hitung jenis isi sel (int, float, str, datetime, kosong) untuk setiap kolom.
    hasil = {}
    for k in kolom:
        hasil[k] = df[k].map(lambda v: "kosong" if kosong(v) else type(v).__name__).value_counts()
    return pd.DataFrame(hasil).fillna(0).astype(int)

### 4.1 Menilai master produk

#### Review awal

In [17]:
review_awal(master_konversi, "master_induk_konversi")

===== master_induk_konversi: 190 baris, 10 kolom isi =====
1. Info & tipe data
<class 'pandas.DataFrame'>
RangeIndex: 190 entries, 0 to 189
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   sku                   190 non-null    str    
 1   product_id            190 non-null    int64  
 2   parent_id             185 non-null    float64
 3   parent_name           190 non-null    str    
 4   base_unit             185 non-null    str    
 5   faktor_per_unit_jual  185 non-null    float64
 6   isi_per_pembelian     185 non-null    float64
 7   satuan_beli           185 non-null    str    
 8   is_bundle             190 non-null    int64  
 9   label                 190 non-null    str    
dtypes: float64(3), int64(2), str(5)
memory usage: 15.0 KB

2. Missing values per kolom
sku                     0
product_id              0
parent_id               5
parent_name             0
base_unit              

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
sku,190,190,AGT,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_id,190.0,NaN,NaN,NaN,95.5,54.992424,1.0,48.25,95.5,142.75,190.0
parent_id,185.0,NaN,NaN,NaN,90.540541,51.918411,1.0,46.0,91.0,134.0,179.0
parent_name,190,169,(paket),5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
base_unit,185,2,pcs,177,NaN,NaN,NaN,NaN,NaN,NaN,NaN
faktor_per_unit_jual,185.0,NaN,NaN,NaN,2.045946,7.148601,0.5,1.0,1.0,1.0,50.0
isi_per_pembelian,185.0,NaN,NaN,NaN,12.216216,10.423466,10.0,10.0,10.0,10.0,100.0
satuan_beli,185,3,pcs (min. beli 10),177,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_bundle,190.0,NaN,NaN,NaN,0.026316,0.160496,0.0,0.0,0.0,0.0,1.0
label,190,1,DUMMY,190,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
review_awal(master_kategori, "master_kategori")

===== master_kategori: 186 baris, 4 kolom isi =====
1. Info & tipe data
<class 'pandas.DataFrame'>
RangeIndex: 186 entries, 0 to 185
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   sku       186 non-null    str  
 1   nama      186 non-null    str  
 2   kategori  186 non-null    str  
 3   label     186 non-null    str  
dtypes: str(4)
memory usage: 5.9 KB

2. Missing values per kolom
sku         0
nama        0
kategori    0
label       0

3. Jumlah duplikat (semua kolom isi sama, dalam berkas/sheet yang sama): 0

4. Jumlah nilai unik per kolom
sku         186
nama        173
kategori      6
label         1

5. Ringkasan statistik


,count,unique,top,freq
sku,186,186,AGT,1
nama,186,173,Jetspray,2
kategori,186,6,Insektisida,52
label,186,1,DUMMY (asumsi tim),186


In [19]:
review_awal(master_harga, "penetapan_harga_dasar")

===== penetapan_harga_dasar: 186 baris, 23 kolom isi =====
1. Info & tipe data
<class 'pandas.DataFrame'>
RangeIndex: 186 entries, 0 to 185
Data columns (total 23 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   KODE SKU                      186 non-null    str    
 1   NO.                           186 non-null    int64  
 2   NAMA                          186 non-null    str    
 3   HARGA PRODUKSI/KULAKAN        186 non-null    int64  
 4   BIAYA PENGEMASAN              186 non-null    int64  
 5   BIAYA PENYIMPANAN             186 non-null    int64  
 6   HPP                           186 non-null    int64  
 7   PROFIT KOTOR %                186 non-null    float64
 8   PROFIT KOTOR Rp               186 non-null    int64  
 9   HARGA JUAL BRUTO              186 non-null    int64  
 10  DISKON PEMBELI %              186 non-null    float64
 11  DISKON PEMBELI Rp             186 non-null    int64  
 

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
KODE SKU,186,186,AGT,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NO.,186.0,NaN,NaN,NaN,94.005376,54.272436,1.0,47.25,94.5,140.75,187.0
NAMA,186,173,Jetspray,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HARGA PRODUKSI/KULAKAN,186.0,NaN,NaN,NaN,31790.322581,23303.374953,2000.0,16000.0,24750.0,42875.0,115000.0
BIAYA PENGEMASAN,186.0,NaN,NaN,NaN,30.913978,100.934676,0.0,0.0,0.0,0.0,500.0
BIAYA PENYIMPANAN,186.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
HPP,186.0,NaN,NaN,NaN,31821.236559,23285.300852,2000.0,16000.0,24750.0,42875.0,115000.0
PROFIT KOTOR %,186.0,NaN,NaN,NaN,0.684892,0.236289,0.3,0.53,0.645,0.77,1.65
PROFIT KOTOR Rp,186.0,NaN,NaN,NaN,20354.22043,15151.800553,1778.0,9746.25,15190.0,26455.0,76950.0
HARGA JUAL BRUTO,186.0,NaN,NaN,NaN,52175.456989,37638.82034,4028.0,26235.0,40045.0,70020.0,186450.0


In [20]:
master_komposisi.info()
print("\nMissing values:", master_komposisi.isna().sum().sum(), "| duplikat:", master_komposisi.duplicated().sum())
display(master_komposisi.head(3))
master_supplier

<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 4 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   sku_paket                                    13 non-null     str    
 1   sku_komponen                                 13 non-null     str    
 2   jumlah_per_paket_dalam_satuan_jual_komponen  13 non-null     float64
 3   label                                        13 non-null     str    
dtypes: float64(1), str(3)
memory usage: 548.0 bytes

Missing values: 0 | duplikat: 0


,sku_paket,sku_komponen,jumlah_per_paket_dalam_satuan_jual_komponen,label
0,PAKETAG,AGB,1.0,DUMMY
1,PAKETAG,AGT,1.0,DUMMY
2,PAKETGDSL,GDSLB,1.0,DUMMY


,supplier_code,nama_dummy,kelompok_produk,lead_time_hari_asumsi,label
0,SUP-01,Supplier 01 (dummy),Pupuk curah dan karung,3,DUMMY
1,SUP-02,Supplier 02 (dummy),Pupuk NPK dan mikro,5,DUMMY
2,SUP-03,Supplier 03 (dummy),Nutrisi dan ZPT,4,DUMMY
3,SUP-04,Supplier 04 (dummy),Insektisida,3,DUMMY
4,SUP-05,Supplier 05 (dummy),Fungisida,5,DUMMY
5,SUP-06,Supplier 06 (dummy),Herbisida,5,DUMMY
6,SUP-07,Supplier 07 (dummy),Perlengkapan,7,DUMMY


**Gejala dari review awal master:**
1. `master_induk_konversi` berisi **190 SKU**, sedangkan `master_kategori` dan `penetapan_harga_dasar` hanya **186 SKU**. Ada SKU yang hanya muncul di satu file. → cek **a**
2. Lima baris di `master_induk_konversi` kosong di kolom `parent_id`, `base_unit`, `faktor_per_unit_jual`, `isi_per_pembelian`, dan `satuan_beli`; kelimanya bertanda `is_bundle = 1` (paket).
3. `HARGA JUAL OFFLINE` kosong di **4 SKU**. Harga dibutuhkan untuk menebak ukuran dari harga (pemetaan SKU) dan untuk margin (Q3). → cek **b**
4. `nama` hanya punya **173 nilai unik dari 186 SKU** (contoh `Jetspray` muncul 2 kali). Artinya ada nama yang dipakai lebih dari satu SKU. Rekap offline hanya mencatat nama, jadi ini berisiko untuk semua pertanyaan per produk. → cek **c**
5. `parent_id` punya 179 nilai unik untuk 185 baris terisi, dan `faktor_per_unit_jual` berkisar 0,5 sampai 50. Belum jelas apakah `parent_id` menunjuk SKU induk. → cek **d**
6. Tidak ada duplikat. `master_supplier` hanya berisi lead time yang ditandai **asumsi** (`lead_time_hari_asumsi`), dibutuhkan untuk Q5 dan Q8.

#### Pengecekan lanjutan

**a. Jumlah SKU antar file master** (gejala 1): kenapa `master_induk_konversi` berisi 190 SKU, sedangkan dua file lain 186?

In [21]:
print("SKU di master_induk_konversi:", master_konversi["sku"].nunique(), "dari", len(master_konversi), "baris")
print("SKU di master_kategori      :", master_kategori["sku"].nunique())
print("SKU di penetapan_harga_dasar:", master_harga["KODE SKU"].nunique())
print("SKU yang tidak ada di master_kategori:", sorted(set(master_konversi["sku"]) - set(master_kategori["sku"])))
print("Selisih SKU kategori vs harga:", set(master_kategori["sku"]) ^ set(master_harga["KODE SKU"]))

SKU di master_induk_konversi: 190 dari 190 baris
SKU di master_kategori      : 186
SKU di penetapan_harga_dasar: 186
SKU yang tidak ada di master_kategori: ['DLMT50KG', 'FRTPHS50KG', 'KCL50KG', 'UREA50KG']
Selisih SKU kategori vs harga: set()


**b. Harga jual offline yang kosong** (gejala 3).

In [22]:
# Harga yang kosong
kosong_harga = master_harga[master_harga[["HARGA JUAL ONLINE", "HARGA JUAL OFFLINE"]].isna().any(axis=1)]
kosong_harga[["KODE SKU", "NAMA", "HARGA JUAL ONLINE", "HARGA JUAL OFFLINE"]]

,KODE SKU,NAMA,HARGA JUAL ONLINE,HARGA JUAL OFFLINE
54,PAKETAG,Paket AG Buah dan Tumbuh,47513,NaN
55,PAKETGDSL,Paket Gandasil B dan D,37748,NaN
58,PAKETGNRTF,Paket Generatif,191573,NaN
123,EXTRAPCKG,extra kardus + bubblewrap,4252,NaN


**c. Nama yang dipakai lebih dari satu SKU** (gejala 4). Penting untuk semua pertanyaan per produk, karena rekap offline hanya mencatat nama.

In [23]:
# Nama produk yang dipakai lebih dari satu SKU (beda ukuran kemasan)
nama_ganda = master_kategori[master_kategori["nama"].str.lower().duplicated(keep=False)]
nama_ganda = nama_ganda.merge(master_harga[["KODE SKU", "HARGA JUAL OFFLINE", "HARGA JUAL ONLINE"]],
                              left_on="sku", right_on="KODE SKU").sort_values("nama")
print("Jumlah nama yang dipakai >1 SKU:", nama_ganda["nama"].str.lower().nunique())
nama_ganda[["nama", "sku", "HARGA JUAL OFFLINE", "HARGA JUAL ONLINE"]]

Jumlah nama yang dipakai >1 SKU: 13


,nama,sku,HARGA JUAL OFFLINE,HARGA JUAL ONLINE
12,Antracol 70 WP,ANTRCL,57500.0,65730
13,Antracol 70 WP,ANTRCL500,125000.0,146855
15,Decis 25 EC,DCSB,48500.0,58958
14,Decis 25 EC,DCSK,26000.0,32235
16,Emkape Maxxi Agri,EMKAPE500,37000.0,45675
5,Emkape Maxxi Agri,EMKAPE1,64000.0,77963
1,Fenite 150 OD,FENITE100,129500.0,157132
2,Fenite 150 OD,FENITE50,71000.0,81322
17,Furadan 3GR,FRD1,31500.0,38115
18,Furadan 3GR,FRD2,48000.0,54495


**d. Arti kolom `parent_id`** (gejala 5): apakah berisi `product_id` induk atau kode kelompok?

In [24]:
# Arti kolom parent_id: apakah berisi product_id induk, atau kode kelompok?
jumlah_per_kelompok = master_konversi["parent_id"].map(master_konversi["parent_id"].value_counts())
kelompok_isi_banyak = master_konversi[jumlah_per_kelompok > 1].sort_values(["parent_id", "faktor_per_unit_jual"])
kelompok_isi_banyak[["parent_id", "parent_name", "sku", "product_id", "faktor_per_unit_jual"]]

,parent_id,parent_name,sku,product_id,faktor_per_unit_jual
11,12.0,Kapur Pertanian Kebomas,DLMT,12,0.5
188,12.0,Kapur Pertanian Kebomas,DLMT50KG,189,50.0
122,71.0,Emkape Maxxi Agri,EMKAPE500,123,0.5
75,71.0,Emkape Maxxi Agri,EMKAPE1,76,1.0
126,121.0,Fertiphos Pak Tani,FRTPHS1,127,1.0
189,121.0,Fertiphos Pak Tani,FRTPHS50KG,190,50.0
133,128.0,Kanitrat Maxxi Agri,KANITRAT500,134,0.5
134,128.0,Kanitrat Maxxi Agri,KANITRAT1,135,1.0
137,131.0,Pupuk KCL Mahkota,KCL,138,0.5
187,131.0,Pupuk KCL Mahkota,KCL50KG,188,50.0


Contoh di atas: `EMKAPE500` dan `EMKAPE1` sama-sama punya `parent_id` 71, padahal `product_id` EMKAPE1 bukan 71. Artinya `parent_id` adalah **kode kelompok produk**, bukan `product_id` induk, sehingga induk pemecahan harus ditentukan sendiri (SKU dengan faktor terbesar dalam kelompok).

**Data Review `products`:**
- **190 SKU unik**, tidak ada SKU ganda.
- Missing values wajar: 4 SKU induk 50 kg tidak punya harga karena tidak dijual, dan 5 SKU paket tidak punya kelompok induk/faktor karena dirakit dari komponen.
- **4 SKU tidak punya harga jual offline**: 3 paket (`PAKETAG`, `PAKETGDSL`, `PAKETGNRTF`) dan `EXTRAPCKG` (kardus tambahan kiriman). Keempatnya memang hanya dijual online.
- **13 nama produk dipakai 2 SKU** berbeda ukuran. Harga kedua ukuran berbeda jauh (mis. Emkape: EMKAPE500 Rp37.000 vs EMKAPE1 Rp64.000 di offline), jadi harga satuan pada bulan transaksi bisa dipakai untuk membedakannya.
- Ada **6 relasi induk → kemasan**: 4 induk 50 kg (→ DLMT, FRTPHS1, KCL, UREA) dan 2 kemasan pabrik 1 kg (EMKAPE1 → EMKAPE500, KANITRAT1 → KANITRAT500).
- Catatan: kolom `parent_id` di `master_induk_konversi.csv` adalah **kode kelompok**, bukan `product_id` induk. Karena itu kolomnya diganti nama menjadi `kelompok_induk` supaya tidak salah dibaca.

### 4.2 Menilai `offline_raw`

#### Review awal

In [25]:
review_awal(offline_raw, "offline_raw")

===== offline_raw: 55776 baris, 8 kolom isi =====
1. Info & tipe data
<class 'pandas.DataFrame'>
RangeIndex: 55776 entries, 0 to 55775
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   tanggal       55722 non-null  object
 1   no_nota       52940 non-null  object
 2   nama_produk   55776 non-null  object
 3   qty           55453 non-null  object
 4   harga_satuan  55207 non-null  object
 5   diskon        6008 non-null   object
 6   subtotal      55776 non-null  object
 7   catatan       27888 non-null  object
dtypes: object(8)
memory usage: 3.4+ MB

2. Missing values per kolom
tanggal            54
no_nota          2836
nama_produk         0
qty               323
harga_satuan      569
diskon          49768
subtotal            0
catatan         27888

3. Jumlah duplikat (semua kolom isi sama, dalam berkas/sheet yang sama): 404

4. Jumlah nilai unik per kolom
tanggal          4689
no_nota         25142
nama_produk   

,count,unique,top,freq
tanggal,55722,4689,2024-11-30 00:00:00,94
no_nota,52940,25142,0396,30
nama_produk,55776,1675,KCL Curah,1445
qty,55453,21,1,32162
harga_satuan,55207,467,12000,1784
diskon,6008,40,500,1791
subtotal,55776,741,24000,1222
catatan,27888,4,langganan,7059


In [26]:
print("Jenis isi sel:")
display(tipe_isi(offline_raw, ["tanggal", "no_nota", "nama_produk", "qty", "harga_satuan", "diskon", "subtotal"]))
coba_jadi_angka(offline_raw, ["qty", "harga_satuan", "diskon", "subtotal"])

Jenis isi sel:


,tanggal,no_nota,nama_produk,qty,harga_satuan,diskon,subtotal
datetime,40228,0,0,0,0,0,0
int,0,0,0,55453,53575,6008,55776
kosong,54,2836,0,323,569,49768,0
str,15494,52940,55776,0,1632,0,0


,terisi,gagal_jadi_angka,contoh_gagal,min,max,negatif
kolom,,,,,,
qty,55453,0,[],-20.0,80.0,58
harga_satuan,55207,1632,"[Rp 112.500, 13,500, 29,000, 21,000]",0.0,164500.0,0
diskon,6008,0,[],500.0,26500.0,0
subtotal,55776,0,[],2000.0,121667500.0,0


In [27]:
offline_raw.sample(8, random_state=1)

,tanggal,no_nota,nama_produk,qty,harga_satuan,diskon,subtotal,catatan,baris_excel,baris_header,berkas,bulan_berkas
1463,26/11/2023,2311-0769,e-trek anti rayap,1,11000,NaN,11000,NaN,1465,1,rekap_penjualan_offline_2023-11.xlsx,2023-11
41550,2026-01-29 00:00:00,1104,E-Trek anti rayap,2,12000,NaN,24000,nego,2020,1,rekap_penjualan_offline_2026-01.xlsx,2026-01
24295,2025-03-06 00:00:00,2503-0160,Kinglon 667,3,Rp 32.500,NaN,97500,NaN,276,1,rekap_penjualan_offline_2025-03.xlsx,2025-03
33098,2025-09-17 00:00:00,2509-0464,BALISTIC 50 SC,1,57000,NaN,57000,nego,853,1,rekap_penjualan_offline_2025-09.xlsx,2025-09
52811,2026-09-12 00:00:00,2609-0393,Avidor 25 WP,1,35000,NaN,35000,langganan,499,1,rekap_penjualan_offline_2026-09.xlsx,2026-09
45079,2026-04-04 00:00:00,2604-0172,TROBOS 200 SL,3,19000,NaN,57000,cash,331,3,rekap_penjualan_offline_2026-04.xlsx,2026-04
12997,4-8-24,0094,ALIKA50,1,39000,NaN,39000,tempo,181,3,rekap_penjualan_offline_2024-08.xlsx,2024-08
2675,21 Des 2023,0586,POLYBAG TANAMAN 35X35,6,19500,NaN,117000,tempo,1045,3,rekap_penjualan_offline_2023-12.xlsx,2023-12


**Gejala dari review awal `offline_raw`:**
1. **Semua kolom bertipe `object`**: isi sel campur. `tanggal` berisi 40.228 tanggal Excel dan 15.494 teks dengan format berbeda-beda (`26/11/2023`, `4-8-24`, `21 Des 2023`). Tanggal dibutuhkan untuk semua analisis bulanan (Q1, Q2, Q7). → cek **d**
2. `tanggal` kosong di **54 baris** dan `subtotal` maksimum **Rp121,7 juta**, jauh di atas transaksi biasa (median Rp37.500). Kemungkinan ada baris subtotal/total di dalam file. → cek **b**
3. `no_nota` kosong di **2.836 baris**, dan nilai paling sering adalah `0396`, yaitu nomor 4 digit tanpa tahun-bulan, padahal sampel lain berbentuk `2311-0769`. → cek **f**
4. `harga_satuan`: **1.632 isi gagal diubah ke angka** (`Rp 112.500`, `13,500`) dan ada harga 0. `qty` kosong 323 dan `harga_satuan` kosong 569. Ketiga kolom ini membentuk omzet (Q1–Q3). → cek **e**
5. `qty` minimum **−20** (58 baris negatif) dan maksimum **80**, padahal hampir semua transaksi 1–3 unit. Dua-duanya perlu dicocokkan dengan `subtotal`. → cek **e** dan **h**
6. **404 baris duplikat** (semua kolom isi sama dalam berkas yang sama). → cek **g**
7. `nama_produk` punya **1.675 nilai unik**, padahal master hanya 186 SKU. Di sampel, produk yang sama ditulis berbeda (`e-trek anti rayap` vs `E-Trek anti rayap`, `BALISTIC 50 SC`) dan ada yang ditulis sebagai kode (`ALIKA50`). → cek **i**
8. `baris_header` bernilai 1 atau 3 (posisi header berbeda antarberkas). → cek **a**
9. `diskon` kosong di 49.768 baris: wajar, artinya tidak ada diskon. `catatan` hanya berisi 4 nilai (`langganan`, `nego`, `cash`, `tempo`) dan tidak dipakai pertanyaan bisnis.

#### Pengecekan lanjutan

**a. Posisi header dan nama kolom (kolom bergeser).**

In [28]:
info_header_offline.groupby(["baris_header", "kolom_asli"]).size().rename("jumlah_berkas")

baris_header  kolom_asli                                                                                 
1             Tanggal | No. Nota | Nama Produk di Nota | Qty | Harga Satuan | Diskon | Subtotal | Catatan    10
              Tgl | No Nota | Nama Barang | Jml | Harga | Disc | Total | Ket                                  9
3             Tanggal | No. Nota | Nama Produk di Nota | Qty | Harga Satuan | Diskon | Subtotal | Catatan    17
Name: jumlah_berkas, dtype: int64

**b. Sel gabungan dan baris subtotal/total.**

In [29]:
jumlah_gabungan = sum(len(load_workbook(p).worksheets[0].merged_cells.ranges) for p in berkas_offline)
print("Sel gabungan di semua berkas offline:", jumlah_gabungan)
is_total_off = (offline_raw["nama_produk"].astype("string").str.upper().str.contains("TOTAL", na=False)
                & offline_raw["tanggal"].isna())
print("Baris subtotal/total:", is_total_off.sum())
offline_raw.loc[is_total_off, "nama_produk"].value_counts()

Sel gabungan di semua berkas offline: 0
Baris subtotal/total: 54


nama_produk
SUBTOTAL MINGGUAN    37
TOTAL                17
Name: count, dtype: int64

**c. Tipe isi sel setelah baris subtotal/total dikeluarkan.** Baris subtotal sudah ditemukan di **b**, jadi pengecekan berikutnya memakai `off_tx` (baris transaksi saja). Dari sini terlihat bahwa sebagian sel kosong di review awal berasal dari baris subtotal.

In [30]:
off_tx = offline_raw[~is_total_off]
kolom_off = ["tanggal", "no_nota", "nama_produk", "qty", "harga_satuan", "diskon", "subtotal", "catatan"]
tipe_isi(off_tx, kolom_off)

,tanggal,no_nota,nama_produk,qty,harga_satuan,diskon,subtotal,catatan
datetime,40228,0,0,0,0,0,0,0
int,0,0,0,55453,53575,6008,55722,0
kosong,0,2782,0,269,515,49714,0,27834
str,15494,52940,55722,0,1632,0,0,27888


**d. Format tanggal dan tanggal di luar bulan berkas.**

In [31]:
# Pola tanggal yang berupa teks (angka diganti 9 supaya polanya terlihat)
tgl_teks = off_tx.loc[off_tx["tanggal"].map(lambda v: isinstance(v, str)), "tanggal"]
tgl_teks.str.replace(r"\d", "9", regex=True).value_counts()

tanggal
99/99/9999     6805
99-9-99        2192
9999/99/99     1085
9-9-99          913
99-99-99        892
9-99-99         366
99 Jan 9999     247
99 Nov 9999     223
99 Des 9999     214
99 Okt 9999     203
99 Sep 9999     193
99 Mei 9999     190
99 Agu 9999     184
99 Jun 9999     172
99 Mar 9999     170
99 Apr 9999     170
99 Jul 9999     159
99 Feb 9999     148
9 Mei 9999      114
9 Nov 9999       97
9 Okt 9999       90
9 Jun 9999       84
9 Jan 9999       83
9 Des 9999       81
9 Mar 9999       78
9 Feb 9999       71
9 Jul 9999       69
9 Agu 9999       69
9 Sep 9999       67
9 Apr 9999       65
Name: count, dtype: int64

Ada empat bentuk tanggal teks: `dd/mm/yyyy`, `d-m-yy`, `yyyy/mm/dd`, dan `d Mmm yyyy` dengan nama bulan Indonesia (`Des`, `Agu`). `pd.to_datetime` tidak mengenali nama bulan Indonesia dan bisa salah membaca urutan hari-bulan, jadi dibuat fungsi `parse_tanggal` yang mengenali keempat bentuk itu secara eksplisit. Bentuk yang tidak dikenal menjadi `NaT` (tidak ditebak).

In [32]:
BULAN_ID = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "mei": 5, "jun": 6,
            "jul": 7, "agu": 8, "sep": 9, "okt": 10, "nov": 11, "des": 12}


def parse_tanggal(nilai):
    # Ubah berbagai format tanggal menjadi pd.Timestamp. Format yang tidak dikenal -> NaT (tidak ditebak).
    # Format: tanggal Excel, '05/11/2023', '1-11-23', '2024/12/05', '12 Jan 2025'.
    if kosong(nilai):
        return pd.NaT
    if hasattr(nilai, "year"):                         # sudah berupa tanggal dari Excel
        return pd.Timestamp(nilai).normalize()
    teks = str(nilai).strip()
    m = re.fullmatch(r"(\d{1,2}) ([A-Za-z]{3}) (\d{4})", teks)        # 12 Jan 2025
    if m and m[2].lower() in BULAN_ID:
        return pd.Timestamp(int(m[3]), BULAN_ID[m[2].lower()], int(m[1]))
    m = re.fullmatch(r"(\d{4})/(\d{1,2})/(\d{1,2})", teks)             # 2024/12/05
    if m:
        return pd.Timestamp(int(m[1]), int(m[2]), int(m[3]))
    m = re.fullmatch(r"(\d{1,2})[/-](\d{1,2})[/-](\d{2}|\d{4})", teks)  # 05/11/2023, 1-11-23
    if m:
        tahun = int(m[3]) + 2000 if len(m[3]) == 2 else int(m[3])
        return pd.Timestamp(tahun, int(m[2]), int(m[1]))
    return pd.NaT


print(parse_tanggal("21 Des 2023"), parse_tanggal("4-8-24"), parse_tanggal("2024/12/05"), parse_tanggal("26/11/2023"))

2023-12-21 00:00:00 2024-08-04 00:00:00 2024-12-05 00:00:00 2023-11-26 00:00:00


Dengan `parse_tanggal`, tanggal setiap baris dibaca lalu dibandingkan dengan bulan berkasnya.

In [33]:
tgl_off = off_tx["tanggal"].map(parse_tanggal)
di_luar = tgl_off.isna() | (tgl_off.dt.strftime("%Y-%m") != off_tx["bulan_berkas"])
hanya_tahun = di_luar & (tgl_off.dt.month == off_tx["bulan_berkas"].str[5:7].astype(int))
print("\nTanggal tidak terbaca        :", tgl_off.isna().sum())
print("Tanggal di luar bulan berkas :", di_luar.sum(), "| yang beda tahunnya saja:", hanya_tahun.sum())
off_tx.loc[di_luar, ["berkas", "baris_excel", "tanggal", "bulan_berkas"]].head()


Tanggal tidak terbaca        : 0
Tanggal di luar bulan berkas : 51 | yang beda tahunnya saja: 51


,berkas,baris_excel,tanggal,bulan_berkas
354,rekap_penjualan_offline_2023-11.xlsx,356,2022-11-07 00:00:00,2023-11
1109,rekap_penjualan_offline_2023-11.xlsx,1111,2022-11-21 00:00:00,2023-11
2392,rekap_penjualan_offline_2023-12.xlsx,762,2022-12-16 00:00:00,2023-12
3759,rekap_penjualan_offline_2024-01.xlsx,497,2023-01-09 00:00:00,2024-01
5101,rekap_penjualan_offline_2024-02.xlsx,172,2023-02-05 00:00:00,2024-02


**e. Angka berupa teks, qty/harga kosong, subtotal tidak cocok.** Di review awal, `harga_satuan` gagal diubah ke angka karena ditulis `Rp 112.500` (titik sebagai pemisah ribuan) atau `13,500` (koma sebagai pemisah ribuan). Pola isinya dilihat dulu, lalu dibuat fungsi `parse_angka`.

In [34]:
harga_teks = off_tx.loc[off_tx["harga_satuan"].map(lambda v: isinstance(v, str)), "harga_satuan"]
harga_teks.str.replace(r"\d", "9", regex=True).value_counts()

harga_satuan
Rp 99.999     815
99,999        555
Rp 9.999      131
9,999          84
Rp 999.999     28
999,999        19
Name: count, dtype: int64

In [35]:
def parse_angka(nilai):
    # Ubah angka teks menjadi float: 'Rp 112.500' -> 112500, '13,500' -> 13500.
    if kosong(nilai):
        return np.nan
    if isinstance(nilai, (int, float, np.integer, np.floating)):
        return float(nilai)
    teks = str(nilai).strip()
    if teks.lower().startswith("rp"):
        return float(re.sub(r"[^\d]", "", teks))
    if re.fullmatch(r"-?\d{1,3}(,\d{3})+", teks):
        return float(teks.replace(",", ""))
    try:
        return float(teks)
    except ValueError:
        return np.nan


print(parse_angka("Rp 112.500"), parse_angka("13,500"), parse_angka(27000), parse_angka("abc"))

112500.0 13500.0 27000.0 nan


In [36]:
qty_num = off_tx["qty"].map(parse_angka)
harga_num = off_tx["harga_satuan"].map(parse_angka)
subtotal_num = off_tx["subtotal"].map(parse_angka)
diskon_num = off_tx["diskon"].map(parse_angka).fillna(0)
print("Contoh harga teks          :", off_tx.loc[off_tx["harga_satuan"].map(lambda v: isinstance(v, str)), "harga_satuan"].head(4).tolist())
print("Qty kosong                 :", qty_num.isna().sum())
print("Harga kosong atau 0        :", (harga_num.isna() | (harga_num == 0)).sum())
selisih = (qty_num.abs() * harga_num - diskon_num - subtotal_num).abs() > 0.5
print("Subtotal != qty x harga - diskon:", (selisih & qty_num.notna() & (harga_num > 0)).sum())
qty_dari_subtotal = (subtotal_num + diskon_num) / harga_num
qty_lain = selisih & np.isclose(qty_dari_subtotal, qty_dari_subtotal.round()) & (qty_dari_subtotal.round() > 0) & (qty_dari_subtotal.round() != qty_num.abs())
print("  di antaranya subtotal = harga x bilangan bulat lain (qty salah ketik):", qty_lain.sum())
pd.DataFrame({"qty": qty_num, "harga": harga_num, "diskon": diskon_num, "subtotal": subtotal_num})[qty_lain].head(4)

Contoh harga teks          : ['Rp 112.500', '13,500', '29,000', '21,000']
Qty kosong                 : 269
Harga kosong atau 0        : 628
Subtotal != qty x harga - diskon: 923
  di antaranya subtotal = harga x bilangan bulat lain (qty salah ketik): 216


,qty,harga,diskon,subtotal
191,1.0,0.0,0.0,29000.0
795,10.0,27000.0,0.0,27000.0
1025,1.0,0.0,1000.0,26000.0
1851,10.0,19500.0,1000.0,18500.0


**f. Nomor nota.**

In [37]:
pola_nota = off_tx["no_nota"].map(lambda v: "kosong" if kosong(v) else re.sub(r"\d", "9", str(v)))
pola_nota.value_counts()

no_nota
9999-9999    36999
9999         15941
kosong        2782
Name: count, dtype: int64

**g. Duplikat persis pada data mentah** (semua kolom identik dalam berkas yang sama).

In [38]:
dup_off = off_tx.duplicated(subset=["berkas"] + kolom_off, keep="first")
print("Baris duplikat persis (salinan ke-2 dst.):", dup_off.sum())
off_tx[off_tx.duplicated(subset=["berkas"] + kolom_off, keep=False)].head(4)[["berkas", "baris_excel"] + kolom_off]

Baris duplikat persis (salinan ke-2 dst.): 404


,berkas,baris_excel,tanggal,no_nota,nama_produk,qty,harga_satuan,diskon,subtotal,catatan
137,rekap_penjualan_offline_2023-11.xlsx,139,2023-11-03 00:00:00,0074,roundup 486 sl,1,88500,4500,84000,NaN
138,rekap_penjualan_offline_2023-11.xlsx,140,2023-11-03 00:00:00,0074,roundup 486 sl,1,88500,4500,84000,NaN
176,rekap_penjualan_offline_2023-11.xlsx,178,2023-11-04 00:00:00,2311-0094,KCL Curah,4,12000,NaN,48000,NaN
177,rekap_penjualan_offline_2023-11.xlsx,179,2023-11-04 00:00:00,2311-0094,KCL Curah,4,12000,NaN,48000,NaN


Untuk mencocokkan nama barang antarbaris, penulisan nama perlu diseragamkan dulu. Di review awal terlihat nama yang sama ditulis dengan huruf besar/kecil dan spasi berbeda (`e-trek anti rayap` vs `E-Trek anti rayap`), jadi dibuat fungsi `normalisasi_teks`.

In [39]:
def normalisasi_teks(teks):
    # Huruf kecil, buang spasi di ujung, rapatkan spasi ganda: '  KCL   Curah ' -> 'kcl curah'.
    if kosong(teks):
        return ""
    return re.sub(r"\s+", " ", str(teks).strip().lower())


print(normalisasi_teks("  E-Trek   anti RAYAP "))

e-trek anti rayap


**h. Qty negatif: salah ketik tanda atau retur?** Dicek tiga hal: apakah subtotalnya positif (uang masuk), apakah ada keterangan, dan apakah ada nota asal (barang yang sama terjual positif di nota yang sama).

In [40]:
neg = off_tx[qty_num < 0].copy()
pos = off_tx[qty_num > 0]
nota4 = lambda s: s.astype("string").str[-4:]          # '2311-0769' dan '0769' dibandingkan 4 digit terakhir
penjualan_asal = set(zip(pos["bulan_berkas"], nota4(pos["no_nota"]), pos["nama_produk"].map(normalisasi_teks)))
neg["subtotal_positif"] = neg["subtotal"].map(parse_angka) > 0
neg["ada_keterangan"] = neg["catatan"].notna()
neg["ada_nota_asal"] = [(b, n, m) in penjualan_asal for b, n, m in
                        zip(neg["bulan_berkas"], nota4(neg["no_nota"]), neg["nama_produk"].map(normalisasi_teks))]
print("Qty negatif:", len(neg))
neg[["subtotal_positif", "ada_keterangan", "ada_nota_asal"]].value_counts()

Qty negatif: 58


subtotal_positif  ada_keterangan  ada_nota_asal
True              False           False            32
                  True            False            25
                  False           True              1
Name: count, dtype: int64

**i. Nama produk tidak baku.** Rekap offline tidak punya kolom SKU. Seberapa banyak nama yang persis sama dengan nama/kode di master?

In [41]:
kode_master = set(master_konversi["sku"].str.upper())
nama_master = set(master_kategori["nama"].map(normalisasi_teks))
nama_norm = off_tx["nama_produk"].map(normalisasi_teks)
print("Variasi penulisan nama produk  :", off_tx["nama_produk"].nunique(), "untuk", master_kategori["sku"].nunique(), "SKU dijual")
print("Baris dengan kode SKU persis   :", nama_norm.str.upper().isin(kode_master).sum())
print("Baris dengan nama master persis:", nama_norm.isin(nama_master).sum())
print("Baris yang tidak persis cocok  :", (~nama_norm.str.upper().isin(kode_master) & ~nama_norm.isin(nama_master)).sum())
print("\nContoh nama yang tidak persis cocok (terbanyak):")
off_tx.loc[~nama_norm.str.upper().isin(kode_master) & ~nama_norm.isin(nama_master), "nama_produk"].value_counts().head(12)

Variasi penulisan nama produk  : 1673 untuk 186 SKU dijual
Baris dengan kode SKU persis   : 7498
Baris dengan nama master persis: 40569
Baris yang tidak persis cocok  : 8694

Contoh nama yang tidak persis cocok (terbanyak):


nama_produk
Trobos 200         436
E-Trek anti        412
Emkape Maxxi       412
Superel 480        329
Kovin Plus         324
MKP Pak            302
Filia 525          246
Dangke 40          212
Avidor 25          208
Balistic 50        206
Winder 100         198
Polybag Tanaman    186
Name: count, dtype: int64

**Data Review `offline_raw`:**
- **Kolom bergeser**: ada 2 versi nama kolom (`Tgl/Jml/Total/Ket` di 9 berkas dan `Tanggal/Qty/Subtotal/Catatan` di 27 berkas). 17 berkas punya judul laporan, jadi header ada di baris 3. Isi per baris tidak bergeser: tipe isi tiap kolom konsisten.
- **Tidak ada sel gabungan.** Ada **54 baris bukan transaksi**: 37 `SUBTOTAL MINGGUAN` dan 17 `TOTAL`.
- **Tanggal campur**: 40.228 sel berupa tanggal Excel dan 15.494 berupa teks dengan banyak pola (`dd/mm/yyyy`, `d-m-yy`, `yyyy/mm/dd`, `12 Jan 2025`). Semuanya bisa dibaca. **51 baris salah tahun**: hari dan bulannya cocok dengan berkas, tapi tahunnya mundur satu.
- **Angka berupa teks**: 1.632 harga ditulis `Rp 112.500` atau `13,500`. Ada **628 harga kosong/0** dan **269 qty kosong**.
- **Nomor nota**: 15.941 hanya 4 digit (`0002`) dan **2.782 kosong**.
- **404 baris duplikat persis** pada data mentah (semua kolom identik, termasuk nota dan keterangan). Baris yang baru terlihat sama setelah nota dan tanggal dibersihkan diperiksa terpisah di tahap cleaning.
- **58 qty negatif**, semuanya dengan subtotal tetap positif. Hanya 1 dari 58 yang barangnya ada di nota yang sama. Retur pembeli sudah dicatat sendiri di stok masuk/keluar, dan subtotal positif berarti uangnya masuk, jadi baris ini adalah salah ketik tanda.
- **923 baris subtotal ≠ qty × harga − diskon**, umumnya selisih kecil (Rp500 sampai Rp1.000). Omzet offline memakai subtotal, jadi nilainya dipertahankan dan baris diberi tanda.
- **Nama produk tidak baku**: ada **1.673 variasi** untuk 186 SKU. Sebanyak 7.498 baris berisi kode SKU persis dan 40.569 baris berisi nama master persis (termasuk 13 nama yang dipakai 2 SKU beda ukuran, sehingga ukurannya harus dipilih dari harga). **8.694 baris (15,6%) tidak persis cocok**: nama terpotong (`Trobos 200`), nama ambigu (`Polybag Tanaman` cocok ke 8 ukuran), dan salah ketik (`KCLC urah`, `Boitek`). Rinciannya per aturan terlihat di langkah Cleaning 5.5.

### 4.3 Menilai `online_raw`

#### Review awal

In [42]:
review_awal(online_raw, "online_raw")

===== online_raw: 5220 baris, 27 kolom isi =====
1. Info & tipe data
<class 'pandas.DataFrame'>
RangeIndex: 5220 entries, 0 to 5219
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   no                    5184 non-null   object
 1   tanggal_pesanan       5184 non-null   object
 2   no_order              5184 non-null   object
 3   daerah                5184 non-null   object
 4   no_hp                 3119 non-null   object
 5   sku_barang            5003 non-null   object
 6   nama_produk           5184 non-null   object
 7   harga                 5184 non-null   object
 8   qty                   5184 non-null   object
 9   bruto                 5184 non-null   object
 10  diskon_persen         5184 non-null   object
 11  rp_diskon_platform    367 non-null    object
 12  rp_diskon             5184 non-null   object
 13  netto                 5184 non-null   object
 14  total                 5184 non

,count,unique,top,freq
no,5184,192,1,60
tanggal_pesanan,5184,1578,2026-10-23 00:00:00,25
no_order,5184,4128,NO ORDER,22
daerah,5184,207,KAB. MALANG,429
no_hp,3119,2046,NO HP,22
sku_barang,5003,313,EXTRAPCKG,316
nama_produk,5184,353,extra kardus + bubblewrap Ready Stock,289
harga,5184,441,4252,212
qty,5184,7,1,4251
bruto,5184,766,4252,216


In [43]:
print("Jenis isi sel (kolom yang dibutuhkan pertanyaan bisnis):")
display(tipe_isi(online_raw, ["tanggal_pesanan", "no_order", "sku_barang", "qty", "harga", "diskon_persen", "netto", "komisi_platform"]))
coba_jadi_angka(online_raw, ["harga", "qty", "bruto", "diskon_persen", "rp_diskon_platform", "rp_diskon", "netto", "total",
                             "ongkir", "komisi_platform", "komisi_affiliator", "laba_kotor", "harga_kulakan", "laba_bersih",
                             "laba_bersih_total"])

Jenis isi sel (kolom yang dibutuhkan pertanyaan bisnis):


,tanggal_pesanan,no_order,sku_barang,qty,harga,diskon_persen,netto,komisi_platform
datetime,3692,0,0,0,0,0,0,0
float,0,0,0,0,0,4444,3966,0
int,0,0,0,5162,5067,571,1196,5108
kosong,36,36,217,36,36,36,36,90
str,1492,5184,5003,22,117,169,22,22


,terisi,gagal_jadi_angka,contoh_gagal,min,max,negatif
kolom,,,,,,
harga,5184,117,"[Rp 78.168, Rp 28.870, Rp 39.550, Rp 191.468]",0.0,191468.00,0
qty,5184,22,[QTY],1.0,10.00,0
bruto,5184,22,[BRUTO],3850.0,957340.00,0
diskon_persen,5184,169,"[8%, 15%, 12%, 10%]",0.0,0.15,0
rp_diskon_platform,367,22,[RP DISKON PLATFORM],506.0,2978.00,0
rp_diskon,5184,22,[RP DISKON],0.0,114880.80,0
netto,5184,22,[NETTO],928.8,842459.20,0
total,5184,22,[TOTAL],928.8,842459.20,0
ongkir,5184,22,[ONGKIR],0.0,22000.00,0


In [44]:
online_raw.sample(5, random_state=1)

,no,tanggal_pesanan,no_order,daerah,no_hp,sku_barang,nama_produk,harga,qty,bruto,diskon_persen,rp_diskon_platform,rp_diskon,netto,total,ongkir,promo_ongkir,biaya_program_ongkir,sisa_ongkir_pembeli,komisi_platform,komisi_affiliator,laba_kotor,harga_kulakan,biaya_packing,laba_bersih,biaya_balik_kulakan,laba_bersih_total,baris_excel,baris_header,platform,bulan_laporan,berkas
3617,98,20/06/2026,515224806316604238,KOTA MALANG,08XX-XXXX-8807,EXTRAPCKG,extra kardus + bubblewrap Ready Stock,4252,1,4252,0.08,2567,340.2,1344.8,1344.8,0,0,40,0,134,537,633.8,2000,0,-1366.2,2000,NaN,99,1,TikTokShop,2026-06,rekap_online_TikTokShop_2026-06.xlsx
1821,100,2026-07-20 00:00:00,260720P03NHCRS,"MALANG, Jawa Timur",NaN,EXTRAPCKG,extra kardus + bubblewrap Ready Stock,4252,1,4252,0.12,NaN,510.2,3741.8,3741.8,0,0,110,0,299,0,3332.8,2000,0,1332.8,2000,NaN,104,1,Shopee,2026-07,rekap_online_Shopee_2026-07.xlsx
778,101,2025-11-26 00:00:00,251126X47M1VLR,KAB. MALANG,08XX-XXXX-7456,MST 2L,Misty Hand Sprayer ORIGINAL,45963,1,45963,0.12,NaN,5515.6,40447.4,40447.4,0,0,1470,0,3236,0,35741.4,28700,1000,6041.4,28700,NaN,102,1,Shopee,2025-11,rekap_online_Shopee_2025-11.xlsx
3991,58,9 Sep 2026,527833456747368644,KAB. SIDOARJO,08XX-XXXX-3654,DANGKE100,Dangke 40 WP,38798,1,38798,0.15,NaN,5819.7,32978.3,32978.3,0,0,1060,0,3298,4291,24329.3,23000,1000,329.3,23000,NaN,59,1,TikTokShop,2026-09,rekap_online_TikTokShop_2026-09.xlsx
1230,7,2026-04-01 00:00:00,260401E421ABG3,kab. lampung selatan,08XX-XXXX-3657,FSTC,Fastac 15 EC,25095,1,25095,0.05,NaN,1254.8,23840.2,23840.2,0,0,0,0,1907,0,21933.2,16000,1000,4933.2,16000,NaN,10,3,Shopee,2026-04,rekap_online_Shopee_2026-04.xlsx


**Gejala dari review awal `online_raw`:**
1. **36 baris kosong** di hampir semua kolom. Selain itu, teks judul kolom muncul sebagai isi (`NO ORDER`, `QTY`, `LABA BERSIH` masing-masing 22 kali) dan `harga_kulakan` berisi teks `TOTAL`. Kemungkinan ada baris kosong, header yang terulang, dan baris total di tengah file. → cek **a**
2. `laba_bersih_total` hanya terisi **1.468 dari 5.184 baris**, kemungkinan karena sel gabungan. → cek **b**
3. `no_order` seluruhnya teks (5.184 sel `str`), jadi aman dari pembulatan angka; panjangnya per kanal tetap perlu dicek karena nomor TikTok Shop 18 digit. → cek **c**
4. `tanggal_pesanan` berisi 3.692 tanggal Excel dan 1.492 teks (`20/06/2026`, `9 Sep 2026`).
5. `harga`: **117 isi gagal diubah ke angka** (`Rp 78.168`) dan ada harga **0**; `diskon_persen`: 169 isi berupa teks persen (`12%`); `qty` maksimum 10 padahal hampir semuanya 1. Kolom-kolom ini dipakai untuk menghitung netto (omzet Q1) dan margin (Q3). → cek **d** dan **e**
6. `komisi_platform` kosong di **54 baris** di luar baris kosong (90 − 36). → cek **e**
7. **16 baris duplikat.** → cek **f**
8. `sku_barang` kosong di **181 baris** di luar baris kosong, dan punya 313 nilai unik untuk produk yang di master hanya 190 SKU. → cek **g**
9. `total` dan `netto` punya jumlah nilai unik dan nilai teratas yang sama, begitu juga `biaya_balik_kulakan` dan `harga_kulakan`, jadi mungkin kolom ganda. `no_hp` adalah data pribadi pembeli, dan nama kanal diambil dari nama file (`TikTokShop`). → cek **h**
10. `laba_bersih` negatif di 302 baris (contoh di sampel: biaya kardus tambahan `EXTRAPCKG`). Ini hasil rumus UMKM sendiri, bukan kesalahan input; dicek bersama rumus di **e**.

#### Pengecekan lanjutan

**a. Posisi header, baris kosong, header terulang, dan baris TOTAL.**

In [45]:
print(info_header_online["baris_header"].value_counts().rename("jumlah_berkas"))
kosong_on = online_raw[KOLOM_ISI_ONLINE].isna().all(axis=1)
header_ulang = online_raw["no"].astype("string").str.strip().str.upper().eq("NO").fillna(False).astype(bool)
total_on = online_raw["no_order"].isna() & ~kosong_on & ~header_ulang
print("Baris kosong                      :", kosong_on.sum())
print("Baris header terulang di tengah   :", header_ulang.sum())
print("Baris TOTAL (tanpa nomor pesanan) :", total_on.sum())

baris_header
1    45
3    14
Name: jumlah_berkas, dtype: int64
Baris kosong                      : 22
Baris header terulang di tengah   : 22
Baris TOTAL (tanpa nomor pesanan) : 14


**b. Sel gabungan.** Di Excel, sel gabungan hanya menyimpan nilai di sel pertama; baris lain terbaca kosong.

In [46]:
gabungan = {}
for path in berkas_online:
    for rentang in load_workbook(path).worksheets[0].merged_cells.ranges:
        kolom_excel = re.match(r"[A-Z]+", str(rentang))[0]
        gabungan[kolom_excel] = gabungan.get(kolom_excel, 0) + 1
print("Sel gabungan per kolom Excel:", gabungan, "(kolom AA = LABA BERSIH TOTAL)")
on_tx = online_raw[~kosong_on & ~header_ulang & ~total_on].copy()
print("LABA BERSIH TOTAL terisi:", on_tx["laba_bersih_total"].notna().sum(), "dari", len(on_tx), "baris")

Sel gabungan per kolom Excel: {'AA': 1107} (kolom AA = LABA BERSIH TOTAL)
LABA BERSIH TOTAL terisi: 1466 dari 5162 baris


**c. Nomor pesanan 18 digit (TikTok Shop).** Jika terbaca sebagai angka, digit terakhirnya rusak karena batas presisi float.

In [47]:
tipe_sel = {"angka": 0, "teks": 0}
for path in berkas_online:
    for (sel,) in load_workbook(path).worksheets[0].iter_rows(min_row=2, min_col=3, max_col=3):
        if sel.value is not None and sel.value != "NO ORDER":
            tipe_sel["angka" if sel.data_type == "n" else "teks"] += 1
print("Tipe sel kolom NO ORDER di Excel:", tipe_sel)
print(on_tx.groupby("platform")["no_order"].apply(lambda s: s.astype(str).str.len().value_counts().to_dict()))
contoh = on_tx.loc[on_tx["platform"] == "TikTokShop", "no_order"].iloc[0]
print("\nContoh nomor asli        :", contoh)
print("Jika dibaca sebagai float :", int(float(contoh)), "<- digit terakhir berubah")

Tipe sel kolom NO ORDER di Excel: {'angka': 0, 'teks': 5162}
platform      
Shopee      14    2338.0
            18       NaN
            26       NaN
TikTokShop  14       NaN
            18    1920.0
            26       NaN
Tokopedia   14       NaN
            18       NaN
            26     904.0
Name: no_order, dtype: float64

Contoh nomor asli        : 584464531018224347
Jika dibaca sebagai float : 584464531018224384 <- digit terakhir berubah


**d. Tipe isi sel setelah baris kosong, header terulang, dan TOTAL dikeluarkan** (gejala 4 dan 5). Pengecekan berikutnya memakai `on_tx` (baris pesanan saja).

In [48]:
tipe_isi(on_tx, ["tanggal_pesanan", "no_order", "sku_barang", "harga", "qty", "diskon_persen",
                 "rp_diskon_platform", "netto", "komisi_platform", "harga_kulakan"])

,tanggal_pesanan,no_order,sku_barang,harga,qty,diskon_persen,rp_diskon_platform,netto,komisi_platform,harga_kulakan
datetime,3692,0,0,0,0,0,0,0,0,0
float,0,0,0,0,0,4444,0,3966,0,0
int,0,0,0,5067,5162,571,345,1196,5108,5162
kosong,0,0,181,0,0,0,4817,0,54,0
str,1470,5162,4981,95,0,147,0,0,0,0


Di review awal online, `diskon_persen` juga berisi teks persen (`12%`, gejala 5). `parse_angka` yang dibuat di bagian offline belum bisa membaca persen, jadi fungsinya **diperluas**: teks yang diakhiri `%` dibagi 100. Isi offline tidak ada yang berakhiran `%`, jadi hasil offline tidak berubah.

In [49]:
def parse_angka(nilai):
    # Ubah angka teks menjadi float: 'Rp 112.500' -> 112500, '13,500' -> 13500, '12%' -> 0.12.
    if kosong(nilai):
        return np.nan
    if isinstance(nilai, (int, float, np.integer, np.floating)):
        return float(nilai)
    teks = str(nilai).strip()
    if teks.endswith("%"):
        return float(teks[:-1].replace(",", ".")) / 100
    if teks.lower().startswith("rp"):
        return float(re.sub(r"[^\d]", "", teks))
    if re.fullmatch(r"-?\d{1,3}(,\d{3})+", teks):
        return float(teks.replace(",", ""))
    try:
        return float(teks)
    except ValueError:
        return np.nan


print(parse_angka("12%"), parse_angka("Rp 78.168"))

0.12 78168.0


**e. Cek rumus kolom turunan.** Rumus di file UMKM:
- `BRUTO = HARGA × QTY`, `RP DISKON = BRUTO × DISKON`
- `NETTO = BRUTO − RP DISKON − RP DISKON PLATFORM`
- `LABA KOTOR = NETTO − BIAYA PROGRAM ONGKIR − KOMISI PLATFORM − KOMISI AFFILIATOR`
- `LABA BERSIH = LABA KOTOR − HARGA KULAKAN − BIAYA PACKING` (HARGA KULAKAN sudah total per baris)

In [50]:
n = on_tx[["harga", "qty", "bruto", "diskon_persen", "rp_diskon_platform", "rp_diskon", "netto",
           "biaya_program_ongkir", "komisi_platform", "komisi_affiliator", "laba_kotor",
           "harga_kulakan", "biaya_packing", "laba_bersih"]].apply(lambda s: s.map(parse_angka))
n["rp_diskon_platform"] = n["rp_diskon_platform"].fillna(0)
cek_rumus = {
    "BRUTO = HARGA x QTY": (n["harga"] * n["qty"] - n["bruto"]).abs() > 1,
    "RP DISKON = BRUTO x DISKON": (n["bruto"] * n["diskon_persen"] - n["rp_diskon"]).abs() > 1,
    "NETTO": (n["bruto"] - n["rp_diskon"] - n["rp_diskon_platform"] - n["netto"]).abs() > 1,
    "LABA KOTOR": (n["netto"] - n["biaya_program_ongkir"] - n["komisi_platform"].fillna(0) - n["komisi_affiliator"] - n["laba_kotor"]).abs() > 1,
    "LABA BERSIH": (n["laba_kotor"] - n["harga_kulakan"] - n["biaya_packing"] - n["laba_bersih"]).abs() > 1,
}
print(pd.Series({k: v.sum() for k, v in cek_rumus.items()}, name="baris_tidak_cocok"))
print("\nContoh HARGA teks :", on_tx.loc[on_tx["harga"].map(lambda v: isinstance(v, str)), "harga"].head(3).tolist())
print("Contoh DISKON teks:", on_tx.loc[on_tx["diskon_persen"].map(lambda v: isinstance(v, str)), "diskon_persen"].unique()[:5].tolist())

BRUTO = HARGA x QTY            8
RP DISKON = BRUTO x DISKON     0
NETTO                          0
LABA KOTOR                    54
LABA BERSIH                    0
Name: baris_tidak_cocok, dtype: int64

Contoh HARGA teks : ['Rp 78.168', 'Rp 28.870', 'Rp 39.550']
Contoh DISKON teks: ['8%', '15%', '12%', '10%', '0%']


In [51]:
print("HARGA = 0:", (n["harga"] == 0).sum())
qty_salah = (n["harga"] > 0) & (n["harga"] * n["qty"] != n["bruto"])
on_tx.loc[qty_salah, ["berkas", "baris_excel", "sku_barang", "harga", "qty", "bruto", "harga_kulakan"]]

HARGA = 0: 4


,berkas,baris_excel,sku_barang,harga,qty,bruto,harga_kulakan
985,rekap_online_Shopee_2026-01.xlsx,77,HUMIGROW500,68750,10,68750,41000
1601,rekap_online_Shopee_2026-06.xlsx,23,NPKM1,35228,10,35228,20250
3550,rekap_online_TikTokShop_2026-06.xlsx,32,AGT,24938,10,24938,9000
4669,rekap_online_Tokopedia_2025-12.xlsx,33,FILIA50,38054,10,38054,26750


In [52]:
# Komisi kosong: apakah bisa dihitung balik dari rumus laba kotor?
komisi_balik = n["netto"] - n["biaya_program_ongkir"] - n["komisi_affiliator"] - n["laba_kotor"]
ada = n["komisi_platform"].notna()
print("KOMISI PLATFORM kosong                 :", (~ada).sum())
print("Laba kotor terisi di baris komisi kosong:", n.loc[~ada, "laba_kotor"].notna().sum())
print("Baris komisi terisi yang SAMA dengan hitung balik (selisih <= Rp1):",
      ((komisi_balik - n["komisi_platform"]).abs() <= 1)[ada].sum(), "dari", ada.sum())

KOMISI PLATFORM kosong                 : 54
Laba kotor terisi di baris komisi kosong: 54
Baris komisi terisi yang SAMA dengan hitung balik (selisih <= Rp1): 5108 dari 5108


**f. Duplikat dan pesanan yang terulang di file bulan berikutnya.**

In [53]:
dup_on = on_tx.duplicated(subset=["berkas"] + KOLOM_ISI_ONLINE, keep="first")
print("Duplikat persis (semua kolom mentah identik):", dup_on.sum())
kolom_inti = ["berkas", "no", "no_order", "sku_barang", "nama_produk", "qty", "bruto", "netto"]
mirip = on_tx.astype({"no": "string"}).duplicated(subset=kolom_inti, keep="first") & ~dup_on
print("Sama di kolom inti tapi beda format di kolom lain:", mirip.sum())

tgl_on = on_tx["tanggal_pesanan"].map(parse_tanggal)
beda_bulan = tgl_on.dt.strftime("%Y-%m") != on_tx["bulan_laporan"]
print("Baris dengan tanggal pesanan di luar bulan laporan:", beda_bulan.sum())
contoh_order = on_tx.loc[beda_bulan, "no_order"].iloc[0]
on_tx.loc[on_tx["no_order"] == contoh_order, ["berkas", "baris_excel", "tanggal_pesanan", "no_order", "sku_barang", "qty", "netto"]]

Duplikat persis (semua kolom mentah identik): 16
Sama di kolom inti tapi beda format di kolom lain: 11
Baris dengan tanggal pesanan di luar bulan laporan: 82


,berkas,baris_excel,tanggal_pesanan,no_order,sku_barang,qty,netto
42,rekap_online_Shopee_2025-02.xlsx,44,27 Feb 2025,250227UZLHTFND,SAPPORO100,2,153013.8
44,rekap_online_Shopee_2025-03.xlsx,2,2025-02-27 00:00:00,250227UZLHTFND,SAPPORO100,2,153013.8


**g. SKU di rekap online.**

In [54]:
sku_on = on_tx["sku_barang"]
print("SKU kosong                     :", sku_on.isna().sum())
print("SKU persis ada di master       :", sku_on.isin(master_konversi["sku"]).sum())
print("SKU beda huruf besar/kecil saja:", (~sku_on.isin(master_konversi["sku"]) & sku_on.str.upper().isin(kode_master)).sum())
lain = sku_on[sku_on.notna() & ~sku_on.str.upper().isin(kode_master)]
print("SKU tidak dikenal (salah ketik):", len(lain), lain.unique()[:10].tolist())
print("Contoh nama listing:", on_tx["nama_produk"].dropna().sample(5, random_state=1).tolist())

SKU kosong                     : 181
SKU persis ada di master       : 4737
SKU beda huruf besar/kecil saja: 144
SKU tidak dikenal (salah ketik): 100 ['SPONTANING', 'TSPN', 'DSCK', 'AMNIONANO', 'ATNRCL', 'WINDR100', 'BSLTC', 'GSDLB', 'KOVINLPUS', 'KINLGON100']
Contoh nama listing: ['Jaring Buah Besar', 'Dangke 40 WP Asli Bergaransi', 'Pegasus 500 SC', 'Green Stick - Murah', 'Mipcinta 50 WP ORIGINAL']


**h. Kolom yang tidak dipakai dan nama kanal.**

In [55]:
total_num = on_tx["total"].map(parse_angka)
balik_num = on_tx["biaya_balik_kulakan"].map(parse_angka)
print("TOTAL sama dengan NETTO                 :", ((total_num - n["netto"]).abs() <= 1).sum(), "dari", len(on_tx))
print("BIAYA BALIK KULAKAN sama dengan KULAKAN :", ((balik_num - n["harga_kulakan"]).abs() <= 1).sum(), "dari", len(on_tx))
print("NO HP terisi (data pribadi pembeli)     :", on_tx["no_hp"].notna().sum())
print("DAERAH terisi                           :", on_tx["daerah"].notna().sum(), "(tidak dipakai di Q1-Q12)")
print("Nama kanal dari nama file               :", sorted(online_raw["platform"].unique()))

TOTAL sama dengan NETTO                 : 5162 dari 5162
BIAYA BALIK KULAKAN sama dengan KULAKAN : 5162 dari 5162
NO HP terisi (data pribadi pembeli)     : 3097
DAERAH terisi                           : 5162 (tidak dipakai di Q1-Q12)
Nama kanal dari nama file               : ['Shopee', 'TikTokShop', 'Tokopedia']


**Data Review `online_raw`:**
- 14 berkas punya judul laporan (header di baris 3). Ada **22 baris kosong**, **22 header terulang** di tengah file, dan **14 baris TOTAL**.
- **1.107 sel gabungan**, semuanya di kolom AA `LABA BERSIH TOTAL`, sehingga hanya 1.466 dari 5.162 baris yang terisi. Kolom ini turunan dari `LABA BERSIH`, jadi tidak dipakai dan laba dihitung ulang per baris.
- **Nomor pesanan**: semua 5.162 sel tersimpan sebagai **teks** di Excel, jadi tidak ada yang rusak. Panjangnya konsisten: Shopee 14, Tokopedia 26, TikTok Shop 18. Contoh di atas menunjukkan risikonya: jika nomor 18 digit terbaca sebagai angka, digit terakhirnya berubah. Karena itu semua file dibaca dengan `dtype=object` dan nomor pesanan disimpan sebagai teks.
- **Angka berupa teks**: 95 `HARGA` (`Rp 78.168`) dan 147 `DISKON` (`12%`).
- **Rumus UMKM terbukti konsisten**. Pengecualian: 8 baris BRUTO ≠ HARGA × QTY (4 baris **HARGA = 0** dan 4 baris **QTY = 10** padahal BRUTO = HARGA dan HARGA KULAKAN sesuai 1 unit), serta 54 baris LABA KOTOR yang komisinya kosong.
- **54 komisi kosong, tetapi laba kotornya terisi.** Di 5.108 baris yang komisinya terisi, komisi selalu sama dengan `netto − biaya program ongkir − komisi affiliator − laba kotor`. Jadi komisi yang kosong bisa **dihitung balik** dengan pasti, tanpa tarif asumsi.
- **16 duplikat persis** pada data mentah (semua kolom identik, termasuk nomor baris `NO`). Ada 11 baris lain yang kolom intinya sama tetapi format kolom lainnya berbeda (mis. tanggal teks vs tanggal Excel); setelah dibersihkan baris ini menjadi sama dan salinannya dibuang.
- **82 baris bertanggal di luar bulan laporan**: pesanan akhir bulan muncul lagi di file bulan berikutnya. Pencocokan memakai kunci **No Order (teks) + SKU**.
- **SKU**: 181 kosong, 144 hanya beda huruf besar/kecil, dan 100 salah ketik (`TSPN`, `ATNRCL`, ...).
- **Kolom tidak dipakai**: `TOTAL` selalu sama dengan `NETTO` dan `BIAYA BALIK KULAKAN` selalu sama dengan `HARGA KULAKAN` (5.162 dari 5.162 baris). `NO HP` (3.097 terisi) adalah data pribadi pembeli, dan `DAERAH` tidak dipakai di Q1–Q12.
- **Nama kanal** diambil dari nama file (`TikTokShop`), sedangkan metrik marketplace menulis `TikTok Shop`. Tanpa dibakukan, join antartabel untuk Q12 akan gagal diam-diam.

### 4.4 Menilai `stok_mk_raw` (restok, retur, rusak)

#### Review awal

In [56]:
review_awal(stok_mk_raw, "stok_mk_raw")

===== stok_mk_raw: 4234 baris, 7 kolom isi =====
1. Info & tipe data
<class 'pandas.DataFrame'>
RangeIndex: 4234 entries, 0 to 4233
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   tanggal     4234 non-null   object
 1   sku_teks    4234 non-null   object
 2   jenis       4234 non-null   object
 3   qty_teks    4234 non-null   object
 4   harga_beli  2009 non-null   object
 5   supplier    2512 non-null   object
 6   keterangan  2247 non-null   object
dtypes: object(7)
memory usage: 231.7+ KB

2. Missing values per kolom
tanggal          0
sku_teks         0
jenis            0
qty_teks         0
harga_beli    2225
supplier      1722
keterangan    1987

3. Jumlah duplikat (semua kolom isi sama, dalam berkas/sheet yang sama): 29

4. Jumlah nilai unik per kolom
tanggal       2048
sku_teks       341
jenis           11
qty_teks        50
harga_beli     575
supplier         7
keterangan       5

5. Ringkasan statistik


,count,unique,top,freq
tanggal,4234,2048,2025-02-02 00:00:00,10
sku_teks,4234,341,ETREKKCL,62
jenis,4234,11,RESTOK,916
qty_teks,4234,50,10,1149
harga_beli,2009,575,10800,23
supplier,2512,7,SUP-04,841
keterangan,2247,5,kiriman,893


In [57]:
display(tipe_isi(stok_mk_raw, ["tanggal", "sku_teks", "jenis", "qty_teks", "harga_beli"]))
display(coba_jadi_angka(stok_mk_raw, ["qty_teks", "harga_beli"]))
stok_mk_raw.sample(6, random_state=1)

,tanggal,sku_teks,jenis,qty_teks,harga_beli
datetime,3022,0,0,0,0
int,0,0,0,648,2009
kosong,0,0,0,0,2225
str,1212,4234,4234,3586,0


,terisi,gagal_jadi_angka,contoh_gagal,min,max,negatif
kolom,,,,,,
qty_teks,4234,952,"[1 karung, 3 karung, 100 pcs, 20 pcs]",1.0,150.0,0
harga_beli,2009,0,[],2400.0,666600.0,0


,tanggal,sku_teks,jenis,qty_teks,harga_beli,supplier,keterangan,baris_excel,baris_header,sheet,berkas
330,2024-02-05 00:00:00,EMKAPE1,Retur,1,NaN,NaN,retur offline,332,1,2023-2024,stok_masuk_keluar.xlsx[2023-2024]
4072,2026-09-29 00:00:00,CPRCD100,RESTOK,20,NaN,SUP-05,NaN,1154,1,2026,stok_masuk_keluar.xlsx[2026]
589,2024-04-28 00:00:00,GRNSTIK,Barang masuk,20,NaN,SUP-07,nota supplier,591,1,2023-2024,stok_masuk_keluar.xlsx[2023-2024]
2136,2025-07-02 00:00:00,MGSULFATE,restock,20 pcs,13700,SUP-02,NaN,693,1,2025,stok_masuk_keluar.xlsx[2025]
3722,2026-07-04 00:00:00,RMPS100,Barang masuk,10,44700,SUP-06,NaN,804,1,2026,stok_masuk_keluar.xlsx[2026]
3009,2026-01-19 00:00:00,STRBN100,Barang masuk,10,NaN,NaN,nota supplier,91,1,2026,stok_masuk_keluar.xlsx[2026]


**Gejala dari review awal `stok_mk_raw`:**
1. `tanggal` berisi 3.022 tanggal Excel dan 1.212 teks. Data dibagi per sheet tahun, jadi urutan dan tahunnya perlu dicek. → cek **d**
2. `jenis` punya **11 nilai unik**, padahal jenis pergerakan seharusnya hanya restok, retur, dan rusak (`RESTOK`, `restock`, `Barang masuk`, `Retur`, ...). → cek **b**
3. `qty_teks`: **952 isi gagal diubah ke angka** karena berisi satuan (`1 karung`, `100 pcs`). Satuan menentukan jumlah stok (Q4–Q6). → cek **b**
4. `harga_beli` kosong di **2.225 baris** dan `supplier` kosong di 1.722 baris. Perlu dibedakan antara retur (memang tanpa harga) dan restok yang tidak mencatat harga/supplier. → cek **c**
5. `sku_teks` punya **341 nilai unik** untuk master 190 SKU, jadi ada kode yang ditulis berbeda. → cek **c**
6. **29 baris duplikat.** → cek **c**

#### Pengecekan lanjutan

**a. Nama kolom per sheet.**

In [58]:
for sheet, kolom in kolom_asli_stok_mk.items():
    print(f"{sheet:10s}: {kolom}")

2023-2024 : ['Tanggal', 'SKU', 'Jenis', 'Qty', 'Harga Beli Satuan', 'Supplier', 'Keterangan']
2025      : ['Tgl', 'Kode', 'Jenis', 'Jumlah', 'Harga Beli', 'Supplier', 'Ket']
2026      : ['Tanggal', 'SKU', 'Jenis', 'Qty', 'Harga Beli Satuan', 'Supplier', 'Keterangan']


**b. Variasi penulisan jenis dan qty** (gejala 2 dan 3).

In [59]:
print("Variasi penulisan jenis:")
print(stok_mk_raw["jenis"].value_counts().to_string())
pola_qty = stok_mk_raw["qty_teks"].map(lambda v: re.sub(r"\d+", "9", str(v)))
print("\nPola penulisan qty:", pola_qty.value_counts().to_dict())
induk_mk = stok_mk_raw[stok_mk_raw["sku_teks"].astype(str).str.contains("50KG")]
print("Qty SKU induk 50 kg:", induk_mk["qty_teks"].astype(str).value_counts().to_dict())

Variasi penulisan jenis:
jenis
RESTOK              916
Barang masuk        901
restock             892
Restok              877
Retur               211
RETUR               202
retur pembeli       192
RUSAK                13
Rusak                12
Rusak/Kadaluarsa     10
kadaluarsa            8

Pola penulisan qty: {'9': 3282, '9 pcs': 840, '9 karung': 66, '9 kg': 46}
Qty SKU induk 50 kg: {'1 karung': 23, '3 karung': 23, '150 kg': 18, '2 karung': 16, '50 kg': 15, '3': 11, '2': 8, '1': 8, '100 kg': 7, '4': 3, '4 karung': 3, '200 kg': 3}


Qty ditulis sebagai angka dengan satuan (`1 karung`, `20 pcs`), dan di pemecahan juga `50.0 kg` dan `2 bks`. Satuannya penting untuk stok SKU induk (karung vs kg), jadi angka dan satuan dipisahkan dengan fungsi `parse_qty`, bukan hanya diambil angkanya.

In [60]:
def parse_qty(nilai):
    # Pisahkan angka dan satuan: '20 pcs' -> (20.0, 'pcs'), '50.0 kg' -> (50.0, 'kg'), 5 -> (5.0, '').
    if kosong(nilai):
        return np.nan, ""
    if isinstance(nilai, (int, float, np.integer, np.floating)):
        return float(nilai), ""
    m = re.fullmatch(r"\s*(-?\d+(?:\.\d+)?)\s*([A-Za-z]*)\s*", str(nilai))
    if not m:
        return np.nan, ""
    return float(m[1]), m[2].lower()


print(parse_qty("3 karung"), parse_qty("50.0 kg"), parse_qty(10), parse_qty("2 bks"))

(3.0, 'karung') (50.0, 'kg') (10.0, '') (2.0, 'bks')


**c. Duplikat, restok tanpa harga/supplier, retur, dan SKU** (gejala 4, 5, 6).

In [61]:
kolom_stok = ["tanggal", "sku_teks", "jenis", "qty_teks", "harga_beli", "supplier", "keterangan"]
print("Duplikat persis          :", stok_mk_raw.duplicated(subset=["berkas"] + kolom_stok).sum())
is_restok = stok_mk_raw["jenis"].str.lower().str.contains("restok|restock|masuk")
print("Restok tanpa harga beli  :", (is_restok & stok_mk_raw["harga_beli"].isna()).sum(), "dari", is_restok.sum())
print("Restok tanpa supplier    :", (is_restok & stok_mk_raw["supplier"].isna()).sum())
print("Baris retur              :", stok_mk_raw["jenis"].str.lower().str.contains("retur").sum(),
      "(tidak ada kolom nomor nota, jadi semua retur tanpa nota asal)")
tidak_di_master = stok_mk_raw.loc[~stok_mk_raw["sku_teks"].isin(master_konversi["sku"]), "sku_teks"]
print("Kode SKU tidak persis di master:", len(tidak_di_master))
tidak_di_master.value_counts().head(10)

Duplikat persis          : 29
Restok tanpa harga beli  : 1577 dari 3586
Restok tanpa supplier    : 1074
Baris retur              : 605 (tidak ada kolom nomor nota, jadi semua retur tanpa nota asal)
Kode SKU tidak persis di master: 243


sku_teks
TMA 6 865 SL                 4
Trobos 200 SL                4
Kovin Plus 80P               4
Biotek                       4
Fertiphos Curah              3
CGN1                         3
Bio Calsium                  3
Magnesium Sulfat Pak Tani    3
Roundup 486 SL               3
Antracol 70 WP               3
Name: count, dtype: int64

**d. Urutan dan tahun tanggal** (gejala 1).

In [62]:
tgl_mk = stok_mk_raw["tanggal"].map(parse_tanggal)
turun = tgl_mk.groupby(stok_mk_raw["sheet"]).diff().dt.days < 0
print("Baris yang tanggalnya lebih awal dari baris sebelumnya:", turun.sum())
print("Tahun tanggal tidak sesuai nama sheet:", sum(str(t.year) not in s for t, s in zip(tgl_mk, stok_mk_raw["sheet"])))

Baris yang tanggalnya lebih awal dari baris sebelumnya: 568
Tahun tanggal tidak sesuai nama sheet: 0


**Data Review `stok_masuk_keluar`:**
- **Nama kolom berbeda di sheet 2025** (`Tgl`, `Kode`, `Jumlah`, `Harga Beli`, `Ket`).
- **11 variasi penulisan jenis** (`RESTOK`, `restock`, `Barang masuk`, `retur pembeli`, `Rusak/Kadaluarsa`, ...) yang sebenarnya hanya 3 jenis: masuk, retur, rusak.
- **Qty bercampur satuan**: 840 `pcs`, 66 `karung`, 46 `kg`. Untuk SKU induk 50 kg, ada 30 baris berupa angka tanpa satuan, nilainya 1 sampai 4. Kalau dibaca sebagai kg tidak masuk akal (pembelian per karung), jadi dibaca sebagai karung.
- **29 duplikat persis**.
- **Restok tanpa harga beli: 1.577 dari 3.586**, dan **tanpa supplier: 1.074**. Ini tidak mengganggu stok, tetapi riwayat harga beli hanya bisa diambil dari baris yang punya harga.
- **605 baris retur tanpa nota asal**: tidak ada kolom nomor nota, jadi retur tidak bisa ditautkan ke penjualan.
- **243 kode SKU tidak persis ada di master**: sebagian berupa nama produk (`TMA 6 865 SL`), sebagian salah ketik (`SUEREL100`). Tiga restok ditulis dengan nama kemasan hasil pemecahan (`Fertiphos Curah`, `KCL Curah`) tetapi qty-nya dalam kg/karung, padahal kemasan itu tidak pernah dibeli dari supplier.
- **568 baris tidak urut tanggal**, tetapi tahunnya selalu sesuai nama sheet. Tanggalnya dianggap benar dan yang teracak adalah posisi baris, jadi tabel cukup diurutkan menurut tanggal. Urutan baris tidak memengaruhi saldo stok, karena stok dihitung dari jumlah pergerakan sampai tanggal tertentu.

### 4.5 Menilai `pemecahan_raw`

#### Review awal

In [63]:
review_awal(pemecahan_raw, "pemecahan_raw")

===== pemecahan_raw: 708 baris, 6 kolom isi =====
1. Info & tipe data
<class 'pandas.DataFrame'>
RangeIndex: 708 entries, 0 to 707
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   tanggal         708 non-null    object
 1   barang_induk    708 non-null    object
 2   bahan_teks      649 non-null    object
 3   sku_hasil_teks  708 non-null    object
 4   kemasan_teks    708 non-null    object
 5   catatan         0 non-null      object
dtypes: object(6)
memory usage: 33.3+ KB

2. Missing values per kolom
tanggal             0
barang_induk        0
bahan_teks         59
sku_hasil_teks      0
kemasan_teks        0
catatan           708

3. Jumlah duplikat (semua kolom isi sama, dalam berkas/sheet yang sama): 27

4. Jumlah nilai unik per kolom
tanggal           583
barang_induk        6
bahan_teks          4
sku_hasil_teks     16
kemasan_teks        9
catatan             0

5. Ringkasan statistik


,count,unique,top,freq
tanggal,708,583,2024-10-24 00:00:00,4
barang_induk,708,6,Emkape Maxxi Agri,424
bahan_teks,649,4,1,372
sku_hasil_teks,708,16,EMKAPE500,413
kemasan_teks,708,9,2,329
catatan,0,0,NaN,NaN


In [64]:
display(coba_jadi_angka(pemecahan_raw, ["bahan_teks", "kemasan_teks"]))
pemecahan_raw.sample(6, random_state=1)

,terisi,gagal_jadi_angka,contoh_gagal,min,max,negatif
kolom,,,,,,
bahan_teks,649,54,"[50.0 kg, 1.0 kg]",1.0,50.0,0
kemasan_teks,708,177,"[100 pcs, 2 bks, 50 bks, 100 bks]",2.0,100.0,0


,tanggal,barang_induk,bahan_teks,sku_hasil_teks,kemasan_teks,catatan,baris_excel,baris_header,sheet,berkas
189,2024-09-15 00:00:00,Emkape Maxxi Agri,1,EMKAPE500,2,NaN,191,1,2023-2024,stok_pemecahan.xlsx[2023-2024]
111,2024-05-10 00:00:00,Emkape Maxxi Agri,1,EMKAPE500,2,NaN,113,1,2023-2024,stok_pemecahan.xlsx[2023-2024]
423,2025-09-12 00:00:00,Pupuk KCL Mahkota,50.0 kg,KCL,100 pcs,NaN,161,1,2025,stok_pemecahan.xlsx[2025]
428,2025-09-20 00:00:00,Kapur Pertanian Kebomas,50,DLMT,100 pcs,NaN,166,1,2025,stok_pemecahan.xlsx[2025]
665,22/08/2026,Emkape Maxxi Agri,NaN,EMKAPE500,2,NaN,159,1,2026,stok_pemecahan.xlsx[2026]
563,2026-03-27 00:00:00,Emkape Maxxi Agri,1,EMKAPE500,2,NaN,57,1,2026,stok_pemecahan.xlsx[2026]


**Gejala dari review awal `pemecahan_raw`:**
1. `bahan_teks` kosong di **59 baris** dan 54 isi berupa teks (`50.0 kg`); `kemasan_teks` 177 isi berupa teks (`100 pcs`, `2 bks`). → cek **b**
2. `barang_induk` hanya 6 nilai, tetapi `sku_hasil_teks` punya **16 nilai unik**. Seharusnya satu induk menghasilkan satu SKU kemasan, jadi kemungkinan ada salah ketik. → cek **a**
3. **27 baris duplikat**. Di pemecahan, baris kembar bisa saja asli (memecah barang yang sama dua kali sehari). → cek **b**
4. Format `tanggal` campur (`22/08/2026` di samping tanggal Excel). `catatan` selalu kosong.

#### Pengecekan lanjutan

**a. Barang induk dan SKU hasil** (gejala 2).

In [65]:
print(pemecahan_raw["barang_induk"].value_counts().to_string())
salah_sku = ~pemecahan_raw["sku_hasil_teks"].isin(master_konversi["sku"])
print("\nSKU hasil tidak persis di master:", salah_sku.sum(), pemecahan_raw.loc[salah_sku, "sku_hasil_teks"].unique().tolist())

barang_induk
Emkape Maxxi Agri          424
Pupuk KCL Mahkota          103
Kapur Pertanian Kebomas     99
Pupuk Urea Petro            44
Fertiphos Pak Tani          28
Kanitrat Maxxi Agri         10

SKU hasil tidak persis di master: 14 ['EKAPE500', 'DMLT', 'UEA', 'EMKAP500', 'EMKAP5E00', 'EMAKPE500', 'EMAPE500', 'EKMAPE500', 'EMKPE500', 'EMKAPE050']


**b. Bahan, kemasan, rumus tanpa susut, dan duplikat** (gejala 1 dan 3).

In [66]:
bahan = pemecahan_raw["bahan_teks"].map(lambda v: parse_qty(v)[0])
kemasan = pemecahan_raw["kemasan_teks"].map(lambda v: parse_qty(v)[0])
faktor = pemecahan_raw["sku_hasil_teks"].map(master_konversi.set_index("sku")["faktor_per_unit_jual"])
print("Bahan dipakai kosong          :", bahan.isna().sum())
print("Bahan berisi teks ('50.0 kg') :", pemecahan_raw["bahan_teks"].map(lambda v: isinstance(v, str)).sum())
print("Kemasan berisi teks ('2 bks') :", pemecahan_raw["kemasan_teks"].map(lambda v: isinstance(v, str)).sum())
cek = bahan.notna() & faktor.notna()
print("Bahan != kemasan x faktor (tanpa susut):", ((bahan - kemasan * faktor).abs() > 1e-6)[cek].sum(), "dari", cek.sum())
kolom_pecah = ["tanggal", "barang_induk", "bahan_teks", "sku_hasil_teks", "kemasan_teks", "catatan"]
print("Duplikat persis:", pemecahan_raw.duplicated(subset=["berkas"] + kolom_pecah).sum())

Bahan dipakai kosong          : 59
Bahan berisi teks ('50.0 kg') : 54
Kemasan berisi teks ('2 bks') : 177
Bahan != kemasan x faktor (tanpa susut): 0 dari 635
Duplikat persis: 27


**Data Review `stok_pemecahan`:**
- Ada 6 barang induk. Emkape (kemasan pabrik 1 kg → 2 × 500 g) paling sering dipecah: 424 kali.
- **14 SKU hasil salah ketik** (`EKAPE500`, `DMLT`, `UEA`, `EMKAP5E00`, ...). Setiap barang induk hanya dipecah ke **satu** SKU hasil, jadi kolom barang induk menyempitkan kandidatnya menjadi satu.
- **59 bahan dipakai kosong**, 54 bahan berupa teks (`50.0 kg`), dan 177 jumlah kemasan berisi satuan (`2 bks`).
- **Semua 635 baris** yang bahannya terisi memenuhi bahan = kemasan × faktor. Ini cocok dengan aturan tanpa susut, jadi bahan yang kosong bisa dihitung dengan rumus yang sama.
- **27 duplikat persis**.

### 4.6 Menilai `rakit_raw`

#### Review awal

In [67]:
review_awal(rakit_raw, "rakit_raw")

===== rakit_raw: 41 baris, 4 kolom isi =====
1. Info & tipe data
<class 'pandas.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   tanggal         41 non-null     object
 1   sku_paket_teks  41 non-null     object
 2   paket_teks      41 non-null     object
 3   catatan         0 non-null      object
dtypes: object(4)
memory usage: 1.4+ KB

2. Missing values per kolom
tanggal            0
sku_paket_teks     0
paket_teks         0
catatan           41

3. Jumlah duplikat (semua kolom isi sama, dalam berkas/sheet yang sama): 0

4. Jumlah nilai unik per kolom
tanggal           41
sku_paket_teks     6
paket_teks         5
catatan            0

5. Ringkasan statistik


,count,unique,top,freq
tanggal,41,41,22-11-23,1
sku_paket_teks,41,6,PAKETTUMBUH250,33
paket_teks,41,5,4,25
catatan,0,0,NaN,NaN


In [68]:
display(coba_jadi_angka(rakit_raw, ["paket_teks"]))
rakit_raw.head(6)

,terisi,gagal_jadi_angka,contoh_gagal,min,max,negatif
kolom,,,,,,
paket_teks,41,13,"[4 pcs, 5 pcs]",3.0,5.0,0


,tanggal,sku_paket_teks,paket_teks,catatan,baris_excel,baris_header,sheet,berkas
0,22-11-23,PAKETTUMBUH250,4 pcs,NaN,2,1,2023-2024,stok_rakit_paket.xlsx[2023-2024]
1,2023-12-07 00:00:00,PAKETTUMBUH250,4 pcs,NaN,3,1,2023-2024,stok_rakit_paket.xlsx[2023-2024]
2,2023-12-20 00:00:00,PAKETTUMBUH250,4,NaN,4,1,2023-2024,stok_rakit_paket.xlsx[2023-2024]
3,2023-12-27 00:00:00,PAKETTUMBUH250,4,NaN,5,1,2023-2024,stok_rakit_paket.xlsx[2023-2024]
4,2023-12-30 00:00:00,PAKETTUMBUH250,4 pcs,NaN,6,1,2023-2024,stok_rakit_paket.xlsx[2023-2024]
5,2024-01-16 00:00:00,PAKETTUMBUH250,4,NaN,7,1,2023-2024,stok_rakit_paket.xlsx[2023-2024]


**Gejala dari review awal `rakit_raw`:**
1. `sku_paket_teks` punya **6 nilai unik**, padahal master hanya punya 5 paket. → cek **a**
2. `paket_teks`: 13 isi berupa teks dengan satuan (`4 pcs`). → cek **a**
3. Format `tanggal` campur (`22-11-23`). Tidak ada duplikat.

#### Pengecekan lanjutan

**a. SKU paket, jumlah paket, duplikat, dan komposisi paket** (gejala 1 dan 2).

In [69]:
print(rakit_raw["sku_paket_teks"].value_counts().to_string())
print("\nJumlah paket berisi teks:", rakit_raw["paket_teks"].map(lambda v: isinstance(v, str)).sum())
print("Duplikat persis:", rakit_raw.duplicated(subset=["berkas", "tanggal", "sku_paket_teks", "paket_teks", "catatan"]).sum())
master_komposisi

sku_paket_teks
PAKETTUMBUH250    33
PAKETBUAH250       3
PAKETGDSL          2
PAKETUTMBUH250     1
PAKETGNRTF         1
PAKEGDSL           1

Jumlah paket berisi teks: 13
Duplikat persis: 0


,sku_paket,sku_komponen,jumlah_per_paket_dalam_satuan_jual_komponen,label
0,PAKETAG,AGB,1.00,DUMMY
1,PAKETAG,AGT,1.00,DUMMY
2,PAKETGDSL,GDSLB,1.00,DUMMY
3,PAKETGDSL,GDSLD,1.00,DUMMY
4,PAKETTUMBUH250,ULTRDP,0.25,DUMMY
5,PAKETTUMBUH250,NPKPT,0.25,DUMMY
6,PAKETTUMBUH250,MGSULFATE,0.25,DUMMY
7,PAKETBUAH250,EMKAPE1,0.25,DUMMY
8,PAKETBUAH250,KNOPTH1,0.25,DUMMY
9,PAKETBUAH250,KRTBRN,0.25,DUMMY


**Data Review `stok_rakit_paket`:**
- 41 kali rakit, didominasi `PAKETTUMBUH250` (33 kali).
- **2 SKU paket salah ketik** (`PAKETUTMBUH250`, `PAKEGDSL`), 13 jumlah berisi satuan (`4 pcs`), dan tidak ada duplikat. Kandidatnya hanya 5 SKU paket.
- `PAKETTUMBUH250` dan `PAKETBUAH250` memakai **0,25 unit** tiap komponen (kemasan 250 g diambil dari kemasan 1 kg), jadi stok komponen bisa berupa pecahan.
- `PAKETAG` tidak pernah dirakit dan tidak pernah terjual di data.

### 4.7 Menilai `opname_raw`

#### Review awal

In [70]:
review_awal(opname_raw, "opname_raw")

===== opname_raw: 1063 baris, 4 kolom isi =====
1. Info & tipe data
<class 'pandas.DataFrame'>
RangeIndex: 1063 entries, 0 to 1062
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   tanggal     1063 non-null   object
 1   sku_teks    1063 non-null   object
 2   stok_fisik  1063 non-null   object
 3   catatan     53 non-null     object
dtypes: object(4)
memory usage: 33.3+ KB

2. Missing values per kolom
tanggal          0
sku_teks         0
stok_fisik       0
catatan       1010

3. Jumlah duplikat (semua kolom isi sama, dalam berkas/sheet yang sama): 0

4. Jumlah nilai unik per kolom
tanggal        59
sku_teks      202
stok_fisik     89
catatan         2

5. Ringkasan statistik


,count,unique,top,freq
tanggal,1063,59,2023-11-01 00:00:00,127
sku_teks,1063,202,SAMITE100,11
stok_fisik,1063,89,10,193
catatan,53,2,ada yang rusak,30


In [71]:
display(coba_jadi_angka(opname_raw, ["stok_fisik"]))
opname_raw.sample(6, random_state=1)

,terisi,gagal_jadi_angka,contoh_gagal,min,max,negatif
kolom,,,,,,
stok_fisik,1063,0,[],0.0,297.2,0


,tanggal,sku_teks,stok_fisik,catatan,baris_excel,baris_header,berkas,tanggal_berkas
805,2026-02-28 00:00:00,NTVK,10,NaN,33,1,stok_opname_2026-02-28.xlsx,2026-02-28
870,2026-08-31 00:00:00,SAPPORO100,37,NaN,14,1,stok_opname_2026-08-31.xlsx,2026-08-31
148,2023-11-01 00:00:00,MAXXIGROB500,20,NaN,150,1,stok_opname_2023-11-01.xlsx,2023-11-01
675,2025-08-31 00:00:00,EM4C,12,NaN,20,1,stok_opname_2025-08-31.xlsx,2025-08-31
156,01/11/2023,NUGRASS,10,NaN,158,1,stok_opname_2023-11-01.xlsx,2023-11-01
27,1-11-23,ISADORA400,10,NaN,29,1,stok_opname_2023-11-01.xlsx,2023-11-01


**Gejala dari review awal `opname_raw`:**
1. `sku_teks` punya **202 nilai unik**, lebih banyak dari master (190 SKU). → cek **a** dan **b**
2. Nilai `stok_fisik` paling sering adalah **10 (193 baris)**. Untuk Q6, opname pertama akan menjadi saldo awal, jadi perlu dicek apakah semua SKU tercatat dan berapa yang bernilai 0 di setiap opname. → cek **a**
3. `tanggal` di isi sel punya 59 nilai unik untuk 13 berkas opname, dengan format campur (`01/11/2023`, `1-11-23`). Tanggal opname sebaiknya diambil dari **nama berkas**, yang konsisten.
4. `catatan` hanya terisi 53 baris (`ada yang rusak`, keterangan SKU induk dalam kg). Semua `stok_fisik` bisa diubah ke angka dan tidak ada yang negatif. → cek **b**

#### Pengecekan lanjutan

**a. SKU tidak dikenal dan stok 0 per opname** (gejala 1 dan 2).

In [72]:
opname_raw.assign(
    sku_tidak_di_master=~opname_raw["sku_teks"].isin(master_konversi["sku"]),
    stok_nol=opname_raw["stok_fisik"].map(parse_angka) == 0,
).groupby("tanggal_berkas").agg(jumlah_sku=("sku_teks", "size"), sku_tidak_di_master=("sku_tidak_di_master", "sum"),
                                stok_nol=("stok_nol", "sum"))

,jumlah_sku,sku_tidak_di_master,stok_nol
tanggal_berkas,,,
2023-11-01,189,1,0
2024-02-29,94,0,3
2024-05-31,83,1,2
2024-08-31,54,2,0
2024-11-30,56,1,0
2025-02-28,74,1,1
2025-05-31,107,4,1
2025-08-31,43,0,0
2025-11-30,74,0,1


**b. Rincian SKU tidak dikenal, catatan, duplikat SKU, dan satuan induk** (gejala 1 dan 4).

In [73]:
print("SKU tidak dikenal:", opname_raw.loc[~opname_raw["sku_teks"].isin(master_konversi["sku"]), "sku_teks"].tolist())
print("Catatan:", opname_raw["catatan"].value_counts().to_dict())
print("Duplikat SKU dalam satu opname:", opname_raw.duplicated(subset=["tanggal_berkas", "sku_teks"]).sum())
opname_raw[opname_raw["sku_teks"].astype(str).str.contains("50KG")].head(4)

SKU tidak dikenal: ['CPRCD00', 'ASAIBORON1', 'GOSONGOTP500', 'PLYB2G5', 'PAKEBUAH250', 'BINIK100', 'KYRS050', 'FTSSCR500', 'MAXXIGRD500', 'PLYGB30', 'KRESNAAT40', 'KOMRH2', 'TRICO500']
Catatan: {'ada yang rusak': 30, 'SKU induk 50 kg, dalam kg (karung utuh dan sisa)': 23}
Duplikat SKU dalam satu opname: 0


,tanggal,sku_teks,stok_fisik,catatan,baris_excel,baris_header,berkas,tanggal_berkas
185,1-11-23,DLMT50KG,100,"SKU induk 50 kg, dalam kg (karung utuh dan sisa)",187,1,stok_opname_2023-11-01.xlsx,2023-11-01
186,2023-11-01 00:00:00,FRTPHS50KG,49.5,"SKU induk 50 kg, dalam kg (karung utuh dan sisa)",188,1,stok_opname_2023-11-01.xlsx,2023-11-01
187,1-11-23,KCL50KG,50,"SKU induk 50 kg, dalam kg (karung utuh dan sisa)",189,1,stok_opname_2023-11-01.xlsx,2023-11-01
188,1 Nov 2023,UREA50KG,50.5,"SKU induk 50 kg, dalam kg (karung utuh dan sisa)",190,1,stok_opname_2023-11-01.xlsx,2023-11-01


**Data Review `stok_opname`:**
- 13 opname. Opname pertama (1 Nov 2023) mencatat 189 SKU, sisanya bergilir 39 sampai 108 SKU.
- **Opname pertama lengkap**: 189 SKU tercatat dan tidak ada yang bernilai 0, jadi bisa dipakai sebagai saldo awal. Stok 0 hanya muncul 12 kali di opname berikutnya (barang yang memang habis).
- **13 kode SKU salah ketik**, satu sampai empat per opname (`CPRCD00`, `PLYB2G5`, ...).
- Stok induk 50 kg dicatat dalam **kg** (mis. 49,5 dan 50,5), yaitu karung utuh ditambah sisa.
- Tidak ada SKU yang tercatat dua kali dalam satu opname.

### 4.8 Menilai `metrik_raw` (metrik marketplace)

#### Review awal

In [74]:
review_awal(metrik_raw, "metrik_raw")

===== metrik_raw: 4130 baris, 10 kolom isi =====
1. Info & tipe data
<class 'pandas.DataFrame'>
RangeIndex: 4130 entries, 0 to 4129
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   periode_mulai  4130 non-null   object
 1   periode_akhir  4130 non-null   object
 2   kanal          4130 non-null   object
 3   produk         4130 non-null   object
 4   impresi        4130 non-null   object
 5   klik           4130 non-null   object
 6   pesanan        4130 non-null   object
 7   rating         2861 non-null   object
 8   jumlah_ulasan  4130 non-null   object
 9   harga_tayang   4130 non-null   object
dtypes: object(10)
memory usage: 322.8+ KB

2. Missing values per kolom
periode_mulai       0
periode_akhir       0
kanal               0
produk              0
impresi             0
klik                0
pesanan             0
rating           1269
jumlah_ulasan       0
harga_tayang        0

3. Jumlah duplikat (semua 

,count,unique,top,freq
periode_mulai,4130,21,2025-06-01 00:00:00,210
periode_akhir,4130,21,2025-06-30 00:00:00,210
kanal,4130,3,Shopee,1470
produk,4130,137,Emkape Maxxi Agri,52
impresi,4130,1841,50,14
klik,4130,188,13,263
pesanan,4130,13,0,2112
rating,2861.0,6.0,4.6,587.0
jumlah_ulasan,4130,32,0,1812
harga_tayang,4130,205,24938,60


In [75]:
display(coba_jadi_angka(metrik_raw, ["impresi", "klik", "pesanan", "rating", "jumlah_ulasan", "harga_tayang"]))
metrik_raw.sample(6, random_state=1)

,terisi,gagal_jadi_angka,contoh_gagal,min,max,negatif
kolom,,,,,,
impresi,4130,0,[],0.0,26352.0,0
klik,4130,0,[],0.0,599.0,0
pesanan,4130,0,[],0.0,12.0,0
rating,2861,0,[],4.4,4.9,0
jumlah_ulasan,4130,0,[],0.0,32.0,0
harga_tayang,4130,0,[],4332.0,191468.0,0


,periode_mulai,periode_akhir,kanal,produk,impresi,klik,pesanan,rating,jumlah_ulasan,harga_tayang,baris_excel,baris_header,sheet,berkas
957,2026-03-01 00:00:00,2026-03-31 00:00:00,Shopee,VLGR400,1113,45,2,4.7,2,55070,959,1,Shopee,metrik_marketplace_OPSIONAL.xlsx[Shopee]
2722,2026-07-01 00:00:00,2026-07-31 00:00:00,Tokopedia,Magnesium Sulfat Pak Tani,27,1,0,NaN,0,25755,1254,1,Tokopedia,metrik_marketplace_OPSIONAL.xlsx[Tokopedia]
2946,2025-06-01 00:00:00,2025-06-30 00:00:00,TikTok Shop,FILIA50,1321,14,0,4.9,0,38054,8,1,TikTok Shop,metrik_marketplace_OPSIONAL.xlsx[TikTok Shop]
299,2025-06-01 00:00:00,2025-06-30 00:00:00,Shopee,Mipcinta 50 WP,1835,48,3,NaN,3,19438,301,1,Shopee,metrik_marketplace_OPSIONAL.xlsx[Shopee]
718,2025-12-01 00:00:00,2025-12-31 00:00:00,Shopee,PETROKUM100,1691,62,2,NaN,2,13106,720,1,Shopee,metrik_marketplace_OPSIONAL.xlsx[Shopee]
3456,2026-01-01 00:00:00,2026-01-31 00:00:00,TikTok Shop,Bionik 400 EC,1449,72,2,4.7,2,28693,518,1,TikTok Shop,metrik_marketplace_OPSIONAL.xlsx[TikTok Shop]


**Gejala dari review awal `metrik_raw`:**
1. `produk` punya **137 nilai unik**, padahal tiap kanal berisi 70 baris per bulan. Di sampel, kolom ini berisi kode SKU (`VLGR400`) dan nama produk (`Mipcinta 50 WP`) sekaligus. → cek **c**
2. Karena isinya campur, satu produk bisa muncul dua kali dalam periode dan kanal yang sama. → cek **d**
3. `kanal` berisi 3 nilai (`Shopee`, `Tokopedia`, `TikTok Shop`). Nama ini perlu dibandingkan dengan nama kanal di rekap online. → cek **a**
4. `rating` kosong di **1.269 baris**. Semua kolom angka bisa diubah ke angka, tidak ada yang negatif, dan tidak ada duplikat persis. → cek **b**

#### Pengecekan lanjutan

**a. Kolom antar sheet, nama kanal, dan jenis isi** (gejala 3).

In [76]:
print("Kolom tiap sheet sama:", len({tuple(k) for k in kolom_asli_metrik.values()}) == 1)
print("Nama kanal di kolom Kanal:", sorted(metrik_raw["kanal"].unique()), "| di nama file rekap:", sorted(online_raw["platform"].unique()))
kolom_metrik = list(KOLOM_METRIK.values())
tipe_isi(metrik_raw, kolom_metrik)

Kolom tiap sheet sama: True
Nama kanal di kolom Kanal: ['Shopee', 'TikTok Shop', 'Tokopedia'] | di nama file rekap: ['Shopee', 'TikTokShop', 'Tokopedia']


,periode_mulai,periode_akhir,kanal,produk,impresi,klik,pesanan,rating,jumlah_ulasan,harga_tayang
datetime,4130,4130,0,0,0,0,0,0,0,0
float,0,0,0,0,0,0,0,2861,0,0
int,0,0,0,0,4130,4130,4130,0,4130,4130
kosong,0,0,0,0,0,0,0,1269,0,0
str,0,0,4130,4130,0,0,0,0,0,0


**b. Periode, kelengkapan per bulan, logika angka, dan rating** (gejala 4).

In [77]:
print("Periode per kanal:")
print(metrik_raw.groupby("kanal").agg(mulai=("periode_mulai", "min"), akhir=("periode_mulai", "max"),
                                    bulan=("periode_mulai", "nunique"), baris=("produk", "size")))
print("\nProduk per kanal per bulan:", metrik_raw.groupby(["kanal", "periode_mulai"]).size().unique())
print("Klik > impresi:", (metrik_raw["klik"] > metrik_raw["impresi"]).sum(), "| pesanan > klik:", (metrik_raw["pesanan"] > metrik_raw["klik"]).sum())
print("Rating kosong:", metrik_raw["rating"].isna().sum(), "dari", len(metrik_raw))

Periode per kanal:
                 mulai      akhir  bulan  baris
kanal                                          
Shopee      2025-02-01 2026-10-01     21   1470
TikTok Shop 2025-06-01 2026-10-01     17   1190
Tokopedia   2025-02-01 2026-10-01     21   1470

Produk per kanal per bulan: [70]
Klik > impresi: 0 | pesanan > klik: 0
Rating kosong: 1269 dari 4130


**c. Isi kolom produk** (gejala 1).

In [78]:
# Kolom produk berisi campuran kode SKU dan nama
produk_norm = metrik_raw["produk"].map(normalisasi_teks)
print("Berisi kode SKU persis :", produk_norm.str.upper().isin(kode_master).sum())
print("Berisi nama master     :", produk_norm.isin(nama_master).sum())
print("Tidak keduanya         :", (~produk_norm.str.upper().isin(kode_master) & ~produk_norm.isin(nama_master)).sum())
nama_dua_ukuran = set(nama_ganda["nama"].map(normalisasi_teks))
print("Berisi nama yang dipakai 2 SKU:", produk_norm.isin(nama_dua_ukuran).sum())

Berisi kode SKU persis : 2489
Berisi nama master     : 1700
Tidak keduanya         : 0
Berisi nama yang dipakai 2 SKU: 280


**d. Duplikat per periode-kanal-produk** (gejala 2).

In [79]:
# Duplikat per periode-kanal-produk
dup_metrik = metrik_raw.duplicated(subset=["periode_mulai", "kanal", "produk"], keep=False)
print("Baris identik penuh                     :", metrik_raw.duplicated(subset=kolom_metrik).sum())
print("Periode-kanal-produk sama (semua salinan):", dup_metrik.sum())
print("  di antaranya nama yang dipakai 2 SKU  :", (dup_metrik & produk_norm.isin(nama_dua_ukuran)).sum())
metrik_raw[dup_metrik].sort_values(["kanal", "periode_mulai", "produk"]).head(4)[["kanal", "periode_mulai", "produk", "klik", "harga_tayang"]]

Baris identik penuh                     : 0
Periode-kanal-produk sama (semua salinan): 70
  di antaranya nama yang dipakai 2 SKU  : 70


,kanal,periode_mulai,produk,klik,harga_tayang
4,Shopee,2025-02-01 00:00:00,Emkape Maxxi Agri,26,67890
20,Shopee,2025-02-01 00:00:00,Emkape Maxxi Agri,6,45675
74,Shopee,2025-03-01 00:00:00,Emkape Maxxi Agri,7,67890
90,Shopee,2025-03-01 00:00:00,Emkape Maxxi Agri,87,45675


**Data Review `metrik_raw`:**
- 3 sheet dengan kolom yang sama. Shopee dan Tokopedia Feb 2025 sampai Okt 2026 (21 bulan), TikTok Shop Jun 2025 sampai Okt 2026 (17 bulan). Setiap kanal punya **70 baris per bulan**.
- Kolom **SKU / Nama Produk** berisi campuran: 2.489 baris kode SKU dan 1.700 baris nama master. Tidak ada teks di luar keduanya.
- **280 baris berisi nama yang dipakai 2 SKU** (Emkape Maxxi Agri, Fenite 150 OD, Antracol 70 WP, dan lain-lain). Ini penyebab satu-satunya **70 baris yang tampak ganda per periode-kanal-produk**: kedua baris punya harga tayang berbeda (mis. Rp67.890 dan Rp45.675), jadi sebenarnya dua ukuran. Tidak ada baris identik penuh.
- Angka masuk akal: tidak ada klik > impresi atau pesanan > klik.
- **1.269 rating kosong** dari 4.130 baris. Kosong berarti belum ada rating, jadi tidak diisi rata-rata.
- Nama kanal ditulis `TikTok Shop`, berbeda dengan nama file rekap online (`TikTokShop`).

#### Insight Assessing Data & Next Steps

Tabel ini merangkum temuan dan rencana tindakan. Keputusan lengkap beserta alasannya diberi nomor **K1–K16** di awal bagian Cleaning.

| Tabel | Insight | Next Step |
|---|---|---|
| semua | Nama kolom dan posisi header berbeda antar berkas/sheet | Deteksi baris header otomatis, seragamkan nama kolom |
| offline | 54 baris SUBTOTAL/TOTAL | Buang, catat di log |
| offline, online, stok | Tanggal dan angka berupa teks dengan banyak format | `parse_tanggal`, `parse_angka`, `parse_qty` |
| offline | 51 tanggal salah tahun | Ganti tahun sesuai bulan berkas |
| offline | Nota 4 digit (15.941) dan kosong (2.782) | Lengkapi `YYMM-` dari tanggal transaksi; nota kosong diisi jika nota baris atas dan bawah sama, sisanya `TANPA-NOTA` (`order_no_generated`) |
| offline, online, stok, pemecahan | Duplikat persis di data mentah (404 / 16 / 29 / 27) | Buang salinan sebelum pembersihan dan laporkan per berkas; yang baru sama setelah dibersihkan juga dibuang, kecuali di pemecahan |
| offline | 58 qty negatif, semua subtotal positif | Balik tandanya, `qty_sign_fixed = True` |
| offline | Qty/harga kosong | Hitung dari subtotal; yang tidak bisa dihitung ditolak |
| offline | 923 subtotal tidak cocok | Pertahankan subtotal, beri tanda `subtotal_selisih` |
| semua kecuali master | Nama/kode SKU tidak baku | Pemetaan strict, nama beberapa ukuran dari harga pada bulan transaksi, salah ketik lewat mapping tersimpan atau kandidat sempit; sisanya ditolak dan dilaporkan |
| online | Kolom tidak dipakai, termasuk `NO HP` | Buang dan catat di log |
| online, metrik | Nama kanal `TikTokShop` vs `TikTok Shop` | Kamus kanal baku |
| online | Nomor pesanan 18 digit | Baca sebagai teks dan validasi pola per kanal |
| online | HARGA = 0, QTY salah, komisi kosong | Perbaiki dari BRUTO; komisi dihitung balik dari laba kotor; hitung ulang semua kolom turunan |
| online | 82 baris pesanan lintas bulan | Kunci No Order + SKU; simpan di file bulan pesanan |
| stok | Satuan karung/kg/pcs bercampur; restok dengan nama kemasan | Induk 50 kg → kg; angka tanpa satuan = karung; restok kemasan dalam kg/karung → SKU induk |
| stok | Retur tanpa nota asal | Catat sebagai stok masuk, tidak mengurangi omzet |
| pemecahan | Bahan kosong | Hitung = kemasan × faktor (tanpa susut) |
| opname | Opname pertama lengkap (189 SKU); opname berikutnya bergilir | Opname pertama menjadi saldo awal (`OPENING`); stok sistem dibandingkan dan disesuaikan ke stok fisik di setiap opname berikutnya |
| metrik | Kolom produk campuran kode/nama, rating kosong | Petakan ke SKU (dengan harga tayang); rating dibiarkan kosong |

## 5. Cleaning Data
Setiap tabel dibersihkan berurutan sesuai temuan di bagian 4. Pola setiap langkah sama:
1. Tandai baris yang bermasalah.
2. **Catat** ke log.
3. Perbaiki, buang, atau tolak (dilaporkan).
4. Cek hasilnya.

### Keputusan berdasarkan temuan assessing
Setiap keputusan diberi nomor **K1–K16**. Nomor ini dipakai sebagai rujukan singkat di komentar kode dan ringkasan tahap Cleaning berikutnya, misalnya "(K5)" berarti baris itu ditangani dengan aturan K5 di tabel ini.

| No | Masalah | Keputusan | Alasan |
|---|---|---|---|
| K1 | Nama produk/kode SKU tidak baku | Otomatis hanya untuk kode/nama persis (abaikan huruf besar/spasi), nama terpotong yang cocok ke tepat satu nama master, dan **mapping tersimpan**. Semua pemetaan dicatat di log. Teks lain ditolak dan dilaporkan. | Salah petakan lebih berbahaya daripada menolak; laporan penolakan memberi tahu admin baris mana yang perlu diperbaiki. |
| K2 | Satu nama dipakai beberapa SKU (beda ukuran) atau nama terpotong yang cocok ke beberapa nama | Ukuran dipilih dari harga yang berlaku pada **bulan transaksi** (`product_price_history`), toleransi ±10%. Harga kosong, jauh dari semua kandidat, atau cocok ke lebih dari satu → ditolak, tidak masuk transaksi maupun stok, dan disimpan di `penjualan_ambigu.csv` (untuk penyesuaian target di modeling). Hasilnya ditandai `sku_inferred = True`. | Harga jual UMKM naik mengikuti harga kulakan, jadi harga sekarang bisa salah memilih ukuran untuk transaksi lama. Pilihan dibatasi pada kandidat yang pasti. |
| K3 | Salah ketik pada data historis | **Mapping tersimpan** (`product_mapping_manual.csv`) diisi sekali dengan aturan yang jelas: teks yang berjarak satu langkah (ganti/tambah/hapus satu huruf atau tukar dua huruf) dari kode/nama milik lebih dari satu SKU dianggap ambigu; selain itu diterima jika hanya beda tukar dua huruf bersebelahan, atau skor kemiripan ≥ 0,80 dan unggul ≥ 0,10 dari SKU lain. Baris hasil mapping di offline, online, dan restok juga dicek harganya (±10% dari harga acuan SKU hasil). Teks yang ambigu dibuang dan dihitung sebagai keterbatasan data. | Typo yang jelas (`boitek` → BIOTEK) tidak perlu dibuang; yang ambigu (`mpk pak tani`: MKP atau NPK?) tidak boleh ditebak. |
| K4 | Kode salah ketik yang kandidatnya sempit | Diperbaiki hanya dalam kandidat yang pasti: SKU hasil pemecahan milik induk yang sama (`UEA` di baris Urea Petro → UREA), atau 5 SKU paket (`PAKETUTMBUH250` → PAKETTUMBUH250). Restok dengan nama kemasan hasil pemecahan dan qty kg/karung dialihkan ke SKU induk. Ditandai `sku_diperbaiki`. | Kemasan hasil pemecahan tidak pernah dibeli dari supplier, dan satu induk hanya dipecah ke satu ukuran SKU. |
| K5 | Qty salah ketik (offline) | Semua qty negatif dengan subtotal positif dianggap salah ketik tanda dan dibalik (`qty_sign_fixed = True`). Qty yang tidak cocok dengan subtotal, padahal subtotal tepat sama dengan harga × bilangan bulat lain (mis. qty 10, subtotal = 1 × harga), diganti dari subtotal (`qty_diperbaiki_dari_subtotal = True`). | Retur pembeli sudah dicatat di stok masuk/keluar; jika baris negatif juga dianggap retur, stok terhitung dua kali. Subtotal adalah uang yang diterima kasir, jadi lebih dipercaya daripada qty. Aturan ini hanya memakai isi baris itu sendiri, sehingga bisa dipakai saat import. |
| K6 | Retur | Retur hanya menambah stok (`RETURN_IN`), **tidak mengurangi omzet**. | Retur tidak punya nota asal dan tidak punya harga. |
| K7 | Baris ganda | Semua kolom mentah identik → dibuang sebelum dibersihkan, **kecuali di pemecahan**. Baris yang baru sama setelah dibersihkan: dibuang di online dan stok masuk/keluar; di **offline dan pemecahan dipertahankan** dan ditandai `kemungkinan_duplikat`. Jumlah dilaporkan per berkas. | Di offline, pembeli bisa membeli barang yang sama dua kali dengan nota berbeda (nota `0074` vs `2311-0074` tidak selalu input ganda), dan memecah barang yang sama dua kali sehari memang wajar. Membuang penjualan asli lebih merugikan daripada menyimpan baris yang ditandai. Online punya nomor pesanan unik, jadi baris yang sama persis di sana memang input ganda. |
| K8 | Tanggal stok penjualan online | `SALE_ONLINE` memakai hari terakhir bulan laporan; tanggal pesanan asli disimpan di `order_date` untuk analisis dan forecast. | Laporan online baru diimpor setelah bulan selesai. Akibatnya stok sistem tengah bulan bisa lebih tinggi dari fisik. |
| K9 | Pesanan online lintas bulan | Kunci **No Order (teks) + SKU**. Disimpan di file bulan pesanan; salinan di file bulan berikutnya dibuang; angka beda → ditolak. | Nomor 18 digit harus dibaca sebagai teks agar tidak rusak. |
| K10 | Komisi platform kosong | Dihitung balik dari rumus laba kotor: `komisi = netto − biaya program ongkir − komisi affiliator − laba kotor`, ditandai `komisi_dihitung_balik`. | Nilainya pasti dari data UMKM sendiri, tanpa tarif asumsi. |
| K11 | Nomor nota 4 digit dan kosong | Nota 4 digit dilengkapi tahun-bulan dari tanggal transaksi. Nota kosong diisi jika baris atas & bawah sama nomor dan tanggalnya; selain itu `TANPA-NOTA-...` per baris (`order_no_generated = True`). | Q1–Q12 tidak memakai jumlah transaksi atau rata-rata nilai belanja, sehingga ID per baris tidak membuat jawaban bias. Menggabungkan per tanggal justru akan mengarang isi keranjang. |
| K12 | Stok fisik saat opname bisa berbeda dengan stok sistem | Opname pertama dipakai sebagai saldo awal (`OPENING`). Setelah tiap opname berikutnya, stok sistem disesuaikan ke stok fisik dengan pergerakan `ADJUSTMENT`. Q6 mengukur selisih sejak opname sebelumnya. | Opname adalah hitungan fisik, sedangkan stok sistem hanya tahu barang yang tercatat. Barang rusak, pemecahan yang tidak dicatat, dan penjualan yang ditolak membuat keduanya berbeda; tanpa penyesuaian, selisih itu menumpuk sampai akhir data. |
| K13 | Nama kanal tidak seragam | Kamus kanal: `Offline`, `Shopee`, `Tokopedia`, `TikTok Shop`, dipakai di semua tabel termasuk metrik marketplace. | Join antartabel (Q12) tidak gagal diam-diam. |
| K14 | Supplier utama | Supplier yang paling sering memasok **kelompok produk induknya**, supaya kemasan hasil pemecahan ikut mendapat supplier. | Dipakai untuk lead time Q5 dan Q8. |
| K15 | Kolom tidak dipakai | `NO HP` (data pribadi), `DAERAH` (tidak dipakai di Q1–Q12), `TOTAL` (= NETTO), `LABA BERSIH TOTAL` (sel gabungan), `BIAYA BALIK KULAKAN` (= harga kulakan) dibuang. | Mengurangi data pribadi dan kolom ganda. |
| K16 | Rating kosong di metrik marketplace | Dibiarkan kosong, tidak diisi rata-rata. | Produk yang belum punya rating berbeda dengan produk berating rata-rata. |

In [80]:
def ringkasan_tahap(tahap, n_masuk, bersih, ditolak):
    # Tampilkan jumlah baris per masalah/tindakan dan cek keseimbangan baris satu tahap.
    log = log_tahap(tahap)
    dibuang = log["tindakan"].str.startswith("dibuang").sum()
    status = "SEIMBANG" if n_masuk == len(bersih) + len(ditolak) + dibuang else "TIDAK SEIMBANG"
    print(f"{tahap}: masuk {n_masuk} = bersih {len(bersih)} + ditolak {len(ditolak)} + dibuang {dibuang} -> {status}")
    return (log.groupby(["masalah", "tindakan"]).size().rename("jumlah_baris").reset_index()
            .sort_values("jumlah_baris", ascending=False).reset_index(drop=True))


# K13: kamus kanal baku, dipakai di semua tabel (rekap online, metrik marketplace, transaksi)
KAMUS_KANAL = {"offline": "Offline", "shopee": "Shopee", "tokopedia": "Tokopedia",
               "tiktokshop": "TikTok Shop", "tiktok shop": "TikTok Shop"}
KANAL_BAKU = ["Offline", "Shopee", "Tokopedia", "TikTok Shop"]

### 5.1 Membersihkan master produk → tabel `products`
**Masalah dari 4.1:**
- 4 SKU induk 50 kg tidak ada di `master_kategori` → nama dan kategorinya dilengkapi dari produk induknya.
- `parent_id` adalah kode kelompok, bukan `product_id` induk → diganti nama menjadi `kelompok_induk`, lalu dibuat kolom `sku_induk` yang jelas.
- 13 nama dipakai 2 SKU → tidak diubah di master; ditangani saat pemetaan (K2).

In [81]:
SKU_INDUK_TIDAK_DIJUAL = ["UREA50KG", "KCL50KG", "DLMT50KG", "FRTPHS50KG"]

# Mulai dari master_induk_konversi karena berisi ke-190 SKU
products = master_konversi.rename(columns={"parent_id": "kelompok_induk", "parent_name": "nama_induk",
                                           "faktor_per_unit_jual": "faktor"})
products = products[["sku", "product_id", "kelompok_induk", "nama_induk", "base_unit", "faktor",
                     "isi_per_pembelian", "satuan_beli", "is_bundle"]]

# Tambahkan nama dan kategori (186 SKU dijual)
products = products.merge(master_kategori[["sku", "nama", "kategori"]], on="sku", how="left")

# Tambahkan harga: hanya harga kulakan, HPP, harga jual online, dan harga jual offline
harga = master_harga.rename(columns={"KODE SKU": "sku", "HARGA PRODUKSI/KULAKAN": "harga_kulakan", "HPP": "hpp",
                                     "HARGA JUAL ONLINE": "harga_jual_online", "HARGA JUAL OFFLINE": "harga_jual_offline",
                                     "SUMBER DATA": "sumber_harga"})
products = products.merge(harga[["sku", "harga_kulakan", "hpp", "harga_jual_online", "harga_jual_offline", "sumber_harga"]],
                          on="sku", how="left")
print(products.shape)

(190, 16)


In [82]:
# Lengkapi nama dan kategori 4 SKU induk 50 kg dari produk lain di kelompok yang sama
induk50 = products["sku"].isin(SKU_INDUK_TIDAK_DIJUAL)
for i in products.index[induk50]:
    kelompok = products.loc[i, "kelompok_induk"]
    anak = products[(products["kelompok_induk"] == kelompok) & ~induk50]
    products.loc[i, "nama"] = products.loc[i, "nama_induk"] + " (karung 50 kg)"
    products.loc[i, "kategori"] = anak["kategori"].iloc[0]
catatan_master = products.loc[induk50, ["sku", "nama"]].assign(berkas="master_kategori.csv", baris_excel=None)
catat("master", catatan_master, "SKU induk tidak ada di master_kategori",
      "nama dan kategori diisi dari produk induknya", kolom="nama,kategori", baru="sku")

products["is_bundle"] = products["is_bundle"].astype(bool)
products["is_sellable"] = ~induk50
products["satuan_stok"] = np.where(induk50, "kg", "unit")     # induk dicatat dalam kg (sesuai opname)
products.loc[induk50, ["sku", "nama", "kategori", "is_sellable", "satuan_stok"]]

,sku,nama,kategori,is_sellable,satuan_stok
186,UREA50KG,Pupuk Urea Petro (karung 50 kg),Pupuk,False,kg
187,KCL50KG,Pupuk KCL Mahkota (karung 50 kg),Pupuk,False,kg
188,DLMT50KG,Kapur Pertanian Kebomas (karung 50 kg),Pupuk,False,kg
189,FRTPHS50KG,Fertiphos Pak Tani (karung 50 kg),Pupuk,False,kg


In [83]:
# Relasi pemecahan: dalam kelompok yang berisi >1 SKU, induk = SKU dengan faktor terbesar
products["sku_induk"] = None
products["faktor_ke_induk"] = np.nan
for kelompok, grup in products.groupby("kelompok_induk"):
    if len(grup) < 2:
        continue
    sku_induk = grup.loc[grup["faktor"].idxmax(), "sku"]
    anak = grup.index[grup["sku"] != sku_induk]
    products.loc[anak, "sku_induk"] = sku_induk
    products.loc[anak, "faktor_ke_induk"] = grup.loc[anak, "faktor"]   # satuan stok induk yang terpakai per 1 unit
products["is_induk_pemecahan"] = products["sku"].isin(products["sku_induk"].dropna())

products = products[["sku", "product_id", "nama", "kategori", "is_sellable", "is_bundle", "satuan_stok",
                     "kelompok_induk", "nama_induk", "sku_induk", "faktor_ke_induk", "is_induk_pemecahan",
                     "base_unit", "faktor", "isi_per_pembelian", "satuan_beli", "harga_kulakan", "hpp",
                     "harga_jual_online", "harga_jual_offline", "sumber_harga"]].sort_values("product_id").reset_index(drop=True)
products.loc[products["sku_induk"].notna(), ["sku", "nama", "sku_induk", "faktor_ke_induk"]]

,sku,nama,sku_induk,faktor_ke_induk
11,DLMT,Kapur Dolomite,DLMT50KG,0.5
122,EMKAPE500,Emkape Maxxi Agri,EMKAPE1,0.5
126,FRTPHS1,Fertiphos Curah,FRTPHS50KG,1.0
133,KANITRAT500,Kanitrat Maxxi Agri,KANITRAT1,0.5
137,KCL,KCL Curah,KCL50KG,0.5
183,UREA,Urea Curah,UREA50KG,0.5


In [84]:
bundle_components = master_komposisi.rename(columns={"jumlah_per_paket_dalam_satuan_jual_komponen": "qty_komponen_per_paket"})
bundle_components = bundle_components[["sku_paket", "sku_komponen", "qty_komponen_per_paket"]]
suppliers = master_supplier.rename(columns={"nama_dummy": "nama_supplier", "lead_time_hari_asumsi": "lead_time_hari"})
suppliers = suppliers[["supplier_code", "nama_supplier", "kelompok_produk", "lead_time_hari"]]
print("products:", products.shape, "| bundle_components:", bundle_components.shape, "| suppliers:", suppliers.shape)

products: (190, 21) | bundle_components: (13, 3) | suppliers: (7, 4)


### 5.2 Fungsi bantu pemetaan SKU (K1, K2, K3)
Semua tabel perlu memetakan teks ke SKU master, jadi fungsinya dibuat sekali di sini.

**Langkah pemetaan:**
1. `cocokkan_teks`: cocokkan satu teks unik dengan aturan K1, berurutan: kode persis → nama persis → nama terpotong → **mapping tersimpan** (K3). Hasilnya `cocok` (satu SKU), `ganda` (beberapa SKU: nama dipakai beberapa ukuran atau nama terpotong yang cocok ke beberapa nama), atau `tidak_dikenal`.
2. `petakan_teks`: jalankan langkah 1 sekali untuk setiap teks unik, lalu tempelkan hasilnya ke semua baris.
3. `riwayat_harga`, `buat_acuan`, `harga_acuan`: harga yang berlaku per SKU per bulan, dari baris yang SKU-nya **pasti**.
4. `tentukan_ukuran` (K2): untuk baris `ganda`, pilih kandidat dari harga pada bulan transaksi. Yang tidak bisa diputuskan ditolak. Baris `tidak_dikenal` juga ditolak.
5. `catat_pemetaan`, `tabel_mapping`, `laporan_ditolak`: catat semua pemetaan ke log, buat ringkasan per teks unik, dan buat laporan baris yang ditolak.

In [85]:
KG_PER_KARUNG = 50
TOLERANSI_HARGA = 0.10        # K2: harga transaksi boleh beda maks. 10% dari harga acuan
SUFIKS_LISTING = [" ready stock", " asli bergaransi", " original", " - murah"]   # akhiran judul listing online

# Kamus pencocokan dari master
kamus_kode = {sku.upper(): sku for sku in products["sku"]}
kamus_nama = {}                                     # nama -> list SKU (satu nama bisa dipakai 2 SKU)
for sku, nama in zip(products["sku"], products["nama"]):
    kamus_nama.setdefault(normalisasi_teks(nama), []).append(sku)
daftar_nama = sorted(kamus_nama)
print("Kode SKU:", len(kamus_kode), "| nama unik:", len(kamus_nama))

Kode SKU: 190 | nama unik: 177


In [86]:
def skor_kemiripan(a, b):
    # Skor kemiripan dua teks (0 sampai 1). Dihitung juga tanpa spasi, lalu diambil yang lebih tinggi.
    return round(max(difflib.SequenceMatcher(None, a, b).ratio(),
                     difflib.SequenceMatcher(None, a.replace(" ", ""), b.replace(" ", "")).ratio()), 2)


def cocokkan_teks(teks, pakai_mapping=True):
    # Aturan K1 untuk satu teks yang sudah dinormalisasi. Hasil: status, aturan, kandidat (list SKU).
    if teks == "":
        return {"status": "tidak_dikenal", "aturan": "kosong", "kandidat": []}
    if teks.upper() in kamus_kode:                                       # 1. kode SKU persis
        return {"status": "cocok", "aturan": "kode", "kandidat": [kamus_kode[teks.upper()]]}
    if teks in kamus_nama:                                               # 2. nama persis
        skus = kamus_nama[teks]
        return {"status": "cocok" if len(skus) == 1 else "ganda", "aturan": "nama", "kandidat": skus}
    nama_awal = [n for n in daftar_nama if n.startswith(teks + " ")]     # 3. nama terpotong
    if nama_awal:
        skus = [s for n in nama_awal for s in kamus_nama[n]]
        aturan = "nama terpotong" if len(nama_awal) == 1 else "nama terpotong ambigu"
        return {"status": "cocok" if len(skus) == 1 else "ganda", "aturan": aturan, "kandidat": skus}
    if pakai_mapping and teks in mapping_tersimpan:                      # 4. mapping tersimpan (K3)
        hasil = cocokkan_teks(mapping_tersimpan[teks], pakai_mapping=False)
        return dict(hasil, aturan="mapping tersimpan -> " + hasil["aturan"])
    return {"status": "tidak_dikenal", "aturan": "tidak dikenal", "kandidat": []}


# Contoh (mapping tersimpan belum dimuat, jadi dicoba tanpa mapping)
for t in ["UREA", "kcl curah", "emkape maxxi agri", "trobos 200", "polybag tanaman", "kclc urah"]:
    print(f"{t:20s} ->", cocokkan_teks(t, pakai_mapping=False))

UREA                 -> {'status': 'cocok', 'aturan': 'kode', 'kandidat': ['UREA']}
kcl curah            -> {'status': 'cocok', 'aturan': 'nama', 'kandidat': ['KCL']}
emkape maxxi agri    -> {'status': 'ganda', 'aturan': 'nama', 'kandidat': ['EMKAPE1', 'EMKAPE500']}
trobos 200           -> {'status': 'cocok', 'aturan': 'nama terpotong', 'kandidat': ['TROBOS100']}
polybag tanaman      -> {'status': 'ganda', 'aturan': 'nama terpotong ambigu', 'kandidat': ['PLYBG15', 'PLYBG20', 'PLYBG25', 'PLYBG30', 'PLYBG35', 'PLYBG40', 'PLYBG45', 'PLYBG50']}
kclc urah            -> {'status': 'tidak_dikenal', 'aturan': 'tidak dikenal', 'kandidat': []}


### 5.3 Mapping tersimpan untuk salah ketik (K3)
`product_mapping_manual.csv` berisi setiap teks historis yang tidak dikenali beserta **teks benar** (kode atau nama master) dan statusnya: `masuk` jika maksudnya jelas, `ambigu` jika tidak. Teks ambigu ikut disimpan sebagai bukti, tetapi tidak dipakai untuk memetakan. File ini dibuat **sekali** dari teks historis yang tidak dikenali, lalu disimpan. Pada run berikutnya, dan saat backend meng-import file, file ini hanya **dibaca**.

**Cek ambigu lebih dulu.** `satu_langkah(a, b)` bernilai True jika `a` bisa menjadi `b` dengan **satu** perubahan saja: ganti satu huruf, tambah satu huruf, hapus satu huruf, atau tukar dua huruf bersebelahan. Jika teks berjarak satu langkah dari kode/nama milik **lebih dari satu SKU**, teks itu **ambigu** dan tidak dimasukkan. Contoh: `mpk pak tani` bisa MKP Pak Tani (tukar `pk`↔`kp`) atau NPK Pak Tani (ganti `m`→`n`).

Jika tidak ambigu, teks dianggap **jelas maksudnya** jika memenuhi salah satu aturan berikut:
1. hanya beda **tukar dua huruf bersebelahan** dari kode/nama master, dan **semua** kode/nama yang berjarak satu langkah menunjuk ke SKU yang sama (`cgn1` → CNG1, `uera` → UREA); atau
2. **skor kemiripan ≥ 0,80** dengan satu kode/nama, dan **unggul ≥ 0,10** dari kode/nama milik SKU lain (`boitek` → BIOTEK 0,83; `kclc urah` → KCL CURAH 0,89).

Teks yang tidak memenuhi aturan (mis. `mpk pak tani`, `metaze 80 wp` yang sama miripnya dengan METAZEB dan MEGAZEB, atau `mk pak`) **tidak dimasukkan**, sehingga barisnya ditolak dan dihitung sebagai keterbatasan data.

**Kenapa cek "satu langkah" juga berlaku untuk teks yang lolos lewat skor kemiripan?** Skor kemiripan mengukur seberapa mirip teks dengan **satu** target, sedangkan salah ketik terjadi pada satu ketikan. Jika ada dua SKU yang sama-sama hanya berjarak satu ketikan, keduanya sama mungkinnya, dan skor yang sedikit lebih tinggi tidak membuktikan mana yang dimaksud. Contoh `mp1`: skor ke `mkp1` 0,86 dan unggul 0,19, sehingga skor saja akan meloloskannya. Padahal `mp1` juga hanya satu ketikan dari `m21` (ganti `p`→`2`), jadi bisa MKP1 atau M21. Karena itu `mp1` ditolak. Aturan ini dipakai untuk semua teks.

In [87]:
def tukar_bersebelahan(a, b):
    # True jika a dan b hanya beda karena dua huruf bersebelahan tertukar (mis. 'uera' vs 'urea').
    if len(a) != len(b) or a == b:
        return False
    beda = [i for i in range(len(a)) if a[i] != b[i]]
    return len(beda) == 2 and beda[1] == beda[0] + 1 and a[beda[0]] == b[beda[1]] and a[beda[1]] == b[beda[0]]


def satu_langkah(a, b):
    # True jika a bisa menjadi b dengan SATU perubahan: ganti, tambah, atau hapus satu huruf,
    # atau tukar dua huruf bersebelahan.
    if a == b or abs(len(a) - len(b)) > 1:
        return False
    if len(a) == len(b):
        beda = sum(x != y for x, y in zip(a, b))
        return beda == 1 or tukar_bersebelahan(a, b)          # ganti satu huruf atau tukar
    pendek, panjang = (a, b) if len(a) < len(b) else (b, a)
    for i in range(len(panjang)):                              # hapus satu huruf dari yang panjang
        if panjang[:i] + panjang[i + 1:] == pendek:
            return True
    return False


# Semua "teks benar" yang mungkin: nama master dan kode SKU (huruf kecil), beserta SKU-nya
target_benar = {nama: set(skus) for nama, skus in kamus_nama.items()}
for sku in products["sku"]:
    target_benar.setdefault(sku.lower(), set()).add(sku)


def cari_teks_benar(teks):
    # Aturan K3. Hasil: (teks_benar, dasar, skor). teks_benar = None jika teks ambigu.
    polos = teks.replace(" ", "")
    cocok_langkah = lambda f, t: f(teks, t) or f(polos, t.replace(" ", ""))
    dekat = {t for t in target_benar if cocok_langkah(satu_langkah, t)}           # berjarak satu langkah
    if len({frozenset(target_benar[t]) for t in dekat}) > 1:                     # menunjuk ke SKU berbeda
        skus = sorted({s for t in dekat for s in target_benar[t]})
        return None, "ambigu: satu langkah dari beberapa SKU (" + ", ".join(skus) + ")", None
    tukar = {t for t in dekat if cocok_langkah(tukar_bersebelahan, t)}
    if tukar:
        return sorted(tukar)[0], "tukar dua huruf bersebelahan (satu-satunya SKU berjarak satu langkah)", 1.0
    skor = sorted(((skor_kemiripan(teks, t), t) for t in target_benar), reverse=True)
    s1, terbaik = skor[0]
    lawan = next((s for s, t in skor[1:] if not (target_benar[t] & target_benar[terbaik])), 0)
    if s1 >= 0.80 and s1 - lawan >= 0.10:
        return terbaik, f"skor {s1:.2f}, unggul {s1 - lawan:.2f} dari SKU lain", s1
    return None, f"ambigu: skor {s1:.2f}, unggul {s1 - lawan:.2f} (perlu >= 0,80 dan >= 0,10)", None


for a, b in [("uera", "urea"), ("mpk pak tani", "npk pak tani"), ("dlmtt", "dlmt"), ("boitek", "biotek")]:
    print(f"satu_langkah({a!r}, {b!r}) = {satu_langkah(a, b)}")

satu_langkah('uera', 'urea') = True
satu_langkah('mpk pak tani', 'npk pak tani') = True
satu_langkah('dlmtt', 'dlmt') = True
satu_langkah('boitek', 'biotek') = True


In [88]:
path_mapping = INTERIM / "product_mapping_manual.csv"
BUAT_ULANG_MAPPING = False            # ubah ke True hanya jika aturan K3 diubah

if path_mapping.exists() and not BUAT_ULANG_MAPPING:
    teks_salah_ketik = pd.read_csv(path_mapping)
    print("Mapping tersimpan dibaca dari file.")
else:
    # Kumpulkan teks historis dari semua tabel yang memakai pemetaan teks bebas
    def teks_online(seri):
        teks = seri.map(normalisasi_teks)
        for sufiks in SUFIKS_LISTING:
            teks = teks.str.removesuffix(sufiks).str.strip()
        return teks
    semua_teks = pd.concat([off_tx["nama_produk"].map(normalisasi_teks),
                            teks_online(on_tx["sku_barang"]), teks_online(on_tx["nama_produk"]),
                            stok_mk_raw["sku_teks"].map(normalisasi_teks),
                            opname_raw["sku_teks"].map(normalisasi_teks),
                            metrik_raw["produk"].map(normalisasi_teks)])
    jumlah = semua_teks.value_counts()
    baris_mapping = []
    for teks, n in jumlah.items():
        if cocokkan_teks(teks, pakai_mapping=False)["status"] != "tidak_dikenal" or teks == "":
            continue
        teks_benar, dasar, skor = cari_teks_benar(teks)
        baris_mapping.append({"teks": teks, "jumlah_baris_historis": n, "teks_benar": teks_benar,
                              "sku_tujuan": ", ".join(sorted(target_benar[teks_benar])) if teks_benar else None,
                              "dasar": dasar, "skor": skor,
                              "status": "masuk" if teks_benar else "ambigu"})
    teks_salah_ketik = pd.DataFrame(baris_mapping)
    teks_salah_ketik.to_csv(path_mapping, index=False)       # teks ambigu ikut disimpan sebagai bukti

print("Teks historis tidak dikenal:", len(teks_salah_ketik))
print(teks_salah_ketik["status"].value_counts().to_string())

# Strict: hanya teks berstatus masuk dan teks benarnya memang kode/nama master
mapping_df = teks_salah_ketik[(teks_salah_ketik["status"] == "masuk") & teks_salah_ketik["teks_benar"].isin(target_benar)]
mapping_tersimpan = dict(zip(mapping_df["teks"], mapping_df["teks_benar"]))
mapping_df.head(8)

Mapping tersimpan dibaca dari file.
Teks historis tidak dikenal: 1010
status
masuk     911
ambigu     99


,teks,jumlah_baris_historis,teks_benar,sku_tujuan,dasar,skor,status
0,kclc urah,11,kcl curah,KCL,tukar dua huruf bersebelahan (satu-satunya SKU berjarak ...,1.00,masuk
1,boitek,10,biotek,BIOTEK,tukar dua huruf bersebelahan (satu-satunya SKU berjarak ...,1.00,masuk
2,klc curah,8,kcl curah,KCL,tukar dua huruf bersebelahan (satu-satunya SKU berjarak ...,1.00,masuk
3,kl curah,7,kcl curah,KCL,"skor 0.94, unggul 0.27 dari SKU lain",0.94,masuk
4,kclcurah,7,kcl curah,KCL,"skor 1.00, unggul 0.41 dari SKU lain",1.00,masuk
5,cgn1,6,cng1,CNG1,tukar dua huruf bersebelahan (satu-satunya SKU berjarak ...,1.00,masuk
6,kcl cruah,6,kcl curah,KCL,tukar dua huruf bersebelahan (satu-satunya SKU berjarak ...,1.00,masuk
7,kapur dolmoite,5,kapur dolomite,DLMT,tukar dua huruf bersebelahan (satu-satunya SKU berjarak ...,1.00,masuk


In [89]:
# Contoh hasil dan contoh teks yang tetap ambigu
for t in ["boitek", "kclc urah", "cgn1", "uera", "mpk pak tani", "metaze 80 wp", "mk pak"]:
    print(f"{t:15s} ->", mapping_tersimpan.get(t, "(tidak ada di mapping: ditolak)"))

boitek          -> biotek
kclc urah       -> kcl curah
cgn1            -> cng1
uera            -> urea
mpk pak tani    -> (tidak ada di mapping: ditolak)
metaze 80 wp    -> (tidak ada di mapping: ditolak)
mk pak          -> (tidak ada di mapping: ditolak)


### 5.4 Fungsi pemetaan per tabel

In [90]:
def petakan_teks(df, kolom_teks, kolom_cadangan=None, buang_sufiks=False):
    # Cocokkan setiap teks unik di `kolom_teks`, lalu tempelkan hasilnya ke semua baris.
    # `kolom_cadangan` (khusus online): dipakai jika kode SKU kosong atau tidak cocok.
    df = df.copy()

    def siapkan(seri):
        teks = seri.map(normalisasi_teks)
        if buang_sufiks:
            for sufiks in SUFIKS_LISTING:
                teks = teks.str.removesuffix(sufiks).str.strip()
        return teks

    df["teks_sku_asli"] = df[kolom_teks]
    df["teks_normal"] = siapkan(df[kolom_teks])
    hasil = {t: cocokkan_teks(t) for t in df["teks_normal"].unique()}        # satu kali per teks unik
    df["dari_cadangan"] = False
    if kolom_cadangan:
        teks2 = siapkan(df[kolom_cadangan])
        hasil.update({t: cocokkan_teks(t) for t in teks2.unique() if t not in hasil})
        utama_gagal = df["teks_normal"].map(lambda t: hasil[t]["status"] != "cocok")
        cadangan_ok = teks2.map(lambda t: hasil[t]["status"] in ("cocok", "ganda")) | (df["teks_normal"] == "")
        pakai = utama_gagal & cadangan_ok
        df.loc[pakai, "teks_sku_asli"] = df.loc[pakai, kolom_cadangan]
        df.loc[pakai, "teks_normal"] = teks2[pakai]
        df["dari_cadangan"] = pakai
    df["status_cocok"] = df["teks_normal"].map(lambda t: hasil[t]["status"])
    df["aturan_sku"] = df["teks_normal"].map(lambda t: hasil[t]["aturan"]) + np.where(df["dari_cadangan"], " (dari nama produk)", "")
    df["kandidat"] = df["teks_normal"].map(lambda t: hasil[t]["kandidat"])
    df["sku"] = [k[0] if s == "cocok" else None for s, k in zip(df["status_cocok"], df["kandidat"])]
    df["sku_inferred"] = False
    df["rincian_harga"] = None
    return df

In [91]:
def riwayat_harga(df, kolom_tanggal, kolom_harga, jenis_harga):
    # Median harga per SKU per bulan dari baris yang SKU-nya pasti (status cocok, bukan hasil mapping salah ketik, tidak ditolak).
    pasti = (df["status_cocok"] == "cocok") & ~df["aturan_sku"].fillna("").str.startswith("mapping tersimpan")
    d = df[pasti & df["alasan_ditolak"].isna() & df[kolom_tanggal].notna() & (df[kolom_harga] > 0)]
    d = d.assign(bulan=d[kolom_tanggal].dt.to_period("M").dt.to_timestamp())
    r = d.groupby(["sku", "bulan"])[kolom_harga].agg(harga="median", jumlah_transaksi="size").reset_index()
    r.insert(0, "jenis_harga", jenis_harga)
    return r


def buat_acuan(riwayat):
    # Ubah tabel riwayat menjadi dict {sku: deret harga per bulan} supaya pencarian per baris cepat.
    return {sku: grup.set_index("bulan")["harga"].sort_index() for sku, grup in riwayat.groupby("sku")}


def harga_acuan(sku, tanggal, acuan, harga_master):
    # Harga SKU yang berlaku pada tanggal: bulan yang sama -> bulan terdekat -> harga master.
    seri = acuan.get(sku)
    if seri is not None and not kosong(tanggal):
        bulan = tanggal.to_period("M").to_timestamp()
        if bulan in seri.index:
            return seri[bulan], "bulan yang sama"
        terdekat = seri.index[np.argmin(abs((seri.index - bulan).days))]
        return seri[terdekat], f"bulan terdekat ({terdekat:%Y-%m})"
    return harga_master.get(sku, np.nan), "harga master"


def pilih_ukuran(kandidat, harga, tanggal, acuan, harga_master):
    # K2: pilih kandidat yang harga acuannya dalam toleransi. Tidak menebak jika 0 atau >1 kandidat cocok.
    if kosong(harga) or harga <= 0:
        return None, "harga kosong", ""
    dekat, rincian = [], []
    for sku in kandidat:
        ref, sumber = harga_acuan(sku, tanggal, acuan, harga_master)
        if kosong(ref):
            rincian.append(f"{sku} (tanpa harga acuan)")
            continue
        rasio = harga / ref
        rincian.append(f"{sku} (rasio {rasio:.2f}, {sumber})")
        if abs(rasio - 1) <= TOLERANSI_HARGA:
            dekat.append(sku)
    if len(dekat) == 1:
        return dekat[0], "cocok ke satu ukuran", "; ".join(rincian)
    if len(dekat) == 0:
        return None, "harga jauh dari semua kandidat", "; ".join(rincian)
    return None, "harga cocok ke lebih dari satu kandidat", "; ".join(rincian)


def tentukan_ukuran(df, kolom_harga, kolom_tanggal, acuan, harga_master):
    # Isi SKU untuk baris 'ganda' memakai harga pada bulan transaksi; tolak yang tidak bisa diputuskan.
    for i in df.index[df["status_cocok"] == "ganda"]:
        if kolom_harga:
            sku, alasan, rincian = pilih_ukuran(df.at[i, "kandidat"], df.at[i, kolom_harga], df.at[i, kolom_tanggal], acuan, harga_master)
        else:
            sku, alasan, rincian = None, "tidak ada kolom harga", ""
        df.at[i, "rincian_harga"] = rincian
        if sku:
            df.at[i, "sku"] = sku
            df.at[i, "sku_inferred"] = True
            df.at[i, "aturan_sku"] += " + harga"
        else:
            df.at[i, "alasan_ditolak"] = f"nama dipakai beberapa SKU: {alasan}"
    # Teks yang tidak dikenal (tidak ada di master maupun mapping tersimpan) -> ditolak
    tidak = (df["status_cocok"] == "tidak_dikenal") & df["alasan_ditolak"].isna()
    df.loc[tidak, "alasan_ditolak"] = "SKU tidak dikenal (tidak ada di master maupun mapping tersimpan)"
    return df

In [92]:
def catat_pemetaan(df, tahap):
    # K1: catat SEMUA pemetaan (teks asli -> SKU, aturan) dan semua teks yang ditolak.
    ok = df[df["sku"].notna() & df["alasan_ditolak"].isna()]
    for aturan, grup in ok.groupby("aturan_sku"):
        if grup["sku_inferred"].iloc[0]:
            catat(tahap, grup.assign(_baru=grup["sku"] + " | " + grup["rincian_harga"]),
                  f"pemetaan SKU dari harga ({aturan})", "dipetakan dari harga (sku_inferred = True)",
                  kolom="sku", lama="teks_sku_asli", baru="_baru")
        else:
            catat(tahap, grup, f"pemetaan SKU otomatis ({aturan})", "dipetakan otomatis",
                  kolom="sku", lama="teks_sku_asli", baru="sku")
    gagal = df[df["status_cocok"].notna() & df["alasan_ditolak"].fillna("").str.match("SKU tidak dikenal|nama dipakai")]
    catat(tahap, gagal, "SKU tidak dikenali atau ambigu", "ditolak (dilaporkan)", kolom="sku", lama="teks_sku_asli", baru="alasan_ditolak")


def tabel_mapping(df, sumber):
    # Satu baris per teks unik: berapa baris, berapa yang masuk, aturan, SKU hasil, dan alasan penolakan.
    d = df.assign(masuk=df["sku"].notna() & df["alasan_ditolak"].isna())
    m = (d.groupby("teks_normal")
         .agg(contoh_teks_asli=("teks_sku_asli", lambda s: next((str(v) for v in s if not kosong(v)), "")),
              jumlah_baris=("masuk", "size"), baris_masuk=("masuk", "sum"), aturan=("aturan_sku", "first"),
              sku_hasil=("sku", lambda s: ", ".join(sorted(set(s.dropna())))))
         .reset_index().rename(columns={"teks_normal": "teks"}))
    m.insert(0, "sumber", sumber)
    m["status"] = np.where(m["baris_masuk"] == m["jumlah_baris"], "masuk",
                           np.where(m["baris_masuk"] == 0, "ditolak", "sebagian ditolak"))
    return m


def laporan_ditolak(df, tahap, kolom_teks):
    # Laporan baris yang ditolak: berkas, baris Excel, teks asli, alasan (untuk diperbaiki di file sumber).
    lap = pd.DataFrame({"tahap": tahap, "berkas": df["berkas"], "baris_excel": df["baris_excel"],
                        "teks": df[kolom_teks].astype(str), "alasan": df["alasan_ditolak"]})
    if "rincian_harga" in df:
        lap["rincian_harga"] = df["rincian_harga"]
    return lap.reset_index(drop=True)


def cek_harga_wajar(df, kolom_harga, kolom_tanggal, acuan, harga_master, tahap):
    # Tolak baris yang harganya < 0,5x atau > 2x harga acuan pada tanggalnya (tanda qty/harga salah ketik).
    ok = df["sku"].notna() & df["alasan_ditolak"].isna()
    rasio = pd.Series(np.nan, index=df.index)
    for i in df.index[ok]:
        rasio[i] = df.at[i, kolom_harga] / harga_acuan(df.at[i, "sku"], df.at[i, kolom_tanggal], acuan, harga_master)[0]
    aneh = ok & ((rasio < 0.5) | (rasio > 2.0))
    df.loc[aneh, "alasan_ditolak"] = "harga satuan tidak wajar vs harga acuan"
    catat(tahap, df[aneh].assign(_rasio=rasio[aneh].round(2)), "harga satuan tidak wajar (<0,5x atau >2x harga acuan)",
          "ditolak (dilaporkan)", kolom=kolom_harga, lama=kolom_harga, baru="_rasio")
    return df


ALASAN_HARGA_MAPPING = "harga tidak sesuai SKU hasil mapping salah ketik"


def cek_harga_mapping(df, kolom_harga, kolom_tanggal, acuan, harga_master, tahap, dicek=None):
    # K3: baris yang SKU-nya dari mapping tersimpan dicek harganya terhadap harga acuan SKU hasil pada bulannya.
    # Di luar +-10% -> ditolak. Harga kosong, acuan kosong, atau baris di luar `dicek` (mis. retur) -> diterima, ditandai.
    dari_mapping = (df["aturan_sku"].fillna("").str.startswith("mapping tersimpan")
                    & df["sku"].notna() & df["alasan_ditolak"].isna())
    if dicek is None:
        dicek = pd.Series(True, index=df.index)
    rasio = pd.Series(np.nan, index=df.index)
    for i in df.index[dari_mapping & dicek]:
        harga = df.at[i, kolom_harga]
        if kosong(harga) or harga <= 0:
            continue
        acuan_i, _ = harga_acuan(df.at[i, "sku"], df.at[i, kolom_tanggal], acuan, harga_master)
        if not kosong(acuan_i) and acuan_i > 0:
            rasio[i] = harga / acuan_i
    tidak_dicek = dari_mapping & rasio.isna()
    luar = dari_mapping & ((rasio - 1).abs() > TOLERANSI_HARGA)
    df["harga_mapping_tidak_dicek"] = tidak_dicek
    df["rasio_harga_mapping"] = rasio
    tolak(df, luar, ALASAN_HARGA_MAPPING)
    catat(tahap, df[luar].assign(_rasio=rasio[luar].round(2)), ALASAN_HARGA_MAPPING + " (di luar +-10%)",
          "ditolak (dilaporkan)", kolom=kolom_harga, lama="teks_sku_asli", baru="_rasio")
    catat(tahap, df[tidak_dicek], "harga tidak bisa dicek pada baris hasil mapping salah ketik",
          "dibiarkan, ditandai harga_mapping_tidak_dicek", kolom=kolom_harga)
    print(f"{tahap}: baris hasil mapping salah ketik {dari_mapping.sum()} | harga sesuai {(dari_mapping & ~luar & ~tidak_dicek).sum()}"
          f" | DITOLAK {luar.sum()} | tidak bisa dicek (ditandai) {tidak_dicek.sum()}")
    return df

### 5.5 Membersihkan `offline`
Tabel kerja `offline` dimulai dari salinan `offline_raw`. Setiap langkah di bawah menangani satu masalah dari bagian 4.2.

In [93]:
offline = offline_raw.copy()
offline["alasan_ditolak"] = None    # diisi jika baris ditolak
n_masuk_offline = len(offline)
print("Baris masuk:", n_masuk_offline)

Baris masuk: 55776


**Masalah 4.2-b: baris kosong dan baris SUBTOTAL/TOTAL** → dicatat, lalu dibuang (bukan transaksi).

In [94]:
is_kosong = offline[kolom_off].isna().all(axis=1)
catat("offline", offline[is_kosong], "baris kosong", "dibuang")
is_total = offline["nama_produk"].astype("string").str.upper().str.contains("TOTAL", na=False) & offline["tanggal"].isna()
catat("offline", offline[is_total], "baris subtotal/total", "dibuang", lama="nama_produk")
offline = offline[~is_kosong & ~is_total].copy()
print("Dibuang:", is_kosong.sum(), "baris kosong +", is_total.sum(), "baris subtotal/total. Sisa:", len(offline))

Dibuang: 0 baris kosong + 54 baris subtotal/total. Sisa: 55722


**Masalah 4.2-g: duplikat persis (K7)** → dicek pada data **mentah** (sebelum apa pun dibersihkan), salinan dibuang, baris pertama disimpan.

In [95]:
dup = offline.duplicated(subset=["berkas"] + kolom_off, keep="first")
catat("offline", offline[dup], "baris duplikat persis (semua kolom mentah identik)", "dibuang (salinan pertama disimpan)")
duplikat_offline = offline[dup].groupby("berkas").size().rename("duplikat_dibuang")
offline = offline[~dup].copy()
print("Duplikat dibuang:", dup.sum())
duplikat_offline.sort_values(ascending=False).head()

Duplikat dibuang: 404


berkas
rekap_penjualan_offline_2025-11.xlsx    21
rekap_penjualan_offline_2026-01.xlsx    18
rekap_penjualan_offline_2026-04.xlsx    18
rekap_penjualan_offline_2026-02.xlsx    17
rekap_penjualan_offline_2025-10.xlsx    16
Name: duplikat_dibuang, dtype: int64

**Masalah 4.2-d: format tanggal campur dan tahun salah ketik** → semua tanggal diubah dengan `parse_tanggal`. Jika bulannya sama dengan bulan berkas tetapi tahunnya beda, tahunnya diganti. Tanggal yang tetap di luar bulan berkas ditolak.

In [96]:
offline["tanggal_asli"] = offline["tanggal"]
offline["tanggal"] = offline["tanggal"].map(parse_tanggal)

tahun_berkas = offline["bulan_berkas"].str[:4].astype(int)
bulan_berkas = offline["bulan_berkas"].str[5:7].astype(int)
salah_tahun = (offline["tanggal"].dt.month == bulan_berkas) & (offline["tanggal"].dt.year != tahun_berkas)
offline["tanggal_sebelum"] = offline["tanggal"]
offline.loc[salah_tahun, "tanggal"] = [t.replace(year=y) for t, y in
                                       zip(offline.loc[salah_tahun, "tanggal"], tahun_berkas[salah_tahun])]
catat("offline", offline[salah_tahun], "tahun tanggal salah ketik", "tahun diganti sesuai bulan berkas",
      kolom="tanggal", lama="tanggal_sebelum", baru="tanggal")

di_luar = offline["tanggal"].isna() | (offline["tanggal"].dt.strftime("%Y-%m") != offline["bulan_berkas"])
tolak(offline, di_luar, "tanggal kosong/di luar bulan berkas")
catat("offline", offline[di_luar], "tanggal kosong/di luar bulan berkas", "ditolak (dilaporkan)", kolom="tanggal", lama="tanggal_asli")
print("Tahun diperbaiki:", salah_tahun.sum(), "| tanggal masih di luar bulan berkas:", di_luar.sum())

Tahun diperbaiki: 51 | tanggal masih di luar bulan berkas: 0


**Masalah 4.2-e: angka berupa teks** → qty, harga, diskon, subtotal diubah ke angka. Diskon kosong berarti 0.

In [97]:
offline["harga_asli"] = offline["harga_satuan"]
for kolom in ["qty", "harga_satuan", "diskon", "subtotal"]:
    offline[kolom] = offline[kolom].map(parse_angka)
harga_teks = offline["harga_asli"].map(lambda v: isinstance(v, str))
catat("offline", offline[harga_teks], "harga berupa teks ('Rp 1.000'/'13,500')", "diubah ke angka",
      kolom="harga_satuan", lama="harga_asli", baru="harga_satuan")
offline["diskon"] = offline["diskon"].fillna(0)
print("Harga teks diubah:", harga_teks.sum())
offline[["qty", "harga_satuan", "diskon", "subtotal"]].dtypes

Harga teks diubah: 1620


qty             float64
harga_satuan    float64
diskon          float64
subtotal        float64
dtype: object

**Masalah 4.2-f: nota 4 digit (K11)** → dilengkapi tahun-bulan dari **tanggal transaksi** yang sudah dibersihkan (`0002` → `2311-0002`).

In [98]:
offline["nota_asli"] = offline["no_nota"]
offline["no_nota"] = offline["no_nota"].map(lambda v: None if kosong(v) else str(v).strip())
empat_digit = offline["no_nota"].map(lambda v: isinstance(v, str) and re.fullmatch(r"\d{4}", v) is not None)
prefiks = offline["tanggal"].dt.strftime("%y%m")                       # 2023-11-05 -> '2311'
prefiks = prefiks.fillna(offline["bulan_berkas"].str[2:4] + offline["bulan_berkas"].str[5:7])   # cadangan jika tanggal kosong
offline.loc[empat_digit, "no_nota"] = prefiks[empat_digit] + "-" + offline.loc[empat_digit, "no_nota"]
catat("offline", offline[empat_digit], "nomor nota tidak lengkap (4 digit)", "dilengkapi YYMM- dari tanggal transaksi",
      kolom="no_nota", lama="nota_asli", baru="no_nota")
print("Nota dilengkapi:", empat_digit.sum())
offline.loc[empat_digit, ["tanggal", "nota_asli", "no_nota"]].head(3)

Nota dilengkapi: 15835


,tanggal,nota_asli,no_nota
3,2023-11-01,0002,2311-0002
5,2023-11-01,0003,2311-0003
7,2023-11-01,0004,2311-0004


**Masalah 4.2-h: qty negatif (K5)** → semua qty negatif dengan subtotal positif dianggap salah ketik tanda dan dibalik (`qty_sign_fixed = True`). Retur pembeli sudah tercatat di stok masuk/keluar, dan retur tidak punya nota asal maupun harga, jadi baris ini tidak diperlakukan sebagai retur. Qty negatif dengan subtotal negatif/nol (jika ada) ditolak.

In [99]:
negatif = offline["qty"] < 0
dibalik = negatif & (offline["subtotal"] > 0)
offline["qty_sebelum"] = offline["qty"]
offline["qty_sign_fixed"] = dibalik
offline.loc[dibalik, "qty"] = offline.loc[dibalik, "qty"].abs()
catat("offline", offline[dibalik], "qty negatif dengan subtotal positif", "dijadikan positif (qty_sign_fixed = True)",
      kolom="qty", lama="qty_sebelum", baru="qty")
tidak_wajar = negatif & ~dibalik
tolak(offline, tidak_wajar, "qty negatif dengan subtotal negatif/nol")
catat("offline", offline[tidak_wajar], "qty negatif dengan subtotal negatif/nol", "ditolak (dilaporkan)", kolom="qty", lama="qty")
print("Qty negatif:", negatif.sum(), "| dibalik:", dibalik.sum(), "| ditolak:", tidak_wajar.sum())

Qty negatif: 57 | dibalik: 57 | ditolak: 0


**Masalah 4.2-e: qty kosong dan harga kosong/0** → dihitung dari subtotal. Qty hanya diisi jika hasilnya bilangan bulat positif; yang tidak bisa dihitung ditolak.

In [100]:
qty_kosong = offline["qty"].isna()
qty_hitung = (offline["subtotal"] + offline["diskon"]) / offline["harga_satuan"]
bisa = qty_kosong & np.isfinite(qty_hitung) & np.isclose(qty_hitung, qty_hitung.round()) & (qty_hitung > 0)
offline.loc[bisa, "qty"] = qty_hitung[bisa].round()
catat("offline", offline[bisa], "qty kosong", "dihitung dari (subtotal + diskon) / harga", kolom="qty", baru="qty")
tidak_bisa = qty_kosong & ~bisa
tolak(offline, tidak_bisa, "qty kosong dan tidak bisa dihitung")
catat("offline", offline[tidak_bisa], "qty kosong dan tidak bisa dihitung", "ditolak (dilaporkan)", kolom="qty")

harga_kosong = (offline["harga_satuan"].isna() | (offline["harga_satuan"] == 0)) & offline["qty"].notna()
offline["harga_sebelum"] = offline["harga_satuan"]
offline.loc[harga_kosong, "harga_satuan"] = (offline["subtotal"] + offline["diskon"]) / offline["qty"].abs()
catat("offline", offline[harga_kosong], "harga kosong atau 0", "dihitung dari (subtotal + diskon) / qty",
      kolom="harga_satuan", lama="harga_sebelum", baru="harga_satuan")
print("Qty diisi:", bisa.sum(), "| qty tidak bisa dihitung:", tidak_bisa.sum(), "| harga diisi:", harga_kosong.sum())

Qty diisi: 260 | qty tidak bisa dihitung: 7 | harga diisi: 625


**Masalah 4.2-e: subtotal tidak cocok dengan qty × harga − diskon** → subtotal **dipertahankan** (omzet offline = subtotal). Jika subtotal tepat sama dengan harga × bilangan bulat lain (mis. qty tertulis 10, subtotal = 1 × harga), yang salah ketik adalah qty, jadi qty diganti dari subtotal (K5, `qty_diperbaiki_dari_subtotal`). Sisanya diberi tanda `subtotal_selisih`.

In [101]:
qty_dari_subtotal = (offline["subtotal"] + offline["diskon"]) / offline["harga_satuan"]
qty_salah = (offline["alasan_ditolak"].isna() & (offline["harga_satuan"] > 0) & np.isfinite(qty_dari_subtotal)
             & np.isclose(qty_dari_subtotal, qty_dari_subtotal.round()) & (qty_dari_subtotal.round() > 0)
             & (qty_dari_subtotal.round() != offline["qty"]))
offline["qty_sebelum_subtotal"] = offline["qty"]
offline.loc[qty_salah, "qty"] = qty_dari_subtotal[qty_salah].round()
offline["qty_diperbaiki_dari_subtotal"] = qty_salah
catat("offline", offline[qty_salah], "qty tidak cocok dengan subtotal (subtotal = harga x bilangan bulat lain)",
      "diganti dari subtotal (qty_diperbaiki_dari_subtotal = True)", kolom="qty", lama="qty_sebelum_subtotal", baru="qty")
print("Qty diganti dari subtotal:", qty_salah.sum())
print("Pola qty lama -> baru:", offline.loc[qty_salah, ["qty_sebelum_subtotal", "qty"]].astype(int).value_counts().head(5).to_dict())

offline["subtotal_hitung"] = offline["qty"].abs() * offline["harga_satuan"] - offline["diskon"]
offline["subtotal_selisih"] = (offline["subtotal_hitung"] - offline["subtotal"]).abs() > 0.5
catat("offline", offline[offline["subtotal_selisih"]], "subtotal tidak sama dengan qty x harga - diskon",
      "subtotal dipertahankan (omzet offline = subtotal), diberi tanda subtotal_selisih",
      kolom="subtotal", lama="subtotal_hitung", baru="subtotal")
print("Baris bertanda subtotal_selisih:", offline["subtotal_selisih"].sum())

Qty diganti dari subtotal: 104
Pola qty lama -> baru: {(10, 1): 61, (20, 2): 28, (60, 6): 6, (40, 4): 3, (30, 3): 3}
Baris bertanda subtotal_selisih: 812


**Masalah 4.2-f: nota kosong (K11)** → diisi nota baris atas jika baris atas & bawah (dalam file yang sama) bernomor sama **dan** tanggalnya sama. Selain itu diberi ID `TANPA-NOTA-...` dan `order_no_generated = True`.

In [102]:
offline = offline.sort_values(["berkas", "baris_excel"])
nota_atas = offline.groupby("berkas")["no_nota"].shift(1)
nota_bawah = offline.groupby("berkas")["no_nota"].shift(-1)
tgl_atas = offline.groupby("berkas")["tanggal"].shift(1)
tgl_bawah = offline.groupby("berkas")["tanggal"].shift(-1)

nota_kosong = offline["no_nota"].isna()
bisa_isi = (nota_kosong & nota_atas.notna() & (nota_atas == nota_bawah)
            & (tgl_atas == offline["tanggal"]) & (tgl_bawah == offline["tanggal"]))
offline.loc[bisa_isi, "no_nota"] = nota_atas[bisa_isi]
catat("offline", offline[bisa_isi], "nomor nota kosong", "diisi dari baris atas & bawah (nota dan tanggal sama)",
      kolom="no_nota", baru="no_nota")

sisa = nota_kosong & ~bisa_isi
offline["order_no_generated"] = sisa
offline.loc[sisa, "no_nota"] = ("TANPA-NOTA-" + offline.loc[sisa, "bulan_berkas"].str.replace("-", "")
                                + "-" + offline.loc[sisa, "baris_excel"].astype(str))
catat("offline", offline[sisa], "nomor nota kosong", "diberi ID TANPA-NOTA (order_no_generated = True)",
      kolom="no_nota", baru="no_nota")
print("Nota kosong:", nota_kosong.sum(), "| diisi dari tetangga:", bisa_isi.sum(), "| diberi ID TANPA-NOTA:", sisa.sum())

Nota kosong: 2758 | diisi dari tetangga: 385 | diberi ID TANPA-NOTA: 2373


**Masalah 4.2-i: nama produk tidak baku (K1, K3)** → setiap nama unik dicocokkan ke master dan mapping tersimpan. Baris yang sudah ditolak di langkah sebelumnya tidak ikut dipetakan.

In [103]:
belum_ditolak = offline["alasan_ditolak"].isna()
dipetakan = petakan_teks(offline[belum_ditolak], "nama_produk")
offline = pd.concat([dipetakan, offline[~belum_ditolak]]).sort_values(["berkas", "baris_excel"])
print("Variasi nama:", dipetakan["teks_normal"].nunique())
dipetakan.groupby(["status_cocok", "aturan_sku"]).size().rename("jumlah_baris")

Variasi nama: 1251


status_cocok   aturan_sku               
cocok          kode                          7446
               mapping tersimpan -> kode      149
               mapping tersimpan -> nama      742
               nama                         33684
               nama terpotong                5917
ganda          mapping tersimpan -> nama      117
               nama                          5553
               nama terpotong                1155
               nama terpotong ambigu          462
tidak_dikenal  tidak dikenal                   86
Name: jumlah_baris, dtype: int64

**Masalah 4.1 / 4.2-i: nama yang dipakai beberapa SKU (K2)** → ukuran dipilih dari harga yang berlaku pada bulan nota. Riwayat harga dibentuk dulu dari baris yang SKU-nya pasti.

In [104]:
harga_master_offline = dict(zip(products["sku"], products["harga_jual_offline"]))
riwayat_offline = riwayat_harga(offline, "tanggal", "harga_satuan", "jual_offline")
acuan_offline = buat_acuan(riwayat_offline)
print("Riwayat harga jual offline:", len(riwayat_offline), "baris (SKU x bulan)")

offline = tentukan_ukuran(offline, "harga_satuan", "tanggal", acuan_offline, harga_master_offline)
print("Ukuran dipilih dari harga (sku_inferred):", offline["sku_inferred"].sum())
print("Nama beberapa ukuran yang ditolak:", offline["alasan_ditolak"].str.startswith("nama dipakai", na=False).sum())
offline.loc[offline["sku_inferred"] == True, ["tanggal", "nama_produk", "harga_satuan", "sku", "rincian_harga"]].head(4)

Riwayat harga jual offline: 4803 baris (SKU x bulan)
Ukuran dipilih dari harga (sku_inferred): 6956
Nama beberapa ukuran yang ditolak: 331


,tanggal,nama_produk,harga_satuan,sku,rincian_harga
1,2023-11-01,furadan 3gr,29000.0,FRD1,"FRD1 (rasio 1.00, bulan yang sama); FRD2 (rasio 0.68, bu..."
2,2023-11-01,Emkape Maxxi Agri,51000.0,EMKAPE1,"EMKAPE1 (rasio 1.00, bulan yang sama); EMKAPE500 (rasio ..."
19,2023-11-01,roundup 486 sl,26000.0,RDPK,"RDPB (rasio 0.29, bulan yang sama); RDPK (rasio 1.00, bu..."
22,2023-11-01,Antracol 70,118000.0,ANTRCL500,"ANTRCL (rasio 2.13, bulan yang sama); ANTRCL500 (rasio 1..."


**Cek harga untuk baris hasil mapping salah ketik (K3).** Salah ketik yang lolos aturan K3 tetap bisa salah SKU. Karena itu harga satuan dibandingkan dengan harga acuan SKU hasil mapping pada bulan nota (`harga_acuan`), dengan toleransi ±10% seperti K2. Di luar toleransi → ditolak. Harga kosong → tetap diterima, tetapi ditandai `harga_mapping_tidak_dicek`. Fungsi `cek_harga_mapping` (bagian 5.4) dipakai juga untuk online dan restok.

In [105]:
offline = cek_harga_mapping(offline, "harga_satuan", "tanggal", acuan_offline, harga_master_offline, "offline")
ditolak_harga = offline["alasan_ditolak"] == ALASAN_HARGA_MAPPING
offline.loc[ditolak_harga, ["berkas", "baris_excel", "nama_produk", "sku", "harga_satuan", "rasio_harga_mapping"]]

offline: baris hasil mapping salah ketik 1008 | harga sesuai 1007 | DITOLAK 1 | tidak bisa dicek (ditandai) 0


,berkas,baris_excel,nama_produk,sku,harga_satuan,rasio_harga_mapping
15434,rekap_penjualan_offline_2024-09.xlsx,1233,KOVINP LUS 80P,KOVINPLUS,3500.0,1.166667


In [106]:
# Catat semua pemetaan ke log, lalu cek SKU induk (tidak boleh terjual) dan harga tidak wajar
catat_pemetaan(offline, "offline")
induk_terjual = offline["sku"].isin(SKU_INDUK_TIDAK_DIJUAL)
tolak(offline, induk_terjual, "SKU induk tidak dijual")
offline = cek_harga_wajar(offline, "harga_satuan", "tanggal", acuan_offline, harga_master_offline, "offline")
offline["alasan_ditolak"].value_counts()

alasan_ditolak
nama dipakai beberapa SKU: harga cocok ke lebih dari satu kandidat    330
SKU tidak dikenal (tidak ada di master maupun mapping tersimpan)       86
qty kosong dan tidak bisa dihitung                                      7
harga tidak sesuai SKU hasil mapping salah ketik                        1
nama dipakai beberapa SKU: harga jauh dari semua kandidat               1
Name: count, dtype: int64

**Masalah 4.2-g lanjutan: baris yang baru sama setelah dibersihkan (K7)** → mis. nota `0074` vs `2311-0074` dengan SKU, qty, dan subtotal yang sama. Baris ini **dipertahankan** dan ditandai `kemungkinan_duplikat`, karena pembeli bisa membeli barang yang sama dua kali. Tanda ini bisa ditampilkan ke admin saat import untuk dicek.

In [107]:
lolos = offline["alasan_ditolak"].isna()
kunci_dup = ["berkas", "tanggal", "no_nota", "sku", "qty", "harga_satuan", "diskon", "subtotal", "catatan"]
offline["kemungkinan_duplikat"] = lolos & offline.duplicated(subset=kunci_dup, keep=False)
catat("offline", offline[offline["kemungkinan_duplikat"]], "kemungkinan duplikat (baru sama setelah dibersihkan)",
      "dipertahankan, diberi tanda kemungkinan_duplikat")
print("Baris bertanda kemungkinan_duplikat (semua salinan):", offline["kemungkinan_duplikat"].sum())

Baris bertanda kemungkinan_duplikat (semua salinan): 114


**Pisahkan baris bersih dan baris yang ditolak, lalu simpan.**

In [108]:
kolom_offline_bersih = ["tanggal", "no_nota", "sku", "nama_produk", "qty", "harga_satuan", "diskon", "subtotal",
                        "aturan_sku", "sku_inferred", "qty_sign_fixed", "order_no_generated", "subtotal_selisih", "qty_diperbaiki_dari_subtotal", "kemungkinan_duplikat", "harga_mapping_tidak_dicek",
                        "catatan", "berkas", "baris_excel"]
offline_clean = offline[offline["alasan_ditolak"].isna()][kolom_offline_bersih].sort_values(["tanggal", "no_nota", "baris_excel"])
offline_clean["qty"] = offline_clean["qty"].astype(int)
kolom_tanda = ["sku_inferred", "qty_sign_fixed", "order_no_generated", "qty_diperbaiki_dari_subtotal",
               "kemungkinan_duplikat", "harga_mapping_tidak_dicek"]
offline_clean[kolom_tanda] = offline_clean[kolom_tanda].fillna(False).astype(bool)
offline_ditolak = laporan_ditolak(offline[offline["alasan_ditolak"].notna()], "offline", "nama_produk")
mapping_offline = tabel_mapping(offline[offline["status_cocok"].notna()], "offline")

offline_clean.to_csv(INTERIM / "offline_clean.csv", index=False)
offline_ditolak.to_csv(INTERIM / "offline_ditolak.csv", index=False)
print(offline_ditolak["alasan"].str.split(":").str[0].value_counts())
ringkasan_tahap("offline", n_masuk_offline, offline_clean, offline_ditolak)

alasan
nama dipakai beberapa SKU                                           331
SKU tidak dikenal (tidak ada di master maupun mapping tersimpan)     86
qty kosong dan tidak bisa dihitung                                    7
harga tidak sesuai SKU hasil mapping salah ketik                      1
Name: count, dtype: int64
offline: masuk 55776 = bersih 54893 + ditolak 425 + dibuang 458 -> SEIMBANG


,masalah,tindakan,jumlah_baris
0,pemetaan SKU otomatis (nama),dipetakan otomatis,33684
1,nomor nota tidak lengkap (4 digit),dilengkapi YYMM- dari tanggal transaksi,15835
2,pemetaan SKU otomatis (kode),dipetakan otomatis,7446
3,pemetaan SKU otomatis (nama terpotong),dipetakan otomatis,5917
4,pemetaan SKU dari harga (nama + harga),dipetakan dari harga (sku_inferred = True),5552
5,nomor nota kosong,diberi ID TANPA-NOTA (order_no_generated = True),2373
6,"harga berupa teks ('Rp 1.000'/'13,500')",diubah ke angka,1620
7,pemetaan SKU dari harga (nama terpotong + harga),dipetakan dari harga (sku_inferred = True),1155
8,subtotal tidak sama dengan qty x harga - diskon,"subtotal dipertahankan (omzet offline = subtotal), diber...",812
9,pemetaan SKU otomatis (mapping tersimpan -> nama),dipetakan otomatis,741


In [109]:
offline_clean.head()

,tanggal,no_nota,sku,nama_produk,qty,harga_satuan,diskon,subtotal,aturan_sku,sku_inferred,qty_sign_fixed,order_no_generated,subtotal_selisih,qty_diperbaiki_dari_subtotal,kemungkinan_duplikat,harga_mapping_tidak_dicek,catatan,berkas,baris_excel
0,2023-11-01,2311-0001,AGT,pupuk amazing grow pertumbuhan,1,19000.0,0.0,19000.0,nama,False,False,False,False,False,False,False,NaN,rekap_penjualan_offline_2023-11.xlsx,2
1,2023-11-01,2311-0002,FRD1,furadan 3gr,1,29000.0,0.0,29000.0,nama + harga,True,False,False,False,False,False,False,NaN,rekap_penjualan_offline_2023-11.xlsx,3
2,2023-11-01,2311-0002,EMKAPE1,Emkape Maxxi Agri,1,51000.0,0.0,51000.0,nama + harga,True,False,False,False,False,False,False,tempo,rekap_penjualan_offline_2023-11.xlsx,4
3,2023-11-01,2311-0002,UREA,UREA CURAH,3,12500.0,0.0,37500.0,nama,False,False,False,False,False,False,False,NaN,rekap_penjualan_offline_2023-11.xlsx,5
4,2023-11-01,2311-0003,MKP1,MKP PAK TANI,4,64500.0,0.0,258000.0,nama,False,False,False,False,False,False,False,NaN,rekap_penjualan_offline_2023-11.xlsx,6


**Dampak baris yang ditolak karena nama ambigu.** Baris seperti `pupuk amazing` (Amazing Grow Pertumbuhan atau Pembuahan, harga sama) dan `polybag tanaman` (8 ukuran) **tetap ditolak**: tidak dibagi proporsional dan tidak dibuat SKU gabungan, karena keduanya sama-sama mengarang data. Yang diukur di sini adalah seberapa besar omzet dan qty yang tidak ikut terhitung.

In [110]:
nama_ambigu = offline[offline["alasan_ditolak"].fillna("").str.startswith("nama dipakai beberapa SKU")].copy()
nama_ambigu["bulan"] = nama_ambigu["bulan_berkas"]
omzet_bulan = offline_clean.assign(bulan=offline_clean["berkas"].str.extract(r"(\d{4}-\d{2})")[0]).groupby("bulan")["subtotal"].sum()
semua_ditolak = offline[offline["alasan_ditolak"].notna()].groupby("bulan_berkas")["subtotal"].sum()

dampak_bulan = (nama_ambigu.groupby("bulan").agg(baris=("subtotal", "size"), qty=("qty", "sum"), subtotal=("subtotal", "sum"))
                .reindex(omzet_bulan.index, fill_value=0))
dampak_bulan["omzet_offline_bersih"] = omzet_bulan
dampak_bulan["persen_omzet"] = (dampak_bulan["subtotal"] / dampak_bulan["omzet_offline_bersih"] * 100).round(2)
dampak_bulan["persen_semua_ditolak"] = (semua_ditolak.reindex(omzet_bulan.index, fill_value=0) / omzet_bulan * 100).round(2)
total_persen = nama_ambigu["subtotal"].sum() / omzet_bulan.sum() * 100
print(f"Nama ambigu ditolak: {len(nama_ambigu)} baris, qty {nama_ambigu['qty'].sum():.0f}, subtotal Rp{nama_ambigu['subtotal'].sum():,.0f}")
print(f"= {total_persen:.2f}% omzet offline bersih | bulan tertinggi: {dampak_bulan['persen_omzet'].idxmax()} ({dampak_bulan['persen_omzet'].max():.2f}%)")
print("Bulan dengan dampak >= 1% omzet:", (dampak_bulan["persen_omzet"] >= 1).sum(), "dari", len(dampak_bulan))
dampak_bulan

Nama ambigu ditolak: 331 baris, qty 583, subtotal Rp11,689,500
= 0.36% omzet offline bersih | bulan tertinggi: 2026-10 (0.65%)
Bulan dengan dampak >= 1% omzet: 0 dari 36


,baris,qty,subtotal,omzet_offline_bersih,persen_omzet,persen_semua_ditolak
bulan,,,,,,
2023-11,13,22.0,419000.0,85072500.0,0.49,0.90
2023-12,12,28.0,531000.0,87521000.0,0.61,0.78
2024-01,7,15.0,285500.0,92195500.0,0.31,0.47
2024-02,8,13.0,254000.0,59675000.0,0.43,0.66
2024-03,6,11.0,214000.0,70859500.0,0.30,0.97
2024-04,7,16.0,318500.0,66799500.0,0.48,0.72
2024-05,13,25.0,513500.0,93155000.0,0.55,1.11
2024-06,16,24.0,477000.0,75240500.0,0.63,0.65
2024-07,6,9.0,180500.0,66111500.0,0.27,0.37


In [111]:
# Rincian per teks dan perbandingan dengan qty SKU yang berhasil dipetakan
per_teks = (nama_ambigu.groupby("teks_normal")
            .agg(baris=("qty", "size"), qty=("qty", "sum"), subtotal=("subtotal", "sum"),
                 kandidat=("kandidat", lambda s: ", ".join(sorted({k for daftar in s for k in daftar}))))
            .sort_values("subtotal", ascending=False))
display(per_teks)

KELOMPOK_BANDING = {"pupuk amazing": ["AGT", "AGB"],
                    "polybag tanaman": ["PLYBG15", "PLYBG20", "PLYBG25", "PLYBG30", "PLYBG35", "PLYBG40", "PLYBG45", "PLYBG50"]}
baris_banding = []
for teks, skus in KELOMPOK_BANDING.items():
    qty_ditolak = nama_ambigu.loc[nama_ambigu["teks_normal"] == teks, "qty"].sum()
    qty_masuk = offline_clean.loc[offline_clean["sku"].isin(skus), "qty"].sum()
    baris_banding.append({"teks_ditolak": teks, "sku_pembanding": ", ".join(skus), "qty_ditolak": qty_ditolak,
                          "qty_terpetakan": qty_masuk, "persen_qty_hilang": round(qty_ditolak / (qty_ditolak + qty_masuk) * 100, 1)})
pd.DataFrame(baris_banding)

,baris,qty,subtotal,kandidat
teks_normal,,,,
polybag tanaman,182,318.0,6661500.0,"PLYBG15, PLYBG20, PLYBG25, PLYBG30, PLYBG35, PLYBG40, PL..."
pupuk amazing,148,255.0,4939500.0,"AGB, AGT"
roundup 486 sl,1,10.0,88500.0,"RDPB, RDPK"


,teks_ditolak,sku_pembanding,qty_ditolak,qty_terpetakan,persen_qty_hilang
0,pupuk amazing,"AGT, AGB",255.0,1115,18.6
1,polybag tanaman,"PLYBG15, PLYBG20, PLYBG25, PLYBG30, PLYBG35, PLYBG40, PL...",318.0,1272,20.0


**Polybag: apakah harganya memang tidak bisa membedakan ukuran?** Untuk setiap baris `polybag tanaman` yang ditolak, harga satuannya dibandingkan dengan harga acuan kedelapan ukuran pada bulan yang sama. Jika banyak baris yang hanya cocok ke satu ukuran, berarti K2 seharusnya bisa memilih dan perlu diperiksa.

In [112]:
UKURAN_POLYBAG = ["PLYBG15", "PLYBG20", "PLYBG25", "PLYBG30", "PLYBG35", "PLYBG40", "PLYBG45", "PLYBG50"]
polybag = nama_ambigu[nama_ambigu["teks_normal"] == "polybag tanaman"].copy()
for sku in UKURAN_POLYBAG:
    polybag[sku] = [harga_acuan(sku, t, acuan_offline, harga_master_offline)[0] for t in polybag["tanggal"]]
rasio_polybag = polybag[UKURAN_POLYBAG].rdiv(polybag["harga_satuan"], axis=0)
polybag["ukuran_cocok"] = ((rasio_polybag - 1).abs() <= TOLERANSI_HARGA).sum(axis=1)
print("Baris polybag ditolak:", len(polybag))
print("Jumlah ukuran yang harganya dalam +-10%:")
print(polybag["ukuran_cocok"].value_counts().sort_index().rename("baris"))
print("\nHarga satuan baris ditolak:", polybag["harga_satuan"].describe()[["min", "50%", "max"]].to_dict())
print("Harga acuan 8 ukuran (median semua bulan):", polybag[UKURAN_POLYBAG].median().round(-2).astype(int).to_dict())

Baris polybag ditolak: 182
Jumlah ukuran yang harganya dalam +-10%:
ukuran_cocok
2     2
3    14
4    36
5    76
6    36
7    12
8     6
Name: baris, dtype: int64

Harga satuan baris ditolak: {'min': 17500.0, '50%': 21000.0, 'max': 23500.0}
Harga acuan 8 ukuran (median semua bulan): {'PLYBG15': 18000, 'PLYBG20': 21500, 'PLYBG25': 19500, 'PLYBG30': 19000, 'PLYBG35': 21500, 'PLYBG40': 22500, 'PLYBG45': 19500, 'PLYBG50': 20500}


In [113]:
# Contoh satu bulan: harga baris ditolak vs harga acuan tiap ukuran
contoh_bulan = polybag["bulan"].mode()[0]
display(polybag.loc[polybag["bulan"] == contoh_bulan, ["tanggal", "harga_satuan", "ukuran_cocok"] + UKURAN_POLYBAG].head(5))
satu_ukuran = polybag[polybag["ukuran_cocok"] == 1]
print("Baris yang hanya cocok ke satu ukuran:", len(satu_ukuran))
if len(satu_ukuran):
    display(satu_ukuran[["tanggal", "harga_satuan", "rincian_harga"]].head())

,tanggal,harga_satuan,ukuran_cocok,PLYBG15,PLYBG20,PLYBG25,PLYBG30,PLYBG35,PLYBG40,PLYBG45,PLYBG50
10311,2024-06-03,21000.0,6,17500.0,21500.0,19500.0,19000.0,21000.0,22000.0,19500.0,19500.0
10489,2024-06-07,21000.0,6,17500.0,21500.0,19500.0,19000.0,21000.0,22000.0,19500.0,19500.0
10575,2024-06-09,21000.0,6,17500.0,21500.0,19500.0,19000.0,21000.0,22000.0,19500.0,19500.0
10593,2024-06-09,19500.0,6,17500.0,21500.0,19500.0,19000.0,21000.0,22000.0,19500.0,19500.0
10691,2024-06-11,21000.0,6,17500.0,21500.0,19500.0,19000.0,21000.0,22000.0,19500.0,19500.0


Baris yang hanya cocok ke satu ukuran: 0


**Simpan `penjualan_ambigu.csv`.** Baris yang ditolak karena nama dipakai beberapa SKU tetap **tidak** masuk `transactions` dan `stock_movements`, tetapi disimpan di `data/processed/penjualan_ambigu.csv`. Tabel ini dipakai di notebook 03 untuk menyesuaikan target forecast (lihat catatan di bawah); dashboard penjualan tetap memakai angka strict.

In [114]:
penjualan_ambigu = pd.DataFrame({
    "tanggal": nama_ambigu["tanggal"], "bulan": nama_ambigu["bulan"], "kanal": "Offline",
    "teks_asli": nama_ambigu["teks_sku_asli"], "qty": nama_ambigu["qty"], "subtotal": nama_ambigu["subtotal"],
    "sku_kandidat": nama_ambigu["kandidat"].map(lambda k: ", ".join(sorted(k))), "alasan": nama_ambigu["alasan_ditolak"],
    "berkas": nama_ambigu["berkas"], "baris_excel": nama_ambigu["baris_excel"]}).reset_index(drop=True)
print("penjualan_ambigu (offline):", len(penjualan_ambigu), "baris")
penjualan_ambigu.head()

penjualan_ambigu (offline): 331 baris


,tanggal,bulan,kanal,teks_asli,qty,subtotal,sku_kandidat,alasan,berkas,baris_excel
0,2023-11-02,2023-11,Offline,Pupuk Amazing,1.0,18500.0,"AGB, AGT",nama dipakai beberapa SKU: harga cocok ke lebih dari sat...,rekap_penjualan_offline_2023-11.xlsx,29
1,2023-11-03,2023-11,Offline,Polybag Tanaman,1.0,21500.0,"PLYBG15, PLYBG20, PLYBG25, PLYBG30, PLYBG35, PLYBG40, PL...",nama dipakai beberapa SKU: harga cocok ke lebih dari sat...,rekap_penjualan_offline_2023-11.xlsx,121
2,2023-11-06,2023-11,Offline,Pupuk Amazing,1.0,18500.0,"AGB, AGT",nama dipakai beberapa SKU: harga cocok ke lebih dari sat...,rekap_penjualan_offline_2023-11.xlsx,305
3,2023-11-07,2023-11,Offline,Pupuk Amazing,1.0,19000.0,"AGB, AGT",nama dipakai beberapa SKU: harga cocok ke lebih dari sat...,rekap_penjualan_offline_2023-11.xlsx,326
4,2023-11-08,2023-11,Offline,Polybag Tanaman,1.0,19000.0,"PLYBG15, PLYBG20, PLYBG25, PLYBG30, PLYBG35, PLYBG40, PL...",nama dipakai beberapa SKU: harga cocok ke lebih dari sat...,rekap_penjualan_offline_2023-11.xlsx,399


> **Catatan untuk notebook 03 (modeling):**
> 1. Sebelum training, qty di `penjualan_ambigu` dibagi ke SKU kandidat sesuai **porsi qty terpetakan** masing-masing kandidat pada bulan yang sama. Bulan tanpa penjualan kandidat memakai porsi keseluruhan. Hasilnya disimpan sebagai kolom terpisah `qty_penyesuaian`, dan target = `qty + qty_penyesuaian`.
> 2. Hasil forecast dengan dan tanpa penyesuaian dibandingkan di data uji.
> 3. Dashboard penjualan tetap memakai angka strict (tanpa penyesuaian).

**Insight dampak baris yang ditolak karena nama ambigu:**
- **Dampak ke omzet di bawah 1%.** 331 baris (qty 583, Rp11,7 juta) setara **0,36% omzet offline bersih**. Tidak ada bulan yang mencapai 1%; bulan tertinggi Okt 2026 (0,65%). Jika semua alasan penolakan dihitung, hanya dua bulan yang sedikit di atas 1% (Mei 2024 1,11% dan Jul 2026 1,19%). Untuk omzet total dan per kanal (Q1, Q2), ini **keterbatasan data yang dapat diterima**.
- **Dampak ke SKU tertentu jauh lebih besar.** Hampir semua baris berasal dari dua teks:
  - `pupuk amazing` (qty 255) setara **18,6%** dari total qty AGT + AGB. Harga kedua SKU sama (Rp20.000), jadi harga tidak bisa membedakannya.
  - `polybag tanaman` (qty 318) setara **20,0%** dari total qty PLYBG15 sampai PLYBG50. Harga delapan ukuran polybag saling berdekatan (Rp19.000–Rp23.500).
- **Bukti harga polybag tumpang tindih.** Harga acuan kedelapan ukuran polybag hanya berkisar Rp18.000–Rp22.500, sehingga setiap baris yang ditolak cocok (±10%) ke **2 sampai 8 ukuran sekaligus** (paling banyak 5 ukuran, 76 baris). **Tidak ada** baris yang hanya cocok ke satu ukuran, jadi K2 menolaknya dengan benar: harga memang tidak bisa membedakan ukuran polybag. Pada Pupuk Amazing, harga AGT dan AGB sama persis.
- **Stok sistem lebih tinggi dari fisik.** Karena baris ini tidak mengurangi stok, stok sistem AGT, AGB, dan PLYBG15–PLYBG50 akan **lebih tinggi** daripada stok fisik. Di Q6, selisih kurang (fisik < sistem) pada SKU ini sebagian berasal dari penjualan yang ditolak, bukan kehilangan barang.
- **Forecast berisiko terlalu rendah** untuk **AGT, AGB, dan PLYBG15–PLYBG50**, sekitar seperlima permintaan offline mereka tidak tercatat. Di modeling, target SKU ini disesuaikan memakai `penjualan_ambigu.csv` (lihat catatan untuk notebook 03), dan alert stok minimum (Q8) sebaiknya diberi margin lebih. Sisanya (1 baris Roundup) dapat diabaikan.

> **Catatan untuk UMKM:** kasir sebaiknya menulis **varian dan ukuran lengkap** di rekap, mis. `Pupuk Amazing Grow Pertumbuhan` / `Pembuahan` dan `Polybag Tanaman 30x30`, atau memakai kode SKU. Satu kata tambahan di rekap mencegah hampir seperlima penjualan dua kelompok produk ini hilang dari analisis.

**Ringkasan tahap `offline`:**
- **Masuk 55.776 baris → bersih 54.893 baris (98,4%)**, ditolak 425, dibuang 458 (404 duplikat mentah, 54 subtotal/total). Jumlah duplikat per berkas ada di `duplikat_per_berkas`.
- **114 baris yang baru sama setelah dibersihkan dipertahankan** dan ditandai `kemungkinan_duplikat` (K7), karena pembeli bisa membeli barang yang sama dua kali dengan nota berbeda.
- Diperbaiki: 15.835 nota dilengkapi dari tanggal transaksi, 2.758 nota kosong diisi (385 dari baris tetangga, 2.373 diberi ID `TANPA-NOTA`), 1.620 harga teks, 625 harga kosong dan 260 qty kosong dihitung dari subtotal, 51 tahun tanggal, **57 qty negatif dibalik** (`qty_sign_fixed = True`), dan **104 qty yang tertulis 10× lipat diganti dari subtotal** (mis. 10 → 1, 20 → 2; `qty_diperbaiki_dari_subtotal = True`, K5). Tanpa perbaikan ini qty offline berlebih 1.728 unit.
- **Semua pemetaan tercatat di log** beserta teks asli, SKU hasil, dan aturannya. Ada **6.956 baris** yang ukurannya dipilih dari harga pada bulan nota (`sku_inferred = True`).
- **Mapping salah ketik (K3)**: dari 1.010 teks historis yang tidak dikenali, **911 teks** masuk mapping tersimpan dan 99 ambigu. Sebagian teks ambigu karena berjarak satu ketikan dari dua SKU, mis. `mpk pak tani` (MKP atau NPK), `mp1` (M21 atau MKP1), dan `metazeb 8 0wp` (METAZEB atau MEGAZEB).
- **Cek harga baris hasil mapping** (`cek_harga_mapping`, dipakai juga di online dan restok): dari 1.008 baris, 1.007 harganya dalam ±10% harga acuan SKU hasil. 1 baris ditolak (`kovinp lus 80p`, harga 1,17× Kovin Plus), dan tidak ada baris berharga kosong.
- **Ditolak 425 baris** dan dilaporkan di `offline_ditolak.csv`:
  - 331 nama yang cocok ke beberapa ukuran dengan harga yang tidak bisa membedakan: `polybag tanaman` (182 baris), `pupuk amazing` (148 baris), dan 1 baris Roundup;
  - 86 teks yang tidak dikenali atau ambigu (`mpk pak tani`, `metaze 80 wp`, ...);
  - 1 baris harga tidak sesuai SKU hasil mapping salah ketik;
  - 7 baris qty kosong yang tidak bisa dihitung dari subtotal.
- Dampak nama ambigu: **0,36% omzet offline**, tetapi qty AGT/AGB dan PLYBG15–50 tercatat sekitar 19–20% lebih rendah (lihat analisis dampak di atas). Ke-331 baris ini disimpan di `penjualan_ambigu.csv` untuk penyesuaian target di modeling.

### 5.6 Membersihkan `online`
Setiap langkah menangani satu masalah dari bagian 4.3.

In [115]:
online = online_raw.copy()
online["alasan_ditolak"] = None
n_masuk_online = len(online)
print("Baris masuk:", n_masuk_online)

Baris masuk: 5220


**Masalah 4.3-a: baris kosong, baris TOTAL, header terulang** → dicatat lalu dibuang.

In [116]:
is_kosong = online[KOLOM_ISI_ONLINE].isna().all(axis=1)
is_header = online["no"].astype("string").str.strip().str.upper().eq("NO").fillna(False).astype(bool)
is_total = (online["no_order"].isna() & ~is_kosong
            & online[KOLOM_ISI_ONLINE].astype("string").apply(lambda s: s.str.strip().str.upper().eq("TOTAL").fillna(False)).any(axis=1))
catat("online", online[is_kosong], "baris kosong", "dibuang")
catat("online", online[is_total], "baris TOTAL", "dibuang")
catat("online", online[is_header], "baris header terulang di tengah file", "dibuang")
online = online[~is_kosong & ~is_total & ~is_header].copy()
print("Dibuang:", is_kosong.sum(), "kosong,", is_total.sum(), "TOTAL,", is_header.sum(), "header. Sisa:", len(online))

Dibuang: 22 kosong, 14 TOTAL, 22 header. Sisa: 5162


**Masalah 4.3-f: duplikat persis (K7)** → semua kolom mentah identik (termasuk nomor baris `NO`), dicek sebelum pembersihan.

In [117]:
dup = online.duplicated(subset=["berkas"] + KOLOM_ISI_ONLINE, keep="first")
catat("online", online[dup], "baris duplikat persis (semua kolom mentah identik)", "dibuang (salinan pertama disimpan)")
duplikat_online = online[dup].groupby("berkas").size().rename("duplikat_dibuang")
online = online[~dup].copy()
print("Duplikat dibuang:", dup.sum())

Duplikat dibuang: 16


**Masalah 4.3-b dan 4.3-h: kolom yang tidak dipakai (K15)** → `NO HP` (data pribadi pembeli), `DAERAH` (tidak dipakai di Q1–Q12), `TOTAL` (sama dengan NETTO), `LABA BERSIH TOTAL` (sel gabungan, hanya terisi di sel pertama), dan `BIAYA BALIK KULAKAN` (sama dengan HARGA KULAKAN) dibuang. Pembuangan dicatat per berkas.

In [118]:
KOLOM_DIBUANG_ONLINE = {"no_hp": "data pribadi pembeli", "daerah": "tidak dipakai di Q1-Q12",
                        "total": "sama dengan NETTO", "laba_bersih_total": "sel gabungan, laba dihitung ulang per baris",
                        "biaya_balik_kulakan": "sama dengan HARGA KULAKAN"}
per_berkas = online.groupby("berkas").size().rename("jumlah").reset_index()
per_berkas["baris_excel"] = None
per_berkas["keterangan"] = per_berkas["jumlah"].astype(str) + " baris di berkas ini"
for kolom, alasan in KOLOM_DIBUANG_ONLINE.items():
    catat("online", per_berkas, f"kolom {kolom} tidak dipakai ({alasan})", "kolom tidak dipakai (dibuang dari tabel)",
          kolom=kolom, lama="keterangan")
online = online.drop(columns=list(KOLOM_DIBUANG_ONLINE))
print("Kolom dibuang:", list(KOLOM_DIBUANG_ONLINE))

Kolom dibuang: ['no_hp', 'daerah', 'total', 'laba_bersih_total', 'biaya_balik_kulakan']


**Masalah 4.3-h: nama kanal tidak seragam (K13)** → nama dari nama file (`TikTokShop`) dibakukan dengan kamus kanal menjadi `TikTok Shop`, sama dengan nama di metrik marketplace.

In [119]:
online["kanal"] = online["platform"].map(lambda v: KAMUS_KANAL[normalisasi_teks(v)])
berubah = online["kanal"] != online["platform"]
catat("online", online[berubah], "nama kanal tidak baku", "dibakukan dengan kamus kanal", kolom="kanal", lama="platform", baru="kanal")
online = online.drop(columns="platform")
print(online["kanal"].value_counts())

kanal
Shopee         2330
TikTok Shop    1915
Tokopedia       901
Name: count, dtype: int64


**Masalah 4.3-c: nomor pesanan 18 digit** → disimpan sebagai **teks**, lalu dicek polanya per platform. Yang tidak sesuai pola ditolak.

In [120]:
POLA_NO_ORDER = {"Shopee": r"[0-9A-Z]{14}", "Tokopedia": r"INV/\d{8}/MPL/\d{9}", "TikTok Shop": r"\d{18}"}
online["no_order"] = online["no_order"].map(lambda v: None if kosong(v) else str(v).strip())
pola_ok = pd.Series([isinstance(o, str) and re.fullmatch(POLA_NO_ORDER[p], o) is not None
                     for p, o in zip(online["kanal"], online["no_order"])], index=online.index)
tolak(online, ~pola_ok, "format nomor pesanan tidak valid")
catat("online", online[~pola_ok], "format nomor pesanan tidak valid", "ditolak (dilaporkan)", kolom="no_order", lama="no_order")
print("Nomor pesanan tidak valid:", (~pola_ok).sum())
print(online.groupby("kanal")["no_order"].apply(lambda s: s.str.len().unique().tolist()))

Nomor pesanan tidak valid: 0
kanal
Shopee         [14]
TikTok Shop    [18]
Tokopedia      [26]
Name: no_order, dtype: object


**Masalah 4.3-d/e: tanggal dan angka berupa teks** → tanggal pesanan disimpan di kolom **`order_date`** (K8); `HARGA` 'Rp 78.168' dan `DISKON` '12%' diubah ke angka.

In [121]:
online["tanggal_asli"] = online["tanggal_pesanan"]
online["order_date"] = online["tanggal_pesanan"].map(parse_tanggal)
online["harga_asli"] = online["harga"]
online["diskon_asli"] = online["diskon_persen"]
kolom_angka_online = ["harga", "qty", "bruto", "diskon_persen", "rp_diskon_platform", "rp_diskon", "netto",
                      "ongkir", "promo_ongkir", "biaya_program_ongkir", "sisa_ongkir_pembeli", "komisi_platform",
                      "komisi_affiliator", "laba_kotor", "harga_kulakan", "biaya_packing", "laba_bersih"]
for kolom in kolom_angka_online:
    online[kolom] = online[kolom].map(parse_angka)
catat("online", online[online["harga_asli"].map(lambda v: isinstance(v, str))], "harga berupa teks ('Rp 78.168')",
      "diubah ke angka", kolom="harga", lama="harga_asli", baru="harga")
catat("online", online[online["diskon_asli"].map(lambda v: isinstance(v, str))], "diskon berupa teks ('12%')",
      "diubah ke pecahan", kolom="diskon_persen", lama="diskon_asli", baru="diskon_persen")
online["rp_diskon_platform"] = online["rp_diskon_platform"].fillna(0)
print("Tanggal tidak terbaca:", online["order_date"].isna().sum())

Tanggal tidak terbaca: 0


**Masalah 4.3-e: HARGA = 0 dan QTY tidak cocok dengan BRUTO** → harga = bruto / qty; qty = bruto / harga (BRUTO konsisten dengan netto, diskon, komisi, dan harga kulakan).

In [122]:
harga_nol = (online["harga"].isna() | (online["harga"] == 0)) & (online["qty"] > 0)
online["harga_sebelum"] = online["harga"]
online.loc[harga_nol, "harga"] = online["bruto"] / online["qty"]
catat("online", online[harga_nol], "harga 0 atau kosong", "dihitung dari bruto / qty", kolom="harga", lama="harga_sebelum", baru="harga")

qty_dari_bruto = online["bruto"] / online["harga"]
qty_salah = qty_dari_bruto.notna() & np.isclose(qty_dari_bruto, qty_dari_bruto.round()) & (qty_dari_bruto.round() != online["qty"])
online["qty_sebelum"] = online["qty"]
online.loc[qty_salah, "qty"] = qty_dari_bruto[qty_salah].round()
catat("online", online[qty_salah], "qty tidak sesuai bruto / harga", "qty diganti = bruto / harga", kolom="qty", lama="qty_sebelum", baru="qty")
print("Harga diperbaiki:", harga_nol.sum(), "| qty diperbaiki:", qty_salah.sum())

Harga diperbaiki: 4 | qty diperbaiki: 4


**Masalah 4.3-g: SKU kosong, huruf kecil, dan salah ketik (K1, K2)** → kode SKU dicocokkan dulu. Jika kosong atau tidak cocok, dicoba dari nama produk (akhiran listing seperti *Ready Stock*/*ORIGINAL* dibuang). Nama yang dipakai beberapa SKU dipilih dari harga online pada bulan pesanan (K2).

In [123]:
belum_ditolak = online["alasan_ditolak"].isna()
dipetakan = petakan_teks(online[belum_ditolak], "sku_barang", kolom_cadangan="nama_produk", buang_sufiks=True)
online = pd.concat([dipetakan, online[~belum_ditolak]]).sort_values(["berkas", "baris_excel"])

harga_master_online = dict(zip(products["sku"], products["harga_jual_online"]))
riwayat_online = riwayat_harga(online, "order_date", "harga", "jual_online")
acuan_online = buat_acuan(riwayat_online)
online = tentukan_ukuran(online, "harga", "order_date", acuan_online, harga_master_online)
catat_pemetaan(online, "online")
print("Ukuran dipilih dari harga:", online["sku_inferred"].sum())
online.groupby("aturan_sku").size().rename("jumlah_baris")

Ukuran dipilih dari harga: 26


aturan_sku
kode                                            4866
kode (dari nama produk)                            8
mapping tersimpan -> kode                         96
mapping tersimpan -> nama                          1
mapping tersimpan -> nama (dari nama produk)       1
nama (dari nama produk)                          147
nama (dari nama produk) + harga                   26
tidak dikenal (dari nama produk)                   1
Name: jumlah_baris, dtype: int64

**Masalah 4.3-f: pesanan lintas bulan (K9)** → kunci **No Order (teks) + SKU**. Jika pasangan di file bulan pesanan punya angka yang sama, salinan dibuang; jika angkanya berbeda, ditolak.

In [124]:
online["kunci_sku"] = online["sku"].fillna(online["teks_normal"])
online["bulan_pesanan"] = online["order_date"].dt.strftime("%Y-%m")
lolos = online["alasan_ditolak"].isna()
beda_bulan = lolos & (online["bulan_pesanan"] != online["bulan_laporan"])
asli = online[lolos & ~beda_bulan]

salinan = pd.Series(False, index=online.index)
beda_angka = pd.Series(False, index=online.index)
for i in online.index[beda_bulan]:
    r = online.loc[i]
    pasangan = asli[(asli["kanal"] == r["kanal"]) & (asli["no_order"] == r["no_order"])
                    & (asli["kunci_sku"] == r["kunci_sku"]) & (asli["bulan_laporan"] == r["bulan_pesanan"])]
    if len(pasangan) == 0:
        continue
    sama = ((pasangan["qty"] == r["qty"]) & np.isclose(pasangan["bruto"], r["bruto"]) & np.isclose(pasangan["netto"], r["netto"])).any()
    salinan[i] = sama
    beda_angka[i] = not sama

catat("online", online[salinan], "pesanan terulang di file bulan berikutnya (No Order + SKU sama)",
      "dibuang (disimpan di file bulan pesanan)", kolom="no_order", lama="no_order")
tolak(online, beda_angka, "pesanan lintas bulan: No Order + SKU sama, angka berbeda")
catat("online", online[beda_angka], "pesanan lintas bulan dengan angka berbeda", "ditolak (dilaporkan)", kolom="no_order", lama="no_order")
tanpa_pasangan = beda_bulan & ~salinan & ~beda_angka
catat("online", online[tanpa_pasangan], "tanggal pesanan di luar bulan laporan (tanpa salinan)",
      "dipertahankan, stok dikurangi di bulan laporan", kolom="order_date", lama="tanggal_asli")
online = online[~salinan].copy()
print("Salinan dibuang:", salinan.sum(), "| angka beda (ditolak):", beda_angka.sum(), "| tanpa pasangan:", tanpa_pasangan.sum())

Salinan dibuang: 82 | angka beda (ditolak): 0 | tanpa pasangan: 0


In [125]:
# SKU induk tidak boleh terjual, dan cek harga tidak wajar
tolak(online, online["sku"].isin(SKU_INDUK_TIDAK_DIJUAL), "SKU induk tidak dijual")
online = cek_harga_wajar(online, "harga", "order_date", acuan_online, harga_master_online, "online")
online = cek_harga_mapping(online, "harga", "order_date", acuan_online, harga_master_online, "online")    # K3
online["alasan_ditolak"].value_counts()

online: baris hasil mapping salah ketik 96 | harga sesuai 96 | DITOLAK 0 | tidak bisa dicek (ditandai) 0


alasan_ditolak
SKU tidak dikenal (tidak ada di master maupun mapping tersimpan)    1
Name: count, dtype: int64

**Masalah 4.3-e: komisi kosong (K10) dan hitung ulang kolom turunan** → komisi kosong dihitung balik dari rumus laba kotor UMKM dengan angka di file itu sendiri: `komisi = netto − biaya program ongkir − komisi affiliator − laba kotor`, ditandai `komisi_dihitung_balik`. Setelah itu bruto, diskon, netto, laba kotor, dan laba bersih dihitung ulang.

In [126]:
online_clean = online[online["alasan_ditolak"].isna()].copy()
online_ditolak_df = online[online["alasan_ditolak"].notna()]

# K10: komisi dihitung balik memakai angka asli di file (sebelum hitung ulang)
komisi_kosong = online_clean["komisi_platform"].isna()
online_clean["komisi_dihitung_balik"] = komisi_kosong
online_clean.loc[komisi_kosong, "komisi_platform"] = (online_clean["netto"] - online_clean["biaya_program_ongkir"]
                                                     - online_clean["komisi_affiliator"] - online_clean["laba_kotor"])[komisi_kosong].round()
catat("online", online_clean[komisi_kosong], "komisi platform kosong",
      "dihitung balik: netto - biaya program ongkir - komisi affiliator - laba kotor (komisi_dihitung_balik = True)",
      kolom="komisi_platform", baru="komisi_platform")
print("Komisi dihitung balik:", komisi_kosong.sum(), "| masih kosong:", online_clean["komisi_platform"].isna().sum())
print("Komisi hasil hitung balik / netto per kanal:")
print((online_clean["komisi_platform"] / online_clean["netto"])[komisi_kosong].groupby(online_clean["kanal"]).describe()[["count", "mean", "min", "max"]].round(3))

lama = online_clean[["bruto", "rp_diskon", "netto", "laba_kotor", "laba_bersih"]].copy()
online_clean["bruto"] = online_clean["harga"] * online_clean["qty"]
online_clean["rp_diskon"] = online_clean["bruto"] * online_clean["diskon_persen"]
online_clean["netto"] = online_clean["bruto"] - online_clean["rp_diskon"] - online_clean["rp_diskon_platform"]
online_clean["laba_kotor"] = (online_clean["netto"] - online_clean["biaya_program_ongkir"]
                              - online_clean["komisi_platform"] - online_clean["komisi_affiliator"])
online_clean["laba_bersih"] = online_clean["laba_kotor"] - online_clean["harga_kulakan"] - online_clean["biaya_packing"]
for kolom in lama.columns:
    berubah = (online_clean[kolom] - lama[kolom]).abs() > 1
    catat("online", online_clean[berubah].assign(_lama=lama.loc[berubah, kolom]), f"nilai {kolom} di file berbeda dengan hitung ulang",
          "diganti hasil hitung ulang", kolom=kolom, lama="_lama", baru=kolom)
print("Selisih rumus setelah hitung ulang:",
      ((online_clean["laba_kotor"] - online_clean["harga_kulakan"] - online_clean["biaya_packing"]) - online_clean["laba_bersih"]).abs().max())

Komisi dihitung balik: 52 | masih kosong: 0
Komisi hasil hitung balik / netto per kanal:
             count  mean   min   max
kanal                               
Shopee        26.0  0.08  0.08  0.08
TikTok Shop   16.0  0.10  0.10  0.10
Tokopedia     10.0  0.06  0.06  0.06
Selisih rumus setelah hitung ulang: 0.0


**Masalah 4.3-f lanjutan: baris yang baru sama setelah dibersihkan (K7)** → salinannya dibuang (baris pertama disimpan). Lalu tabel disimpan.

In [127]:
dup_bersih = online_clean.duplicated(subset=["berkas", "no", "no_order", "sku", "qty", "bruto", "netto", "order_date"], keep="first")
catat("online", online_clean[dup_bersih], "duplikat setelah dibersihkan (isi sama)", "dibuang (salinan pertama disimpan)")
online_clean = online_clean[~dup_bersih]
print("Duplikat setelah dibersihkan dibuang:", dup_bersih.sum())

kolom_online_bersih = ["kanal", "bulan_laporan", "order_date", "no_order", "sku", "nama_produk", "harga", "qty", "bruto",
                       "diskon_persen", "rp_diskon", "rp_diskon_platform", "netto", "ongkir", "promo_ongkir",
                       "biaya_program_ongkir", "sisa_ongkir_pembeli", "komisi_platform", "komisi_dihitung_balik",
                       "komisi_affiliator", "laba_kotor", "harga_kulakan", "biaya_packing", "laba_bersih",
                       "aturan_sku", "sku_inferred", "harga_mapping_tidak_dicek", "berkas", "baris_excel"]
online_clean = online_clean[kolom_online_bersih].sort_values(["order_date", "kanal", "no_order"])
online_clean["qty"] = online_clean["qty"].astype(int)
kolom_tanda_online = ["sku_inferred", "komisi_dihitung_balik", "harga_mapping_tidak_dicek"]
online_clean[kolom_tanda_online] = online_clean[kolom_tanda_online].fillna(False).astype(bool)
online_ditolak = laporan_ditolak(online_ditolak_df.assign(teks=online_ditolak_df["sku_barang"].fillna(online_ditolak_df["nama_produk"])), "online", "teks")
mapping_online = tabel_mapping(online[online["status_cocok"].notna()], "online")

online_clean.to_csv(INTERIM / "online_clean.csv", index=False)
online_ditolak.to_csv(INTERIM / "online_ditolak.csv", index=False)
print("Tipe nomor pesanan:", online_clean["no_order"].map(type).value_counts().to_dict())
ringkasan_tahap("online", n_masuk_online, online_clean, online_ditolak)

Duplikat setelah dibersihkan dibuang: 10
Tipe nomor pesanan: {<class 'str'>: 5053}
online: masuk 5220 = bersih 5053 + ditolak 1 + dibuang 166 -> SEIMBANG


,masalah,tindakan,jumlah_baris
0,pemetaan SKU otomatis (kode),dipetakan otomatis,4866
1,nama kanal tidak baku,dibakukan dengan kamus kanal,1915
2,pemetaan SKU otomatis (nama (dari nama produk)),dipetakan otomatis,147
3,diskon berupa teks ('12%'),diubah ke pecahan,147
4,pemetaan SKU otomatis (mapping tersimpan -> kode),dipetakan otomatis,96
5,harga berupa teks ('Rp 78.168'),diubah ke angka,95
6,pesanan terulang di file bulan berikutnya (No Order + SK...,dibuang (disimpan di file bulan pesanan),82
7,kolom biaya_balik_kulakan tidak dipakai (sama dengan HAR...,kolom tidak dipakai (dibuang dari tabel),59
8,"kolom laba_bersih_total tidak dipakai (sel gabungan, lab...",kolom tidak dipakai (dibuang dari tabel),59
9,kolom total tidak dipakai (sama dengan NETTO),kolom tidak dipakai (dibuang dari tabel),59


**Ringkasan tahap `online`:**
- **Masuk 5.220 baris → bersih 5.053 baris (96,8%)**, ditolak 1, dibuang 166 (82 salinan lintas bulan dengan kunci No Order + SKU, 22 kosong, 22 header terulang, 16 duplikat mentah, 14 TOTAL, 10 duplikat setelah dibersihkan).
- Diperbaiki: 95 harga teks, 147 diskon teks, 4 harga 0, 4 qty, 1.915 nama kanal dibakukan menjadi `TikTok Shop`, dan **52 komisi dihitung balik** (`komisi_dihitung_balik = True`). Hasil hitung baliknya tepat 8% (Shopee), 6% (Tokopedia), dan 10% (TikTok Shop) dari netto, sama dengan pola baris lain. Ada 26 baris yang ukurannya dipilih dari harga online pada bulan pesanan (`sku_inferred`).
- **Cek harga mapping salah ketik**: 96 baris online hasil mapping tersimpan, semuanya dalam ±10% harga jual online SKU hasil pada bulan pesanan. Tidak ada yang ditolak.
- Lima kolom dibuang dan dicatat: `no_hp`, `daerah`, `total`, `laba_bersih_total`, `biaya_balik_kulakan`.
- Tanggal pesanan asli disimpan di `order_date`. Setelah hitung ulang, netto, laba kotor, dan laba bersih tidak punya selisih dengan rumus. Nomor pesanan tetap teks dengan panjang utuh (14/26/18).
- 1 baris ditolak: SKU kosong dan nama produknya `EM4 Perikanan & Tambak Ready Sotck`, yang salah ketik di akhiran listing sehingga tidak cocok dengan satu nama pun.

### 5.7 Membersihkan `stok_masuk_keluar`
Setiap langkah menangani satu masalah dari bagian 4.4.

In [128]:
stok_mk = stok_mk_raw.copy()
stok_mk["alasan_ditolak"] = None
n_masuk_stok_mk = len(stok_mk)

is_kosong = stok_mk[kolom_stok].isna().all(axis=1)
catat("stok_masuk_keluar", stok_mk[is_kosong], "baris kosong", "dibuang")
stok_mk = stok_mk[~is_kosong]

# Masalah 4.4: duplikat persis (K7), dicek pada data mentah
dup = stok_mk.duplicated(subset=["berkas"] + kolom_stok, keep="first")
catat("stok_masuk_keluar", stok_mk[dup], "baris duplikat persis (semua kolom mentah identik)", "dibuang (salinan pertama disimpan)")
duplikat_stok_mk = stok_mk[dup].groupby("berkas").size().rename("duplikat_dibuang")
stok_mk = stok_mk[~dup].copy()
print("Baris masuk:", n_masuk_stok_mk, "| kosong:", is_kosong.sum(), "| duplikat dibuang:", dup.sum())

Baris masuk: 4234 | kosong: 0 | duplikat dibuang: 29


**Masalah 4.4: tanggal, harga, qty bercampur satuan, dan 11 variasi jenis** → diubah ke nilai baku. Jenis menjadi `RESTOCK_IN` (masuk dari supplier), `RETURN_IN` (retur pembeli), `DAMAGED_OUT` (rusak/kadaluarsa).

In [129]:
JENIS_STOK = {"restok": "RESTOCK_IN", "restock": "RESTOCK_IN", "barang masuk": "RESTOCK_IN",
              "retur": "RETURN_IN", "retur pembeli": "RETURN_IN",
              "rusak": "DAMAGED_OUT", "kadaluarsa": "DAMAGED_OUT", "rusak/kadaluarsa": "DAMAGED_OUT"}

stok_mk["tanggal"] = stok_mk["tanggal"].map(parse_tanggal)
stok_mk["harga_beli"] = stok_mk["harga_beli"].map(parse_angka)
stok_mk["qty_angka"] = stok_mk["qty_teks"].map(lambda v: parse_qty(v)[0])
stok_mk["satuan"] = stok_mk["qty_teks"].map(lambda v: parse_qty(v)[1])
stok_mk["jenis_asli"] = stok_mk["jenis"]
stok_mk["jenis"] = stok_mk["jenis"].map(lambda v: JENIS_STOK.get(normalisasi_teks(v)))
catat("stok_masuk_keluar", stok_mk[stok_mk["jenis"].notna()], "jenis pencatatan tidak seragam (RESTOK/restock/Barang masuk/...)",
      "diseragamkan", kolom="jenis", lama="jenis_asli", baru="jenis")

tak_jelas = stok_mk["jenis"].isna() | stok_mk["qty_angka"].isna() | stok_mk["tanggal"].isna()
tolak(stok_mk, tak_jelas, "jenis/qty/tanggal tidak terbaca")
catat("stok_masuk_keluar", stok_mk[tak_jelas], "jenis/qty/tanggal tidak terbaca", "ditolak (dilaporkan)")
print(stok_mk["jenis"].value_counts(dropna=False))

jenis
RESTOCK_IN     3562
RETURN_IN       600
DAMAGED_OUT      43
Name: count, dtype: int64


**Masalah 4.4: kode SKU berupa nama atau salah ketik (K1, K2, K3)** → dicocokkan ke master dan mapping tersimpan. Nama yang dipakai beberapa SKU dipilih dari harga beli pada bulan restok; jika harga beli kosong, ditolak.

In [130]:
belum_ditolak = stok_mk["alasan_ditolak"].isna()
dipetakan = petakan_teks(stok_mk[belum_ditolak], "sku_teks")
stok_mk = pd.concat([dipetakan, stok_mk[~belum_ditolak]]).sort_values(["berkas", "baris_excel"])
harga_master_beli = dict(zip(products["sku"], products["harga_kulakan"]))
acuan_beli = buat_acuan(riwayat_harga(stok_mk, "tanggal", "harga_beli", "beli"))
stok_mk = tentukan_ukuran(stok_mk, "harga_beli", "tanggal", acuan_beli, harga_master_beli)
catat_pemetaan(stok_mk, "stok_masuk_keluar")
stok_mk.groupby("aturan_sku").size().rename("jumlah_baris")

aturan_sku
kode                         3970
mapping tersimpan -> kode      81
nama                          146
nama + harga                    8
Name: jumlah_baris, dtype: int64

**Masalah 4.4: restok ditulis dengan nama kemasan hasil pemecahan (K4)** → mis. `Fertiphos Curah 50 kg`. Kemasan hasil pemecahan tidak pernah dibeli dari supplier; yang dibeli adalah karung induknya. Jadi restok dengan qty dalam kg/karung dialihkan ke SKU induk (`sku_dialihkan_ke_induk = True`).

In [131]:
info = products.set_index("sku")
induk_dari_sku = stok_mk["sku"].map(info["sku_induk"])
dialihkan = (stok_mk["alasan_ditolak"].isna() & (stok_mk["jenis"] == "RESTOCK_IN")
             & induk_dari_sku.isin(SKU_INDUK_TIDAK_DIJUAL) & stok_mk["satuan"].isin(["kg", "karung"]))
stok_mk["sku_sebelum"] = stok_mk["sku"]
stok_mk["sku_dialihkan_ke_induk"] = dialihkan
stok_mk.loc[dialihkan, "sku"] = induk_dari_sku[dialihkan]
catat("stok_masuk_keluar", stok_mk[dialihkan], "restok memakai nama kemasan hasil pemecahan dengan qty kg/karung",
      "dialihkan ke SKU induk (sku_dialihkan_ke_induk = True)", kolom="sku", lama="sku_sebelum", baru="sku")
stok_mk.loc[dialihkan, ["berkas", "baris_excel", "sku_teks", "qty_teks", "sku_sebelum", "sku"]]

,berkas,baris_excel,sku_teks,qty_teks,sku_sebelum,sku
94,stok_masuk_keluar.xlsx[2023-2024],96,Fertiphos Curah,50 kg,FRTPHS1,FRTPHS50KG
1302,stok_masuk_keluar.xlsx[2023-2024],1304,KCL Curah,150 kg,KCL,KCL50KG
2102,stok_masuk_keluar.xlsx[2025],659,Fertiphos Curah,1 karung,FRTPHS1,FRTPHS50KG


**Masalah 4.4: satuan karung/kg/pcs bercampur** → stok SKU induk 50 kg dicatat dalam **kg**: `karung` × 50, dan angka tanpa satuan dianggap karung (nilainya 1–4, tidak masuk akal sebagai kg). Harga beli induk tercatat per karung, jadi dibagi 50 menjadi harga per kg. Satuan yang tidak sesuai SKU ditolak.

In [132]:
induk = stok_mk["sku"].isin(SKU_INDUK_TIDAK_DIJUAL)
stok_mk["qty"] = stok_mk["qty_angka"]
karung = induk & stok_mk["satuan"].isin(["karung", ""])
stok_mk.loc[karung, "qty"] = stok_mk.loc[karung, "qty_angka"] * KG_PER_KARUNG
catat("stok_masuk_keluar", stok_mk[karung & (stok_mk["satuan"] == "")], "qty induk tanpa satuan",
      "dianggap karung (x50 kg)", kolom="qty", lama="qty_teks", baru="qty")
catat("stok_masuk_keluar", stok_mk[karung & (stok_mk["satuan"] == "karung")], "qty induk dalam karung",
      "dikonversi ke kg (x50)", kolom="qty", lama="qty_teks", baru="qty")

satuan_aneh = (stok_mk["alasan_ditolak"].isna()
               & ((~induk & stok_mk["satuan"].isin(["karung", "kg"])) | (induk & ~stok_mk["satuan"].isin(["karung", "kg", ""]))))
tolak(stok_mk, satuan_aneh, "satuan qty tidak sesuai SKU")
catat("stok_masuk_keluar", stok_mk[satuan_aneh], "satuan qty tidak sesuai SKU", "ditolak (dilaporkan)", lama="qty_teks")
stok_mk["harga_beli_per_satuan_stok"] = np.where(induk, stok_mk["harga_beli"] / KG_PER_KARUNG, stok_mk["harga_beli"])
print("Dikonversi ke kg:", karung.sum(), "| satuan tidak sesuai:", satuan_aneh.sum())
stok_mk[satuan_aneh][["berkas", "baris_excel", "sku_teks", "qty_teks"]]

Dikonversi ke kg: 96 | satuan tidak sesuai: 0


,berkas,baris_excel,sku_teks,qty_teks


**Cek harga restok hasil mapping salah ketik (K3).** Harga beli per satuan stok (SKU induk 50 kg: per kg) dibandingkan dengan harga beli acuan SKU itu pada bulan restok, toleransi ±10%. Restok tanpa harga beli dan retur tetap diterima, tetapi ditandai `harga_mapping_tidak_dicek`.

In [133]:
harga_master_beli_satuan = dict(zip(products["sku"], products["harga_kulakan"]))     # induk 50 kg: kosong
restok_saja = stok_mk[stok_mk["jenis"] == "RESTOCK_IN"]
acuan_beli_satuan = buat_acuan(riwayat_harga(restok_saja, "tanggal", "harga_beli_per_satuan_stok", "beli"))
stok_mk = cek_harga_mapping(stok_mk, "harga_beli_per_satuan_stok", "tanggal", acuan_beli_satuan, harga_master_beli_satuan,
                            "stok_masuk_keluar", dicek=stok_mk["jenis"] == "RESTOCK_IN")
ditolak_harga = stok_mk["alasan_ditolak"] == ALASAN_HARGA_MAPPING
tidak_dicek = stok_mk["harga_mapping_tidak_dicek"]
print("Tidak bisa dicek per jenis:", stok_mk.loc[tidak_dicek, "jenis"].value_counts().to_dict())
stok_mk.loc[ditolak_harga, ["berkas", "baris_excel", "sku_teks", "sku", "jenis", "harga_beli_per_satuan_stok", "rasio_harga_mapping"]].head(10)

stok_masuk_keluar: baris hasil mapping salah ketik 81 | harga sesuai 40 | DITOLAK 0 | tidak bisa dicek (ditandai) 41
Tidak bisa dicek per jenis: {'RESTOCK_IN': 31, 'RETURN_IN': 9, 'DAMAGED_OUT': 1}


,berkas,baris_excel,sku_teks,sku,jenis,harga_beli_per_satuan_stok,rasio_harga_mapping


**Masalah 4.4: retur tanpa nota asal, restok tanpa harga/supplier** → nilainya tidak diubah, hanya dicatat. Retur menjadi stok masuk dan tidak mengoreksi omzet.

In [134]:
lolos = stok_mk["alasan_ditolak"].isna()
catat("stok_masuk_keluar", stok_mk[lolos & (stok_mk["jenis"] == "RETURN_IN")], "retur tanpa nota asal",
      "dicatat sebagai stok masuk (RETURN_IN), tidak mengoreksi omzet")
restok = lolos & (stok_mk["jenis"] == "RESTOCK_IN")
catat("stok_masuk_keluar", stok_mk[restok & stok_mk["harga_beli"].isna()], "restok tanpa harga beli",
      "dibiarkan kosong (tidak dipakai di riwayat harga)")
catat("stok_masuk_keluar", stok_mk[restok & stok_mk["supplier"].isna()], "restok tanpa supplier", "dibiarkan kosong")

dup_bersih = lolos & stok_mk.duplicated(subset=["sheet", "tanggal", "sku", "jenis", "qty", "harga_beli", "supplier", "keterangan"], keep="first")
catat("stok_masuk_keluar", stok_mk[dup_bersih], "duplikat setelah dibersihkan (isi sama)", "dibuang (salinan pertama disimpan)")
stok_mk = stok_mk[~dup_bersih]
lolos = stok_mk["alasan_ditolak"].isna()
print("Duplikat setelah dibersihkan dibuang:", dup_bersih.sum())

stok_mk_clean = stok_mk[lolos][["tanggal", "sku", "jenis", "qty", "harga_beli", "harga_beli_per_satuan_stok", "supplier",
                                "keterangan", "aturan_sku", "sku_inferred", "sku_dialihkan_ke_induk", "harga_mapping_tidak_dicek",
                                "berkas", "baris_excel"]]
stok_mk_clean = stok_mk_clean.sort_values(["tanggal", "baris_excel"])
kolom_tanda_stok = ["sku_inferred", "sku_dialihkan_ke_induk", "harga_mapping_tidak_dicek"]
stok_mk_clean[kolom_tanda_stok] = stok_mk_clean[kolom_tanda_stok].fillna(False).astype(bool)
stok_mk_ditolak = laporan_ditolak(stok_mk[~lolos], "stok_masuk_keluar", "sku_teks")
mapping_stok_mk = tabel_mapping(stok_mk[stok_mk["status_cocok"].notna()], "stok_masuk_keluar")
stok_mk_clean.to_csv(INTERIM / "stok_masuk_keluar_clean.csv", index=False)
stok_mk_ditolak.to_csv(INTERIM / "stok_masuk_keluar_ditolak.csv", index=False)
print(stok_mk_ditolak["alasan"].str.split(":").str[0].value_counts())
ringkasan_tahap("stok_masuk_keluar", n_masuk_stok_mk, stok_mk_clean, stok_mk_ditolak)

Duplikat setelah dibersihkan dibuang: 1
alasan
nama dipakai beberapa SKU    14
Name: count, dtype: int64
stok_masuk_keluar: masuk 4234 = bersih 4190 + ditolak 14 + dibuang 30 -> SEIMBANG


,masalah,tindakan,jumlah_baris
0,jenis pencatatan tidak seragam (RESTOK/restock/Barang ma...,diseragamkan,4205
1,pemetaan SKU otomatis (kode),dipetakan otomatis,3970
2,restok tanpa harga beli,dibiarkan kosong (tidak dipakai di riwayat harga),1559
3,restok tanpa supplier,dibiarkan kosong,1066
4,retur tanpa nota asal,"dicatat sebagai stok masuk (RETURN_IN), tidak mengoreksi...",598
5,pemetaan SKU otomatis (nama),dipetakan otomatis,132
6,pemetaan SKU otomatis (mapping tersimpan -> kode),dipetakan otomatis,81
7,qty induk dalam karung,dikonversi ke kg (x50),66
8,harga tidak bisa dicek pada baris hasil mapping salah ketik,"dibiarkan, ditandai harga_mapping_tidak_dicek",41
9,qty induk tanpa satuan,dianggap karung (x50 kg),30


**Ringkasan tahap `stok_masuk_keluar`:**
- **Masuk 4.234 baris → bersih 4.190 baris (99,0%)**, ditolak 14, dibuang 30 (29 duplikat mentah, 1 duplikat setelah dibersihkan).
- Hasil: 3.551 restok (`RESTOCK_IN`), 597 retur (`RETURN_IN`), 42 rusak/kadaluarsa (`DAMAGED_OUT`). Qty induk 50 kg sudah dalam kg, dan harga beli induk dibagi 50 menjadi harga per kg.
- 81 kode salah ketik dipetakan lewat mapping tersimpan, 8 baris ukurannya dipilih dari harga beli pada bulan restok, dan **3 restok dialihkan ke SKU induk** (`Fertiphos Curah` → `FRTPHS50KG`, `KCL Curah` → `KCL50KG`, `sku_dialihkan_ke_induk = True`).
- **Cek harga restok hasil mapping salah ketik**: dari 81 baris hasil mapping tersimpan, 40 restok harganya dalam ±10% harga beli acuan (induk 50 kg per kg), **0 ditolak**. 41 baris tidak bisa dicek dan ditandai `harga_mapping_tidak_dicek`: 31 restok tanpa harga beli, 9 retur, dan 1 barang rusak.
- **14 baris ditolak**: nama yang dipakai beberapa ukuran (`Furadan 3GR`, `Antracol 70 WP`, `Emkape Maxxi Agri`, ...) tanpa harga beli, sehingga ukurannya tidak bisa dipilih. Sebagian besar adalah retur, yang memang tidak pernah punya harga.
- Retur hanya menambah stok dan **tidak mengurangi omzet**, karena retur tidak punya nota asal maupun harga (K6).

### 5.8 Membersihkan `pemecahan`
Setiap langkah menangani satu masalah dari bagian 4.5. Rumusnya: `qty_induk_keluar = jumlah_kemasan × faktor_ke_induk` (tanpa susut).

In [135]:
pemecahan = pemecahan_raw.copy()
pemecahan["alasan_ditolak"] = None
n_masuk_pemecahan = len(pemecahan)
is_kosong = pemecahan[kolom_pecah].isna().all(axis=1)
catat("pemecahan", pemecahan[is_kosong], "baris kosong", "dibuang")
pemecahan = pemecahan[~is_kosong]
# K7: baris identik TIDAK dibuang di pemecahan (memecah barang yang sama 2x sehari wajar);
# baris ini ikut ditandai kemungkinan_duplikat di langkah akhir
dup = pemecahan.duplicated(subset=["berkas"] + kolom_pecah, keep="first")
pemecahan = pemecahan.copy()

# Masalah 4.5: bahan '50.0 kg' dan kemasan '2 bks' berisi satuan
pemecahan["tanggal"] = pemecahan["tanggal"].map(parse_tanggal)
pemecahan["bahan_kg"] = pemecahan["bahan_teks"].map(lambda v: parse_qty(v)[0])
pemecahan["jumlah_kemasan"] = pemecahan["kemasan_teks"].map(lambda v: parse_qty(v)[0])
catat("pemecahan", pemecahan[pemecahan["kemasan_teks"].map(lambda v: isinstance(v, str))],
      "jumlah kemasan berisi satuan ('2 bks')", "diambil angkanya", kolom="jumlah_kemasan", lama="kemasan_teks", baru="jumlah_kemasan")
print("Baris masuk:", n_masuk_pemecahan, "| baris identik di file mentah (dipertahankan, ditandai nanti):", dup.sum())

Baris masuk: 708 | baris identik di file mentah (dipertahankan, ditandai nanti): 27


**Masalah 4.5: SKU hasil salah ketik (K1, K4)** → dipetakan strict. Kode yang tidak dikenal diperbaiki **hanya** jika kandidatnya sempit: SKU hasil pemecahan milik barang induk yang tertulis di baris itu (mis. `UEA` di baris *Pupuk Urea Petro* → `UREA`). Perbaikan ditandai `sku_diperbaiki = True`. Fungsi `perbaiki_kandidat_sempit` juga dipakai di rakit paket dan opname.

In [136]:
def perbaiki_kandidat_sempit(df, baris, kandidat_per_baris, tahap, masalah):
    # K4: untuk baris yang SKU-nya tidak dikenal, pilih kode dari daftar kandidat sempit.
    # Diterima jika hanya beda tukar dua huruf bersebelahan dari tepat satu kandidat, atau skor kemiripan
    # kandidat terbaik >= 0,5 dan unggul >= 0,1 dari kandidat kedua (atau hanya 1 kandidat).
    df["sku_diperbaiki"] = df.get("sku_diperbaiki", False)
    for i in df.index[baris]:
        kandidat = kandidat_per_baris[i]
        if not kandidat:
            continue
        teks = normalisasi_teks(df.at[i, "teks_sku_asli"]).replace(" ", "")
        tukar = [k for k in kandidat if tukar_bersebelahan(teks, k.lower())]
        skor = sorted(((skor_kemiripan(teks, k.lower()), k) for k in kandidat), reverse=True)
        kedua = skor[1][0] if len(skor) > 1 else 0
        if len(tukar) == 1:
            skor = [(1.0, tukar[0])]
        if len(tukar) == 1 or (skor[0][0] >= 0.5 and skor[0][0] - kedua >= 0.1):
            df.at[i, "sku"] = skor[0][1]
            df.at[i, "status_cocok"] = "cocok"
            df.at[i, "aturan_sku"] = f"kandidat sempit (skor {skor[0][0]:.2f})"
            df.at[i, "alasan_ditolak"] = None
            df.at[i, "sku_diperbaiki"] = True
    diperbaiki = baris & df["sku_diperbaiki"].astype(bool)
    catat(tahap, df[diperbaiki], masalah, "diperbaiki dari kandidat sempit (sku_diperbaiki = True)",
          kolom="sku", lama="teks_sku_asli", baru="sku")
    return df

In [137]:
pemecahan = petakan_teks(pemecahan, "sku_hasil_teks")
pemecahan = tentukan_ukuran(pemecahan, None, None, {}, {})        # tidak ada kolom harga

# SKU induk dari nama barang induk, lalu kandidat sempit = SKU hasil milik induk itu
induk_pecah = products[products["is_induk_pemecahan"]]
sku_induk_dari_nama = dict(zip(induk_pecah["nama_induk"].map(normalisasi_teks), induk_pecah["sku"]))
pemecahan["sku_induk"] = pemecahan["barang_induk"].map(lambda v: sku_induk_dari_nama.get(normalisasi_teks(v)))
info = products.set_index("sku")
anak_per_induk = products[products["sku_induk"].notna()].groupby("sku_induk")["sku"].apply(list).to_dict()
tidak_dikenal = pemecahan["status_cocok"] == "tidak_dikenal"
kandidat = {i: anak_per_induk.get(pemecahan.at[i, "sku_induk"], []) for i in pemecahan.index}
pemecahan = perbaiki_kandidat_sempit(pemecahan, tidak_dikenal, kandidat, "pemecahan", "kode SKU hasil salah ketik")
catat_pemetaan(pemecahan, "pemecahan")
pemecahan = pemecahan.rename(columns={"sku": "sku_hasil"})
display(pemecahan.loc[pemecahan["sku_diperbaiki"], ["berkas", "baris_excel", "barang_induk", "sku_hasil_teks", "sku_hasil"]].head(6))

salah_induk = pemecahan["alasan_ditolak"].isna() & (pemecahan["sku_hasil"].map(info["sku_induk"]) != pemecahan["sku_induk"])
tolak(pemecahan, salah_induk, "induk tidak cocok dengan SKU hasil")
catat("pemecahan", pemecahan[salah_induk], "induk tidak cocok dengan SKU hasil", "ditolak (dilaporkan)")
print("Diperbaiki dari kandidat sempit:", pemecahan["sku_diperbaiki"].sum(), "| induk tidak cocok:", salah_induk.sum(),
      "| masih ditolak:", pemecahan["alasan_ditolak"].notna().sum())

,berkas,baris_excel,barang_induk,sku_hasil_teks,sku_hasil
48,stok_pemecahan.xlsx[2023-2024],50,Emkape Maxxi Agri,EKAPE500,EMKAPE500
53,stok_pemecahan.xlsx[2023-2024],55,Pupuk Urea Petro,UEA,UREA
155,stok_pemecahan.xlsx[2023-2024],157,Emkape Maxxi Agri,EMKAP500,EMKAPE500
223,stok_pemecahan.xlsx[2023-2024],225,Emkape Maxxi Agri,EMKAP5E00,EMKAPE500
354,stok_pemecahan.xlsx[2025],92,Emkape Maxxi Agri,EMAKPE500,EMKAPE500
385,stok_pemecahan.xlsx[2025],123,Emkape Maxxi Agri,EMKAP5E00,EMKAPE500


Diperbaiki dari kandidat sempit: 11 | induk tidak cocok: 0 | masih ditolak: 0


**Masalah 4.5: bahan dipakai kosong** → dihitung dari jumlah kemasan × faktor (tanpa susut).

In [138]:
pemecahan["qty_induk_keluar"] = pemecahan["jumlah_kemasan"] * pemecahan["sku_hasil"].map(info["faktor_ke_induk"])
lolos = pemecahan["alasan_ditolak"].isna()
bahan_kosong = lolos & pemecahan["bahan_kg"].isna()
catat("pemecahan", pemecahan[bahan_kosong], "bahan dipakai (kg) kosong", "dihitung dari jumlah kemasan x faktor (tanpa susut)",
      kolom="bahan_kg", baru="qty_induk_keluar")
beda = lolos & pemecahan["bahan_kg"].notna() & ((pemecahan["bahan_kg"] - pemecahan["qty_induk_keluar"]).abs() > 1e-6)
catat("pemecahan", pemecahan[beda], "bahan dipakai tidak sama dengan kemasan x faktor", "mengikuti jumlah kemasan (tanpa susut)",
      kolom="bahan_kg", lama="bahan_kg", baru="qty_induk_keluar")

pemecahan["kemungkinan_duplikat"] = False
pemecahan.loc[lolos, "kemungkinan_duplikat"] = pemecahan[lolos].duplicated(
    subset=["sheet", "tanggal", "sku_induk", "sku_hasil", "jumlah_kemasan", "catatan"], keep=False)
catat("pemecahan", pemecahan[pemecahan["kemungkinan_duplikat"]], "kemungkinan duplikat (baru sama setelah dibersihkan)",
      "dipertahankan, diberi tanda kemungkinan_duplikat (memecah 2x di hari yang sama mungkin terjadi)")

pemecahan_clean = pemecahan[lolos][["tanggal", "sku_induk", "qty_induk_keluar", "sku_hasil", "jumlah_kemasan", "aturan_sku",
                                   "sku_diperbaiki", "kemungkinan_duplikat", "berkas", "baris_excel"]].sort_values(["tanggal", "baris_excel"])
pemecahan_ditolak = laporan_ditolak(pemecahan[~lolos], "pemecahan", "sku_hasil_teks")
mapping_pemecahan = tabel_mapping(pemecahan.rename(columns={"sku_hasil": "sku"}), "pemecahan")
pemecahan_clean.to_csv(INTERIM / "pemecahan_clean.csv", index=False)
pemecahan_ditolak.to_csv(INTERIM / "pemecahan_ditolak.csv", index=False)
print("Bahan dihitung:", bahan_kosong.sum(), "| bahan bertentangan:", beda.sum())
display(pemecahan_clean.groupby(["sku_induk", "sku_hasil"]).agg(kali_pecah=("tanggal", "size"),
        induk_keluar=("qty_induk_keluar", "sum"), kemasan_masuk=("jumlah_kemasan", "sum")))
ringkasan_tahap("pemecahan", n_masuk_pemecahan, pemecahan_clean, pemecahan_ditolak)

Bahan dihitung: 59 | bahan bertentangan: 0


,,kali_pecah,induk_keluar,kemasan_masuk
sku_induk,sku_hasil,,,
DLMT50KG,DLMT,99,4950.0,9900.0
EMKAPE1,EMKAPE500,424,424.0,848.0
FRTPHS50KG,FRTPHS1,28,1400.0,1400.0
KANITRAT1,KANITRAT500,10,10.0,20.0
KCL50KG,KCL,103,5150.0,10300.0
UREA50KG,UREA,44,2200.0,4400.0


pemecahan: masuk 708 = bersih 708 + ditolak 0 + dibuang 0 -> SEIMBANG


,masalah,tindakan,jumlah_baris
0,pemetaan SKU otomatis (kode),dipetakan otomatis,694
1,jumlah kemasan berisi satuan ('2 bks'),diambil angkanya,177
2,kemungkinan duplikat (baru sama setelah dibersihkan),"dipertahankan, diberi tanda kemungkinan_duplikat (memeca...",156
3,bahan dipakai (kg) kosong,dihitung dari jumlah kemasan x faktor (tanpa susut),59
4,kode SKU hasil salah ketik,diperbaiki dari kandidat sempit (sku_diperbaiki = True),11
5,pemetaan SKU otomatis (kandidat sempit (skor 1.00)),dipetakan otomatis,7
6,pemetaan SKU otomatis (kandidat sempit (skor 0.94)),dipetakan otomatis,3
7,pemetaan SKU otomatis (mapping tersimpan -> kode),dipetakan otomatis,3
8,pemetaan SKU otomatis (kandidat sempit (skor 0.86)),dipetakan otomatis,1


**Ringkasan tahap `pemecahan`:**
- **Masuk 708 baris → bersih 708 baris (100%)**, tanpa baris ditolak atau dibuang.
- 14 kode SKU hasil salah ketik semuanya terpetakan: 10 lewat kandidat sempit dari barang induknya (mis. `EKAPE500` di baris Emkape → `EMKAPE500`, `UEA` di baris Urea Petro → `UREA`, `sku_diperbaiki = True`) dan sisanya lewat mapping tersimpan.
- 58 bahan kosong dihitung dari kemasan × faktor. Tidak ada baris yang bahannya bertentangan dengan aturan tanpa susut.
- **27 baris yang identik di file mentah tidak dibuang** (K7). Bersama baris yang baru sama setelah dibersihkan, total 156 baris diberi tanda `kemungkinan_duplikat`. Memecah barang yang sama lebih dari sekali di hari yang sama (mis. 2 × Emkape 1 kg) wajar terjadi di toko, dan membuang pemecahan asli membuat stok kemasan terlalu rendah.

### 5.9 Membersihkan `rakit_paket`
Masalah dari 4.6: jumlah berisi satuan (`4 pcs`) dan 2 SKU paket salah ketik. Kode paket yang salah ketik diperbaiki dengan kandidat sempit: hanya 5 SKU paket (K4).

In [139]:
rakit = rakit_raw.copy()
rakit["alasan_ditolak"] = None
n_masuk_rakit = len(rakit)
kolom_rakit = ["tanggal", "sku_paket_teks", "paket_teks", "catatan"]
is_kosong = rakit[kolom_rakit].isna().all(axis=1)
catat("rakit_paket", rakit[is_kosong], "baris kosong", "dibuang")
rakit = rakit[~is_kosong]
dup = rakit.duplicated(subset=["berkas"] + kolom_rakit, keep="first")
catat("rakit_paket", rakit[dup], "baris duplikat persis (semua kolom mentah identik)", "dibuang (salinan pertama disimpan)")
rakit = rakit[~dup].copy()

rakit["tanggal"] = rakit["tanggal"].map(parse_tanggal)
rakit["jumlah_paket"] = rakit["paket_teks"].map(lambda v: parse_qty(v)[0])
catat("rakit_paket", rakit[rakit["paket_teks"].map(lambda v: isinstance(v, str))], "jumlah paket berisi satuan ('4 pcs')",
      "diambil angkanya", kolom="jumlah_paket", lama="paket_teks", baru="jumlah_paket")

rakit = petakan_teks(rakit, "sku_paket_teks")
rakit = tentukan_ukuran(rakit, None, None, {}, {})
SKU_PAKET = products.loc[products["is_bundle"], "sku"].tolist()
tidak_dikenal = rakit["status_cocok"] == "tidak_dikenal"
rakit = perbaiki_kandidat_sempit(rakit, tidak_dikenal, {i: SKU_PAKET for i in rakit.index}, "rakit_paket", "kode SKU paket salah ketik")
catat_pemetaan(rakit, "rakit_paket")
display(rakit.loc[rakit["sku_diperbaiki"], ["berkas", "baris_excel", "sku_paket_teks", "sku", "aturan_sku"]])
bukan_paket = rakit["alasan_ditolak"].isna() & ~rakit["sku"].map(info["is_bundle"]).fillna(False).astype(bool)
tolak(rakit, bukan_paket, "SKU bukan paket")

lolos = rakit["alasan_ditolak"].isna()
rakit_clean = rakit[lolos].rename(columns={"sku": "sku_paket"})[["tanggal", "sku_paket", "jumlah_paket", "aturan_sku", "sku_diperbaiki", "berkas", "baris_excel"]]
rakit_ditolak = laporan_ditolak(rakit[~lolos], "rakit_paket", "sku_paket_teks")
mapping_rakit = tabel_mapping(rakit, "rakit_paket")
rakit_clean.to_csv(INTERIM / "rakit_paket_clean.csv", index=False)
rakit_ditolak.to_csv(INTERIM / "rakit_paket_ditolak.csv", index=False)
print("Ditolak:", len(rakit_ditolak))
ringkasan_tahap("rakit_paket", n_masuk_rakit, rakit_clean, rakit_ditolak)

,berkas,baris_excel,sku_paket_teks,sku,aturan_sku
18,stok_rakit_paket.xlsx[2025],3,PAKETUTMBUH250,PAKETTUMBUH250,kandidat sempit (skor 1.00)
26,stok_rakit_paket.xlsx[2025],11,PAKEGDSL,PAKETGDSL,kandidat sempit (skor 0.94)


Ditolak: 0
rakit_paket: masuk 41 = bersih 41 + ditolak 0 + dibuang 0 -> SEIMBANG


,masalah,tindakan,jumlah_baris
0,pemetaan SKU otomatis (kode),dipetakan otomatis,39
1,jumlah paket berisi satuan ('4 pcs'),diambil angkanya,13
2,kode SKU paket salah ketik,diperbaiki dari kandidat sempit (sku_diperbaiki = True),2
3,pemetaan SKU otomatis (kandidat sempit (skor 0.94)),dipetakan otomatis,1
4,pemetaan SKU otomatis (kandidat sempit (skor 1.00)),dipetakan otomatis,1


**Ringkasan tahap `rakit_paket`:**
- **Masuk 41 → bersih 41**, tanpa baris ditolak. `PAKETUTMBUH250` dan `PAKEGDSL` diperbaiki menjadi `PAKETTUMBUH250` dan `PAKETGDSL` dari kandidat sempit 5 SKU paket (`sku_diperbaiki = True`).

### 5.10 Membersihkan `opname`
Masalah dari 4.7: format tanggal campur di isi sel dan kode SKU salah ketik. Typo dicocokkan lewat mapping tersimpan (K3); kode berawalan `PAKE` diperbaiki dengan kandidat sempit 5 SKU paket (K4). Tanggal opname diambil dari **nama berkas** (satu berkas = satu kali opname). Stok induk 50 kg tetap dalam kg.

In [140]:
opname = opname_raw.copy()
opname["alasan_ditolak"] = None
n_masuk_opname = len(opname)
kolom_opname = ["tanggal", "sku_teks", "stok_fisik", "catatan"]
is_kosong = opname[["tanggal", "sku_teks", "stok_fisik"]].isna().all(axis=1)
catat("opname", opname[is_kosong], "baris kosong", "dibuang")
opname = opname[~is_kosong]
dup = opname.duplicated(subset=["berkas"] + kolom_opname, keep="first")
catat("opname", opname[dup], "baris duplikat persis (semua kolom mentah identik)", "dibuang (salinan pertama disimpan)")
opname = opname[~dup].copy()

opname["tanggal_isi"] = opname["tanggal"].map(parse_tanggal)
opname["tanggal"] = pd.to_datetime(opname["tanggal_berkas"])
beda_tgl = opname["tanggal_isi"] != opname["tanggal"]
catat("opname", opname[beda_tgl], "tanggal di isi berbeda dengan nama berkas", "memakai tanggal nama berkas",
      kolom="tanggal", lama="tanggal_isi", baru="tanggal")
opname["stok_fisik"] = opname["stok_fisik"].map(parse_angka)
tolak(opname, opname["stok_fisik"].isna() | (opname["stok_fisik"] < 0), "stok fisik kosong/negatif")

belum_ditolak = opname["alasan_ditolak"].isna()
dipetakan = tentukan_ukuran(petakan_teks(opname[belum_ditolak], "sku_teks"), None, None, {}, {})
teks_paket = (dipetakan["status_cocok"] == "tidak_dikenal") & dipetakan["teks_normal"].str.startswith("pake")
dipetakan = perbaiki_kandidat_sempit(dipetakan, teks_paket, {i: SKU_PAKET for i in dipetakan.index}, "opname", "kode SKU paket salah ketik")
opname = pd.concat([dipetakan, opname[~belum_ditolak]])
catat_pemetaan(opname, "opname")
sku_ganda = opname["alasan_ditolak"].isna() & opname.duplicated(subset=["tanggal", "sku"], keep=False) & opname["sku"].notna()
tolak(opname, sku_ganda, "SKU tercatat >1 kali di opname yang sama")
catat("opname", opname[sku_ganda], "SKU tercatat >1 kali di opname yang sama", "ditolak (dilaporkan)")

lolos = opname["alasan_ditolak"].isna()
opname_clean = opname[lolos].merge(products[["sku", "satuan_stok"]], on="sku", how="left")
opname_clean = opname_clean[["tanggal", "sku", "stok_fisik", "satuan_stok", "catatan", "aturan_sku", "berkas", "baris_excel"]].sort_values(["tanggal", "sku"])
opname_ditolak = laporan_ditolak(opname[~lolos], "opname", "sku_teks")
mapping_opname = tabel_mapping(opname[opname["status_cocok"].notna()], "opname")
opname_clean.to_csv(INTERIM / "opname_clean.csv", index=False)
opname_ditolak.to_csv(INTERIM / "opname_ditolak.csv", index=False)
display(opname_ditolak)
ringkasan_tahap("opname", n_masuk_opname, opname_clean, opname_ditolak)

,tahap,berkas,baris_excel,teks,alasan,rincian_harga
0,opname,stok_opname_2025-05-31.xlsx,89,MAXXIGRD500,SKU tidak dikenal (tidak ada di master maupun mapping te...,None


opname: masuk 1063 = bersih 1062 + ditolak 1 + dibuang 0 -> SEIMBANG


,masalah,tindakan,jumlah_baris
0,pemetaan SKU otomatis (kode),dipetakan otomatis,1050
1,pemetaan SKU otomatis (mapping tersimpan -> kode),dipetakan otomatis,12
2,SKU tidak dikenali atau ambigu,ditolak (dilaporkan),1


**Ringkasan tahap `opname`:**
- **Masuk 1.063 → bersih 1.062.** 12 kode salah ketik dipetakan lewat mapping tersimpan (mis. `CPRCD00`, `PLYB2G5`) dan `PAKEBUAH250` lewat kandidat paket.
- 1 baris ditolak: `MAXXIGRD500` (opname 31 Mei 2025), yang sama miripnya dengan dua SKU Maxxi Grow, sehingga tidak ditebak.

### 5.11 Membersihkan `metrik_marketplace` → tabel `marketplace_metrics`
Masalah dari 4.8:
- kolom **SKU / Nama Produk** berisi campuran kode SKU dan nama produk → dipetakan ke kode SKU dengan aturan yang sama (K1–K3);
- nama yang dipakai beberapa SKU (mis. *Emkape Maxxi Agri*) → ukuran dipilih dari **harga tayang** dibanding harga jual online pada bulan itu (K2), sehingga baris yang tampak ganda per periode-kanal-produk terpisah ke dua SKU;
- nama kanal dibakukan dengan kamus kanal (K13);
- rating kosong dibiarkan kosong (K16).

In [141]:
metrik = metrik_raw.copy()
metrik["alasan_ditolak"] = None
n_masuk_metrik = len(metrik)

is_kosong = metrik[kolom_metrik].isna().all(axis=1)
catat("metrik_marketplace", metrik[is_kosong], "baris kosong", "dibuang")
metrik = metrik[~is_kosong]
dup = metrik.duplicated(subset=["berkas"] + kolom_metrik, keep="first")                    # K7
catat("metrik_marketplace", metrik[dup], "baris duplikat persis (semua kolom mentah identik)", "dibuang (salinan pertama disimpan)")
metrik = metrik[~dup].copy()
print("Baris masuk:", n_masuk_metrik, "| kosong:", is_kosong.sum(), "| duplikat persis dibuang:", dup.sum())

Baris masuk: 4130 | kosong: 0 | duplikat persis dibuang: 0


**Kanal, periode, dan angka** → kanal lewat kamus kanal; periode menjadi tanggal; metrik menjadi angka.

In [142]:
metrik["kanal_asli"] = metrik["kanal"]
metrik["kanal"] = metrik["kanal"].map(lambda v: KAMUS_KANAL.get(normalisasi_teks(v)))
berubah = metrik["kanal"] != metrik["kanal_asli"]
catat("metrik_marketplace", metrik[berubah & metrik["kanal"].notna()], "nama kanal tidak baku", "dibakukan dengan kamus kanal",
      kolom="kanal", lama="kanal_asli", baru="kanal")
tolak(metrik, metrik["kanal"].isna(), "kanal tidak dikenal")

metrik["periode_mulai"] = metrik["periode_mulai"].map(parse_tanggal)
metrik["periode_akhir"] = metrik["periode_akhir"].map(parse_tanggal)
for kolom in ["impresi", "klik", "pesanan", "rating", "jumlah_ulasan", "harga_tayang"]:
    metrik[kolom] = metrik[kolom].map(parse_angka)
tolak(metrik, metrik["periode_mulai"].isna(), "periode tidak terbaca")
print(metrik["kanal"].value_counts(dropna=False))
print("Periode tidak terbaca:", metrik["periode_mulai"].isna().sum())

kanal
Shopee         1470
Tokopedia      1470
TikTok Shop    1190
Name: count, dtype: int64
Periode tidak terbaca: 0


**Masalah 4.8: kolom produk campuran kode dan nama (K1–K3)** → dipetakan ke SKU. Nama yang dipakai beberapa SKU dipilih dari harga tayang terhadap harga jual online pada bulan yang sama (`acuan_online` dari rekap online).

In [143]:
belum_ditolak = metrik["alasan_ditolak"].isna()
dipetakan = petakan_teks(metrik[belum_ditolak], "produk")
metrik = pd.concat([dipetakan, metrik[~belum_ditolak]]).sort_values(["berkas", "baris_excel"])
metrik = tentukan_ukuran(metrik, "harga_tayang", "periode_mulai", acuan_online, harga_master_online)
catat_pemetaan(metrik, "metrik_marketplace")
print("Ukuran dipilih dari harga tayang:", metrik["sku_inferred"].sum())
metrik.groupby(["status_cocok", "aturan_sku"]).size().rename("jumlah_baris")

Ukuran dipilih dari harga tayang: 280


status_cocok  aturan_sku  
cocok         kode            2489
              nama            1361
ganda         nama + harga     280
Name: jumlah_baris, dtype: int64

**Masalah 4.8: duplikat per periode-kanal-SKU** → dicek lagi **setelah** pemetaan. Jika masih ada SKU yang muncul dua kali di periode dan kanal yang sama, semua salinannya ditolak karena tidak bisa ditentukan mana yang benar.

In [144]:
lolos = metrik["alasan_ditolak"].isna()
dup_sku = lolos & metrik.duplicated(subset=["periode_mulai", "kanal", "sku"], keep=False)
tolak(metrik, dup_sku, "SKU muncul >1 kali di periode dan kanal yang sama")
catat("metrik_marketplace", metrik[dup_sku], "SKU muncul >1 kali di periode dan kanal yang sama", "ditolak (dilaporkan)",
      kolom="sku", lama="produk", baru="sku")
print("Duplikat periode-kanal-SKU setelah pemetaan:", dup_sku.sum())

Duplikat periode-kanal-SKU setelah pemetaan: 0


**Rating kosong (K16)** → tidak diisi. Produk yang belum punya rating berbeda dengan produk yang ratingnya rata-rata; jika diisi rata-rata, analisis Q12 akan menganggap produk baru sudah punya rating.

In [145]:
lolos = metrik["alasan_ditolak"].isna()
rating_kosong = lolos & metrik["rating"].isna()
catat("metrik_marketplace", metrik[rating_kosong], "rating kosong", "dibiarkan kosong (tidak diisi rata-rata)", kolom="rating")
print("Rating kosong:", rating_kosong.sum(), "| di antaranya jumlah ulasan 0:", (rating_kosong & (metrik["jumlah_ulasan"] == 0)).sum())

Rating kosong: 1269 | di antaranya jumlah ulasan 0: 537


In [146]:
marketplace_metrics = metrik[lolos].copy()
marketplace_metrics["periode"] = marketplace_metrics["periode_mulai"].dt.strftime("%Y-%m")
marketplace_metrics = marketplace_metrics[["periode", "periode_mulai", "periode_akhir", "kanal", "sku", "impresi", "klik", "pesanan",
                                           "rating", "jumlah_ulasan", "harga_tayang", "aturan_sku", "sku_inferred",
                                           "berkas", "baris_excel"]].sort_values(["periode", "kanal", "sku"]).reset_index(drop=True)
marketplace_metrics["sku_inferred"] = marketplace_metrics["sku_inferred"].astype(bool)
metrik_ditolak = laporan_ditolak(metrik[~lolos], "metrik_marketplace", "produk")
mapping_metrik = tabel_mapping(metrik[metrik["status_cocok"].notna()], "metrik_marketplace")
marketplace_metrics.to_csv(INTERIM / "marketplace_metrics_clean.csv", index=False)
metrik_ditolak.to_csv(INTERIM / "metrik_marketplace_ditolak.csv", index=False)
print("SKU per kanal:", marketplace_metrics.groupby("kanal")["sku"].nunique().to_dict())
ringkasan_tahap("metrik_marketplace", n_masuk_metrik, marketplace_metrics, metrik_ditolak)

SKU per kanal: {'Shopee': 70, 'TikTok Shop': 70, 'Tokopedia': 70}
metrik_marketplace: masuk 4130 = bersih 4130 + ditolak 0 + dibuang 0 -> SEIMBANG


,masalah,tindakan,jumlah_baris
0,pemetaan SKU otomatis (kode),dipetakan otomatis,2489
1,pemetaan SKU otomatis (nama),dipetakan otomatis,1361
2,rating kosong,dibiarkan kosong (tidak diisi rata-rata),1269
3,pemetaan SKU dari harga (nama + harga),dipetakan dari harga (sku_inferred = True),280


**Pembanding (informasi, tidak mengubah data):** kolom `Pesanan` di metrik dibandingkan dengan jumlah nomor pesanan di rekap online per bulan-kanal-SKU. Keduanya berasal dari sumber yang berbeda (dashboard platform vs rekap UMKM), jadi tidak harus sama persis.

In [147]:
pesanan_rekap = (online_clean.assign(periode=online_clean["order_date"].dt.strftime("%Y-%m"))
                 .groupby(["periode", "kanal", "sku"])["no_order"].nunique().rename("pesanan_rekap"))
banding = marketplace_metrics.set_index(["periode", "kanal", "sku"])[["pesanan"]].join(pesanan_rekap, how="left").fillna({"pesanan_rekap": 0})
banding["selisih"] = banding["pesanan"] - banding["pesanan_rekap"]
print("Baris metrik:", len(banding), "| pesanan sama persis:", (banding["selisih"] == 0).sum())
print("Korelasi pesanan metrik vs rekap:", round(banding["pesanan"].corr(banding["pesanan_rekap"]), 3))
banding.groupby("kanal")[["pesanan", "pesanan_rekap"]].sum()

Baris metrik: 4130 | pesanan sama persis: 4101
Korelasi pesanan metrik vs rekap: 0.998


,pesanan,pesanan_rekap
kanal,,
Shopee,1815.0,1799.0
TikTok Shop,1396.0,1387.0
Tokopedia,695.0,690.0


**Ringkasan tahap `metrik_marketplace`:**
- **Masuk 4.130 → bersih 4.130**, tanpa baris ditolak atau dibuang. Setiap kanal punya 70 SKU, dan semua SKU ada di master.
- 280 baris nama dua ukuran dipisahkan ke SKU yang benar dari harga tayang terhadap harga jual online pada bulan yang sama (`sku_inferred = True`). Setelah pemetaan, **tidak ada duplikat periode-kanal-SKU**.
- Rating kosong (1.269, di antaranya 537 dengan jumlah ulasan 0) dibiarkan kosong.
- Pembanding: kolom `Pesanan` sama persis dengan jumlah nomor pesanan di rekap online pada 4.101 dari 4.130 baris (korelasi 0,998). Selisih kecil wajar karena sumbernya berbeda (dashboard platform vs rekap UMKM).

### 5.12 Tabel `product_mapping` dan laporan baris yang ditolak (K1)
`product_mapping` berisi satu baris per teks unik per sumber: berapa baris, berapa yang masuk, aturan yang dipakai, dan SKU hasilnya. `laporan_ditolak` menggabungkan semua baris yang ditolak dari semua tabel (berkas, baris Excel, teks, alasan) supaya admin tahu baris mana yang perlu diperbaiki di file sumber.

In [148]:
product_mapping = pd.concat([mapping_offline, mapping_online, mapping_stok_mk, mapping_pemecahan, mapping_rakit,
                             mapping_opname, mapping_metrik], ignore_index=True)
product_mapping.pivot_table(index="sumber", columns="status", values="jumlah_baris", aggfunc="sum", fill_value=0)

status,ditolak,masuk,sebagian ditolak
sumber,,,
metrik_marketplace,0,4130,0
offline,234,54226,851
online,1,5063,0
opname,1,1062,0
pemecahan,0,708,0
rakit_paket,0,41,0
stok_masuk_keluar,8,4186,10


In [149]:
laporan_ditolak_semua = pd.concat([offline_ditolak, online_ditolak, stok_mk_ditolak, pemecahan_ditolak, rakit_ditolak,
                                   opname_ditolak, metrik_ditolak], ignore_index=True)
laporan_ditolak_semua["jenis_alasan"] = laporan_ditolak_semua["alasan"].str.split(":").str[0].str.split(" \\(").str[0]
laporan_ditolak_semua.to_csv(INTERIM / "laporan_ditolak.csv", index=False)
print("Total baris ditolak:", len(laporan_ditolak_semua))
laporan_ditolak_semua.pivot_table(index="jenis_alasan", columns="tahap", values="baris_excel", aggfunc="size", fill_value=0)

Total baris ditolak: 441


tahap,offline,online,opname,stok_masuk_keluar
jenis_alasan,,,,
SKU tidak dikenal,86,1,1,0
harga tidak sesuai SKU hasil mapping salah ketik,1,0,0,0
nama dipakai beberapa SKU,331,0,0,14
qty kosong dan tidak bisa dihitung,7,0,0,0


In [150]:
# Teks yang ditolak karena tidak dikenal, diurutkan dari yang paling sering (keterbatasan data K3)
tidak_dikenal = laporan_ditolak_semua[laporan_ditolak_semua["jenis_alasan"] == "SKU tidak dikenal"]
(tidak_dikenal.assign(teks=tidak_dikenal["teks"].map(normalisasi_teks))
 .groupby("teks").agg(jumlah_baris=("tahap", "size"), tahap=("tahap", lambda s: ", ".join(sorted(set(s)))))
 .sort_values("jumlah_baris", ascending=False).head(10))

,jumlah_baris,tahap
teks,,
mpk pak tani,4,offline
metaze 80 wp,3,offline
gibgor 10,2,offline
metazeb 8 0wp,2,offline
mkpp ak,2,offline
mk pak,2,offline
mp pak tani,2,offline
bfuos 150,1,offline
abaecl 18,1,offline


### 5.13 Log perubahan gabungan (`change_log`)
Setiap tindakan dikelompokkan ke kategori supaya mudah dihitung: dibuang, diperbaiki, dipetakan otomatis, dipetakan dari harga, ditolak, ditandai, atau kolom tidak dipakai.

In [151]:
def kategori_tindakan(tindakan):
    """Kelompokkan teks tindakan berdasarkan kata pertamanya."""
    if tindakan.startswith("dibuang"):
        return "dibuang"
    if tindakan.startswith("ditolak"):
        return "ditolak (dilaporkan)"
    if tindakan.startswith("kolom tidak dipakai"):
        return "kolom tidak dipakai"
    if tindakan.startswith(("dibiarkan", "dicatat", "subtotal dipertahankan", "dipertahankan")):
        return "ditandai (nilai tidak diubah)"
    if tindakan.startswith("dipetakan otomatis"):
        return "dipetakan otomatis"
    if tindakan.startswith("dipetakan dari harga"):
        return "dipetakan dari harga (inferred)"
    return "diperbaiki"


change_log = pd.concat(daftar_log, ignore_index=True)
change_log["kategori"] = change_log["tindakan"].map(kategori_tindakan)
print("Jumlah catatan log:", len(change_log))
change_log.pivot_table(index="tahap", columns="kategori", values="masalah", aggfunc="size", fill_value=0)

Jumlah catatan log: 105272


kategori,dibuang,diperbaiki,dipetakan dari harga (inferred),dipetakan otomatis,ditandai (nilai tidak diubah),ditolak (dilaporkan),kolom tidak dipakai
tahap,,,,,,,
master,0,4,0,0,0,0,0
metrik_marketplace,0,0,280,3850,1269,0,0
offline,458,21310,6956,47937,926,425,0
online,166,2217,26,5119,0,1,295
opname,0,0,0,1062,0,1,0
pemecahan,0,247,0,708,156,0,0
rakit_paket,0,15,0,41,0,0,0
stok_masuk_keluar,30,4304,8,4183,3264,14,0


In [152]:
# Jumlah baris per jenis masalah (dibuang / diperbaiki / ditolak / ditandai), semua tahap
(change_log[~change_log["kategori"].str.startswith("dipetakan")]
 .groupby(["tahap", "kategori", "masalah"]).size().rename("jumlah_baris").reset_index())

,tahap,kategori,masalah,jumlah_baris
0,master,diperbaiki,SKU induk tidak ada di master_kategori,4
1,metrik_marketplace,ditandai (nilai tidak diubah),rating kosong,1269
2,offline,dibuang,baris duplikat persis (semua kolom mentah identik),404
3,offline,dibuang,baris subtotal/total,54
4,offline,diperbaiki,"harga berupa teks ('Rp 1.000'/'13,500')",1620
5,offline,diperbaiki,harga kosong atau 0,625
6,offline,diperbaiki,nomor nota kosong,2758
7,offline,diperbaiki,nomor nota tidak lengkap (4 digit),15835
8,offline,diperbaiki,qty kosong,260
9,offline,diperbaiki,qty negatif dengan subtotal positif,57


In [153]:
duplikat_per_berkas = (change_log[change_log["masalah"].str.startswith(("baris duplikat persis", "duplikat setelah dibersihkan"))]
                       .assign(jenis=lambda d: np.where(d["masalah"].str.startswith("baris duplikat persis"), "duplikat mentah", "duplikat setelah dibersihkan"))
                       .groupby(["tahap", "berkas", "jenis"]).size().rename("baris_dibuang").reset_index())
duplikat_per_berkas.pivot_table(index="tahap", columns="jenis", values="baris_dibuang", aggfunc="sum", fill_value=0)

jenis,duplikat mentah,duplikat setelah dibersihkan
tahap,,
offline,404,0
online,16,10
stok_masuk_keluar,29,1


## 6. Membangun Tabel Akhir

### 6.1 `channels`, `transactions`, `transaction_items`, `online_item_financials`
- Nama kanal mengikuti kamus kanal (K13), dan `channels` menjadi tabel acuan kanal (dipakai Q1).
- `revenue` offline = subtotal, online = netto.
- Tanggal transaksi online = **`order_date`** (tanggal pesanan asli, K8).
- `transaction_id`: `OFF-<nota>`, `SHP-`/`TKP-`/`TTS-<nomor pesanan>`.

In [154]:
channels = pd.DataFrame({
    "channel_id": [1, 2, 3, 4],
    "kanal": KANAL_BAKU,
    "kode": ["OFF", "SHP", "TKP", "TTS"],
    "jenis": ["toko fisik", "marketplace", "marketplace", "marketplace"],
    "dasar_pendapatan": ["subtotal nota", "netto", "netto", "netto"],
    "frekuensi_data": ["harian (nota)", "laporan bulanan", "laporan bulanan", "laporan bulanan"]})
KODE_KANAL = dict(zip(channels["kanal"], channels["kode"]))
channels

,channel_id,kanal,kode,jenis,dasar_pendapatan,frekuensi_data
0,1,Offline,OFF,toko fisik,subtotal nota,harian (nota)
1,2,Shopee,SHP,marketplace,netto,laporan bulanan
2,3,Tokopedia,TKP,marketplace,netto,laporan bulanan
3,4,TikTok Shop,TTS,marketplace,netto,laporan bulanan


In [155]:
# Item offline
off = offline_clean.copy()
off["kanal"] = "Offline"
off["transaction_id"] = "OFF-" + off["no_nota"]
off["item_id"] = "OFF-" + off["berkas"].str.extract(r"(\d{4}-\d{2})")[0] + "-" + off["baris_excel"].astype(str)
off = off.rename(columns={"diskon": "diskon_rp", "subtotal": "revenue"})

# Item online
on = online_clean.copy()
on["transaction_id"] = on["kanal"].map(KODE_KANAL) + "-" + on["no_order"]
on["item_id"] = on["kanal"].map(KODE_KANAL) + "-" + on["bulan_laporan"] + "-" + on["baris_excel"].astype(str)
on["diskon_rp"] = on["rp_diskon"] + on["rp_diskon_platform"]
on = on.rename(columns={"order_date": "tanggal", "harga": "harga_satuan", "netto": "revenue"})

kolom_item = ["item_id", "transaction_id", "kanal", "tanggal", "sku", "qty", "harga_satuan", "diskon_rp", "revenue", "berkas", "baris_excel"]
transaction_items = pd.concat([off[kolom_item], on[kolom_item]], ignore_index=True).sort_values(["tanggal", "transaction_id"]).reset_index(drop=True)
transactions = (transaction_items.groupby(["transaction_id", "kanal"])
                .agg(tanggal=("tanggal", "min"), jumlah_item=("item_id", "size"), total_qty=("qty", "sum"), total_revenue=("revenue", "sum"))
                .reset_index().sort_values("tanggal").reset_index(drop=True))
transactions["order_no_generated"] = transactions["transaction_id"].str.startswith("OFF-TANPA-NOTA")
online_item_financials = online_clean.assign(item_id=on["item_id"], transaction_id=on["transaction_id"]).rename(columns={"harga_kulakan": "hpp_total"})[
    ["item_id", "transaction_id", "kanal", "bulan_laporan", "order_date", "sku", "qty", "bruto", "rp_diskon", "rp_diskon_platform",
     "netto", "ongkir", "promo_ongkir", "biaya_program_ongkir", "sisa_ongkir_pembeli", "komisi_platform", "komisi_dihitung_balik",
     "komisi_affiliator", "laba_kotor", "hpp_total", "biaya_packing", "laba_bersih"]]

print("transactions:", transactions.shape, "| transaction_items:", transaction_items.shape, "| online_item_financials:", online_item_financials.shape)
transaction_items.groupby("kanal").agg(baris=("item_id", "size"), qty=("qty", "sum"), revenue=("revenue", "sum"),
                                       mulai=("tanggal", "min"), sampai=("tanggal", "max"))

transactions: (35436, 7) | transaction_items: (59946, 11) | online_item_financials: (5053, 22)


,baris,qty,revenue,mulai,sampai
kanal,,,,,
Offline,54893,109954,3.237620e+09,2023-11-01,2026-10-31
Shopee,2293,2889,1.169944e+08,2025-02-01,2026-10-31
TikTok Shop,1876,2396,9.358527e+07,2025-06-01,2026-10-31
Tokopedia,884,1166,4.656427e+07,2025-02-09,2026-10-31


**Cek bias ID `TANPA-NOTA` (K11).** Baris tanpa nota diberi ID per baris, jadi setiap baris dihitung sebagai satu transaksi. Sel di bawah mengukur seberapa besar pengaruhnya jika jumlah transaksi offline dihitung.

In [156]:
trx_off = transactions[transactions["kanal"] == "Offline"]
tanpa = trx_off["order_no_generated"]
print("Transaksi offline:", len(trx_off), "| di antaranya TANPA-NOTA:", tanpa.sum(), f"({tanpa.mean():.1%})")
print("Rata-rata item per nota asli:", round(trx_off.loc[~tanpa, "jumlah_item"].mean(), 2), "| per ID TANPA-NOTA: 1")
print("Porsi revenue offline dari TANPA-NOTA:", f"{trx_off.loc[tanpa, 'total_revenue'].sum() / trx_off['total_revenue'].sum():.1%}")
print("Jika baris tanpa nota digabung per tanggal, jumlah transaksi offline menjadi:",
      (~tanpa).sum() + trx_off.loc[tanpa, "tanggal"].nunique())

Transaksi offline: 31310 | di antaranya TANPA-NOTA: 2366 (7.6%)
Rata-rata item per nota asli: 1.81 | per ID TANPA-NOTA: 1
Porsi revenue offline dari TANPA-NOTA: 4.4%
Jika baris tanpa nota digabung per tanggal, jumlah transaksi offline menjadi: 29847


### 6.2 `stock_movements` dan `stock_opname`
Semua pergerakan stok dalam satu tabel (`qty` positif = masuk, negatif = keluar):

| jenis | sumber | catatan |
|---|---|---|
| OPENING | opname pertama | saldo awal, sebelum transaksi hari itu |
| RESTOCK_IN, RETURN_IN, DAMAGED_OUT | stok_masuk_keluar | |
| SPLIT_OUT / SPLIT_IN | pemecahan | induk keluar, kemasan masuk |
| ASSEMBLE_OUT / ASSEMBLE_IN | rakit paket × komposisi | komponen keluar, paket masuk |
| SALE_OFFLINE | rekap offline | tanggal nota |
| SALE_ONLINE | rekap online | **hari terakhir bulan laporan** (K8); tanggal pesanan asli di `order_date` |
| ADJUSTMENT | setiap opname setelah opname pertama | stok fisik − stok sistem (K12) |

Opname dianggap dilakukan di **akhir hari**, jadi `ADJUSTMENT` diurutkan paling akhir pada tanggal itu. Stok sistem saat opname sudah memuat penyesuaian opname sebelumnya, sehingga **selisih di setiap opname adalah selisih sejak opname sebelumnya** (dipakai Q6).

> **Catatan K8:** karena penjualan online baru mengurangi stok saat laporan bulanan diimpor, stok sistem di **tengah bulan bisa lebih tinggi dari stok fisik**. Selisih opname tengah bulan sebagian berasal dari penjualan online yang belum diimpor.

In [157]:
tgl_awal = opname_clean["tanggal"].min()
awal = opname_clean[opname_clean["tanggal"] == tgl_awal]
tanda = stok_mk_clean["jenis"].map({"RESTOCK_IN": 1, "RETURN_IN": 1, "DAMAGED_OUT": -1})
komponen = rakit_clean.merge(bundle_components, on="sku_paket")
tgl_impor = pd.to_datetime(online_clean["bulan_laporan"] + "-01") + pd.offsets.MonthEnd(0)    # hari terakhir bulan laporan

bagian = [
    pd.DataFrame({"tanggal": awal["tanggal"], "sku": awal["sku"], "jenis": "OPENING", "qty": awal["stok_fisik"],
                  "ref": f"opname {tgl_awal:%Y-%m-%d}", "sumber": awal["berkas"]}),
    pd.DataFrame({"tanggal": stok_mk_clean["tanggal"], "sku": stok_mk_clean["sku"], "jenis": stok_mk_clean["jenis"],
                  "qty": stok_mk_clean["qty"] * tanda, "ref": stok_mk_clean["supplier"], "sumber": stok_mk_clean["berkas"]}),
    pd.DataFrame({"tanggal": pemecahan_clean["tanggal"], "sku": pemecahan_clean["sku_induk"], "jenis": "SPLIT_OUT",
                  "qty": -pemecahan_clean["qty_induk_keluar"], "ref": "-> " + pemecahan_clean["sku_hasil"], "sumber": pemecahan_clean["berkas"]}),
    pd.DataFrame({"tanggal": pemecahan_clean["tanggal"], "sku": pemecahan_clean["sku_hasil"], "jenis": "SPLIT_IN",
                  "qty": pemecahan_clean["jumlah_kemasan"], "ref": "<- " + pemecahan_clean["sku_induk"], "sumber": pemecahan_clean["berkas"]}),
    pd.DataFrame({"tanggal": rakit_clean["tanggal"], "sku": rakit_clean["sku_paket"], "jenis": "ASSEMBLE_IN",
                  "qty": rakit_clean["jumlah_paket"], "ref": "rakit paket", "sumber": rakit_clean["berkas"]}),
    pd.DataFrame({"tanggal": komponen["tanggal"], "sku": komponen["sku_komponen"], "jenis": "ASSEMBLE_OUT",
                  "qty": -komponen["jumlah_paket"] * komponen["qty_komponen_per_paket"], "ref": "-> " + komponen["sku_paket"],
                  "sumber": komponen["berkas"]}),
    pd.DataFrame({"tanggal": offline_clean["tanggal"], "sku": offline_clean["sku"], "jenis": "SALE_OFFLINE",
                  "qty": -offline_clean["qty"], "ref": "OFF-" + offline_clean["no_nota"], "sumber": offline_clean["berkas"]}),
    pd.DataFrame({"tanggal": tgl_impor, "sku": online_clean["sku"], "jenis": "SALE_ONLINE", "qty": -online_clean["qty"],
                  "ref": online_clean["kanal"].map(KODE_KANAL) + "-" + online_clean["no_order"], "sumber": online_clean["berkas"],
                  "order_date": online_clean["order_date"]}),
]
gerak = pd.concat(bagian, ignore_index=True)
print("Pergerakan sebelum ADJUSTMENT:", len(gerak))

Pergerakan sebelum ADJUSTMENT: 65902


In [158]:
# K12: ADJUSTMENT setelah tiap opname, berurutan dari opname kedua sampai terakhir
penyesuaian, opname_banding = [], []
for tgl in sorted(opname_clean["tanggal"].unique())[1:]:
    sudah = pd.concat([gerak] + penyesuaian)
    stok_sistem = sudah[sudah["tanggal"] <= tgl].groupby("sku")["qty"].sum()        # termasuk penyesuaian sebelumnya
    fisik = opname_clean[opname_clean["tanggal"] == tgl].copy()
    fisik["stok_sistem"] = fisik["sku"].map(stok_sistem).fillna(0)
    fisik["selisih"] = fisik["stok_fisik"] - fisik["stok_sistem"]
    opname_banding.append(fisik)
    ada_selisih = fisik[fisik["selisih"].abs() > 1e-9]
    penyesuaian.append(pd.DataFrame({"tanggal": ada_selisih["tanggal"], "sku": ada_selisih["sku"], "jenis": "ADJUSTMENT",
                                     "qty": ada_selisih["selisih"], "ref": f"opname {pd.Timestamp(tgl):%Y-%m-%d}",
                                     "sumber": ada_selisih["berkas"]}))

URUTAN_JENIS = {"OPENING": 0, "RESTOCK_IN": 1, "RETURN_IN": 2, "SPLIT_OUT": 3, "SPLIT_IN": 4, "ASSEMBLE_OUT": 5,
                "ASSEMBLE_IN": 6, "SALE_OFFLINE": 7, "SALE_ONLINE": 8, "DAMAGED_OUT": 9, "ADJUSTMENT": 10}
stock_movements = pd.concat([gerak] + penyesuaian, ignore_index=True)
stock_movements["urutan"] = stock_movements["jenis"].map(URUTAN_JENIS)
stock_movements = stock_movements.sort_values(["tanggal", "urutan", "sku"]).drop(columns="urutan").reset_index(drop=True)
stock_movements.insert(0, "movement_id", np.arange(1, len(stock_movements) + 1))

stock_opname = pd.concat([awal.assign(stok_sistem=np.nan, selisih=np.nan)] + opname_banding, ignore_index=True)
stock_opname = stock_opname[["tanggal", "sku", "stok_fisik", "stok_sistem", "selisih", "satuan_stok", "catatan",
                             "aturan_sku", "berkas", "baris_excel"]].sort_values(["tanggal", "sku"]).reset_index(drop=True)
print(stock_movements.shape)
stock_movements.groupby("jenis")["qty"].agg(["size", "sum"]).round(1)

(66542, 8)


,size,sum
jenis,,
ADJUSTMENT,640,-1801.4
ASSEMBLE_IN,41,166.0
ASSEMBLE_OUT,120,-150.0
DAMAGED_OUT,42,-79.0
OPENING,189,3773.0
RESTOCK_IN,3551,103503.0
RETURN_IN,597,1132.0
SALE_OFFLINE,54893,-109954.0
SALE_ONLINE,5053,-6451.0


In [159]:
# Cek: setelah ADJUSTMENT, stok sistem di setiap tanggal opname = stok fisik
cek = stock_opname[stock_opname["tanggal"] > tgl_awal].copy()      # opname pertama = saldo awal (sebelum transaksi hari itu)
for tgl in cek["tanggal"].unique():
    saldo = stock_movements[stock_movements["tanggal"] <= tgl].groupby("sku")["qty"].sum()
    di_tgl = cek["tanggal"] == tgl
    cek.loc[di_tgl, "saldo"] = cek.loc[di_tgl, "sku"].map(saldo).fillna(0)
print("Opname yang saldo akhirnya != stok fisik:", ((cek["saldo"] - cek["stok_fisik"]).abs() > 1e-6).sum())

tgl_akhir = opname_clean["tanggal"].max()
stok_akhir = stock_movements[stock_movements["tanggal"] <= tgl_akhir].groupby("sku")["qty"].sum()
stok_tanpa_penyesuaian = gerak[gerak["tanggal"] <= tgl_akhir].groupby("sku")["qty"].sum()
print("SKU dengan stok sistem negatif per", tgl_akhir.date(), ":", (stok_akhir < -1e-9).sum(),
      "| jika tanpa ADJUSTMENT:", (stok_tanpa_penyesuaian < -1e-9).sum())

Opname yang saldo akhirnya != stok fisik: 0
SKU dengan stok sistem negatif per 2026-10-31 : 8 | jika tanpa ADJUSTMENT: 21


In [160]:
# Ringkasan selisih per opname (bahan Q6): selisih sejak opname sebelumnya
(stock_opname.dropna(subset=["selisih"]).groupby("tanggal")
 .agg(sku=("sku", "size"), sku_selisih=("selisih", lambda s: (s.abs() > 1e-9).sum()),
      selisih_kurang=("selisih", lambda s: s[s < 0].sum()), selisih_lebih=("selisih", lambda s: s[s > 0].sum())).round(1))

,sku,sku_selisih,selisih_kurang,selisih_lebih
tanggal,,,,
2024-02-29,94,70,-366.0,174.0
2024-05-31,83,63,-577.5,180.0
2024-08-31,54,30,-233.0,569.0
2024-11-30,56,36,-256.0,55.0
2025-02-28,74,49,-380.0,518.5
2025-05-31,106,78,-469.0,625.0
2025-08-31,43,35,-390.0,125.5
2025-11-30,74,57,-466.0,179.0
2026-02-28,39,26,-322.3,113.0


In [161]:
# Contoh: pemecahan urea (kg -> kemasan 0,5 kg) dan rakit paket
display(stock_movements[stock_movements["jenis"].isin(["SPLIT_OUT", "SPLIT_IN"]) & stock_movements["ref"].str.contains("UREA")].head(4))
stock_movements[stock_movements["jenis"].isin(["ASSEMBLE_IN", "ASSEMBLE_OUT"])].head(4)

,movement_id,tanggal,sku,jenis,qty,ref,sumber,order_date
410,411,2023-11-06,UREA50KG,SPLIT_OUT,-50.0,-> UREA,stok_pemecahan.xlsx[2023-2024],NaT
412,413,2023-11-06,UREA,SPLIT_IN,100.0,<- UREA50KG,stok_pemecahan.xlsx[2023-2024],NaT
1503,1504,2023-11-24,UREA50KG,SPLIT_OUT,-50.0,-> UREA,stok_pemecahan.xlsx[2023-2024],NaT
1504,1505,2023-11-24,UREA,SPLIT_IN,100.0,<- UREA50KG,stok_pemecahan.xlsx[2023-2024],NaT


,movement_id,tanggal,sku,jenis,qty,ref,sumber,order_date
1373,1374,2023-11-22,MGSULFATE,ASSEMBLE_OUT,-1.0,-> PAKETTUMBUH250,stok_rakit_paket.xlsx[2023-2024],NaT
1374,1375,2023-11-22,NPKPT,ASSEMBLE_OUT,-1.0,-> PAKETTUMBUH250,stok_rakit_paket.xlsx[2023-2024],NaT
1375,1376,2023-11-22,ULTRDP,ASSEMBLE_OUT,-1.0,-> PAKETTUMBUH250,stok_rakit_paket.xlsx[2023-2024],NaT
1376,1377,2023-11-22,PAKETTUMBUH250,ASSEMBLE_IN,4.0,rakit paket,stok_rakit_paket.xlsx[2023-2024],NaT


### 6.3 `data_status`: `data_complete_until` per kanal

In [162]:
status = [{"kanal": "Offline", "periode_mulai": offline_clean["tanggal"].min(),
           "data_complete_until": offline_clean["tanggal"].max(), "catatan": "rekap harian dari nota toko"}]
for kanal, grup in online_clean.groupby("kanal"):
    status.append({"kanal": kanal,
                   "periode_mulai": pd.Timestamp(grup["bulan_laporan"].min() + "-01"),
                   "data_complete_until": pd.Timestamp(grup["bulan_laporan"].max() + "-01") + pd.offsets.MonthEnd(0),
                   "catatan": ("laporan bulanan; stok berkurang saat laporan diimpor (akhir bulan), jadi stok sistem tengah "
                               "bulan bisa lebih tinggi dari fisik. Analisis penjualan dan forecast memakai order_date")})
status.append({"kanal": "Stok (restok/retur/pemecahan/rakit)", "periode_mulai": stok_mk_clean["tanggal"].min(),
               "data_complete_until": stok_mk_clean["tanggal"].max(), "catatan": "pencatatan gudang"})
status.append({"kanal": "Metrik marketplace", "periode_mulai": marketplace_metrics["periode_mulai"].min(),
               "data_complete_until": marketplace_metrics["periode_akhir"].max(), "catatan": "metrik bulanan dari dashboard platform"})
data_status = pd.DataFrame(status)
data_status

,kanal,periode_mulai,data_complete_until,catatan
0,Offline,2023-11-01,2026-10-31,rekap harian dari nota toko
1,Shopee,2025-02-01,2026-10-31,laporan bulanan; stok berkurang saat laporan diimpor (ak...
2,TikTok Shop,2025-06-01,2026-10-31,laporan bulanan; stok berkurang saat laporan diimpor (ak...
3,Tokopedia,2025-02-01,2026-10-31,laporan bulanan; stok berkurang saat laporan diimpor (ak...
4,Stok (restok/retur/pemecahan/rakit),2023-11-04,2026-10-31,pencatatan gudang
5,Metrik marketplace,2025-02-01,2026-10-31,metrik bulanan dari dashboard platform


### 6.4 `product_price_history` dan supplier utama (K14)
Harga historis tidak ada di data mentah, jadi riwayat harga dibentuk dari transaksi yang SKU-nya **pasti** (bukan hasil pemilihan dari harga): median harga per SKU per bulan untuk `jual_offline`, `jual_online`, dan `beli` (per satuan stok, dari restok). Tabel ini juga dipakai untuk HPP di Q3.

In [163]:
riwayat_offline_final = riwayat_harga(offline_clean.assign(status_cocok="cocok", alasan_ditolak=None)[offline_clean["sku_inferred"] == False],
                                      "tanggal", "harga_satuan", "jual_offline")
riwayat_online_final = riwayat_harga(online_clean.assign(status_cocok="cocok", alasan_ditolak=None)[online_clean["sku_inferred"] == False],
                                     "order_date", "harga", "jual_online")
restok_berharga = stok_mk_clean[stok_mk_clean["jenis"] == "RESTOCK_IN"].assign(status_cocok="cocok", alasan_ditolak=None)
riwayat_beli_final = riwayat_harga(restok_berharga, "tanggal", "harga_beli_per_satuan_stok", "beli")
product_price_history = pd.concat([riwayat_offline_final, riwayat_online_final, riwayat_beli_final], ignore_index=True)
print(product_price_history.groupby("jenis_harga").agg(sku=("sku", "nunique"), baris=("sku", "size")))
contoh = product_price_history[(product_price_history["jenis_harga"] == "jual_offline") & (product_price_history["sku"] == "EMKAPE1")]
contoh.groupby(contoh["bulan"].dt.year)["harga"].median()      # harga jual Emkape 1 kg naik per tahun

              sku  baris
jenis_harga             
beli          159   1825
jual_offline  179   4803
jual_online   164   1721


bulan
2023    52500.0
2024    55500.0
2025    58000.0
2026    61000.0
Name: harga, dtype: float64

**Supplier utama per kelompok produk induk (K14).** Kemasan hasil pemecahan tidak pernah direstok, jadi supplier dihitung per `kelompok_induk`: supplier yang paling sering memasok kelompok itu dipakai untuk semua SKU di kelompoknya.

In [164]:
restok_bersupplier = stok_mk_clean[(stok_mk_clean["jenis"] == "RESTOCK_IN") & stok_mk_clean["supplier"].notna()]
restok_bersupplier = restok_bersupplier.assign(kelompok_induk=restok_bersupplier["sku"].map(info["kelompok_induk"]))
supplier_sku = restok_bersupplier.groupby("sku")["supplier"].agg(lambda s: s.value_counts().index[0])
supplier_kelompok = restok_bersupplier.groupby("kelompok_induk")["supplier"].agg(lambda s: s.value_counts().index[0])

products_final = products.copy()
products_final["supplier_utama"] = products_final["kelompok_induk"].map(supplier_kelompok)
print("SKU tanpa supplier: per SKU =", (~products["sku"].isin(supplier_sku.index)).sum(),
      "| per kelompok induk =", products_final["supplier_utama"].isna().sum())
print("Masih tanpa supplier:", products_final.loc[products_final["supplier_utama"].isna(), "sku"].tolist())
products_final["supplier_utama"].value_counts()

SKU tanpa supplier: per SKU = 27 | per kelompok induk = 23
Masih tanpa supplier: ['LINDOMIN400', 'MANOHARA1', 'GEMPUR1', 'ISADORA400', 'SERENDY', 'KALINET500', 'PAKETAG', 'PAKETGDSL', 'PAKETTUMBUH250', 'PAKETBUAH250', 'PAKETGNRTF', 'EM4B', 'GDSLB500', 'ANTILA500', 'CPH', 'EXTRAPCKG', 'JARINGSDG', 'KNOMRH2', 'MEGAZEB', 'PLYBG25', 'PLYBG30', 'RADOCKCL', 'SPRFS100']


supplier_utama
SUP-04    48
SUP-06    32
SUP-03    25
SUP-02    22
SUP-05    17
SUP-07    15
SUP-01     8
Name: count, dtype: int64

### 6.5 Validasi akhir
Setiap aturan ditulis sebagai `assert`. Jika ada yang gagal, notebook berhenti di sel ini sehingga masalahnya langsung terlihat.

In [165]:
sku_master = set(products_final["sku"])
tabel_ber_sku = {"transaction_items": transaction_items, "online_item_financials": online_item_financials,
                 "stock_movements": stock_movements, "stock_opname": stock_opname, "marketplace_metrics": marketplace_metrics,
                 "product_price_history": product_price_history}
hasil_validasi = []

def periksa(nama, lolos):
    hasil_validasi.append({"aturan": nama, "lolos": bool(lolos)})
    assert lolos, nama

# 1. Semua SKU ada di master
for nama, df in tabel_ber_sku.items():
    periksa(f"SKU {nama} ada di master", df["sku"].isin(sku_master).all())
periksa("SKU paket di bundle_components ada di master", bundle_components[["sku_paket", "sku_komponen"]].isin(sku_master).all().all())

# 2. Tidak ada nilai kosong di kolom inti
kolom_inti = {"transaction_items": ["item_id", "transaction_id", "kanal", "tanggal", "sku", "qty", "revenue"],
              "stock_movements": ["movement_id", "tanggal", "sku", "jenis", "qty"],
              "stock_opname": ["tanggal", "sku", "stok_fisik"],
              "online_item_financials": ["netto", "komisi_platform", "laba_kotor", "laba_bersih"],
              "marketplace_metrics": ["periode", "kanal", "sku", "impresi", "klik", "pesanan"]}
for nama, kolom in kolom_inti.items():
    periksa(f"{nama}: kolom inti tidak kosong", tabel_ber_sku[nama][kolom].notna().all().all())

# 3. Rumus netto dan laba online
f = online_item_financials
periksa("netto = bruto - diskon - diskon platform", ((f["bruto"] - f["rp_diskon"] - f["rp_diskon_platform"] - f["netto"]).abs() <= 1).all())
periksa("laba kotor = netto - program ongkir - komisi - affiliator",
        ((f["netto"] - f["biaya_program_ongkir"] - f["komisi_platform"] - f["komisi_affiliator"] - f["laba_kotor"]).abs() <= 1).all())
periksa("laba bersih = laba kotor - kulakan - packing", ((f["laba_kotor"] - f["hpp_total"] - f["biaya_packing"] - f["laba_bersih"]).abs() <= 1).all())

# 4. Panjang nomor pesanan per kanal
PANJANG_NO_ORDER = {"Shopee": 14, "Tokopedia": 26, "TikTok Shop": 18}
for kanal, panjang in PANJANG_NO_ORDER.items():
    periksa(f"nomor pesanan {kanal} = {panjang} karakter", (online_clean.loc[online_clean["kanal"] == kanal, "no_order"].str.len() == panjang).all())

# 5. Nama kanal baku di semua tabel
for nama, seri in {"transaction_items": transaction_items["kanal"], "online_item_financials": online_item_financials["kanal"],
                   "marketplace_metrics": marketplace_metrics["kanal"], "data_status (kanal penjualan)": data_status["kanal"].iloc[:4]}.items():
    periksa(f"kanal baku di {nama}", seri.isin(KANAL_BAKU).all())

# 6. Lain-lain
periksa("item_id unik", transaction_items["item_id"].is_unique)
periksa("SKU induk 50 kg tidak terjual", ~transaction_items["sku"].isin(SKU_INDUK_TIDAK_DIJUAL).any())
periksa("qty transaksi positif", (transaction_items["qty"] > 0).all())
periksa("periode-kanal-SKU di metrik unik", ~marketplace_metrics.duplicated(subset=["periode", "kanal", "sku"]).any())
periksa("stok sistem = fisik di setiap opname setelah ADJUSTMENT", ((cek["saldo"] - cek["stok_fisik"]).abs() <= 1e-6).all())
pd.DataFrame(hasil_validasi)

,aturan,lolos
0,SKU transaction_items ada di master,True
1,SKU online_item_financials ada di master,True
2,SKU stock_movements ada di master,True
3,SKU stock_opname ada di master,True
4,SKU marketplace_metrics ada di master,True
5,SKU product_price_history ada di master,True
6,SKU paket di bundle_components ada di master,True
7,transaction_items: kolom inti tidak kosong,True
8,stock_movements: kolom inti tidak kosong,True
9,stock_opname: kolom inti tidak kosong,True


## 7. Menyimpan Data
- `data/interim/`: hasil tiap tabel (`*_clean.csv`), laporan baris yang ditolak (`*_ditolak.csv` dan `laporan_ditolak.csv`), dan `product_mapping_manual.csv`.
- `data/processed/`: tabel akhir dalam CSV, termasuk `penjualan_ambigu` (baris yang ditolak karena nama ambigu, untuk penyesuaian target di modeling).

In [166]:
# penjualan_ambigu: offline (dari bagian 5.5) + online jika ada baris dengan alasan yang sama
on_ambigu = online[online["alasan_ditolak"].fillna("").str.startswith("nama dipakai beberapa SKU")]
if len(on_ambigu):
    penjualan_ambigu = pd.concat([penjualan_ambigu, pd.DataFrame({
        "tanggal": on_ambigu["order_date"], "bulan": on_ambigu["order_date"].dt.strftime("%Y-%m"), "kanal": on_ambigu["kanal"],
        "teks_asli": on_ambigu["teks_sku_asli"], "qty": on_ambigu["qty"], "subtotal": on_ambigu["netto"],
        "sku_kandidat": on_ambigu["kandidat"].map(lambda k: ", ".join(sorted(k))), "alasan": on_ambigu["alasan_ditolak"],
        "berkas": on_ambigu["berkas"], "baris_excel": on_ambigu["baris_excel"]})], ignore_index=True)
print("penjualan_ambigu:", len(penjualan_ambigu), "baris | per kanal:", penjualan_ambigu["kanal"].value_counts().to_dict())

tabel_akhir = {
    "channels": channels,
    "products": products_final,
    "bundle_components": bundle_components,
    "suppliers": suppliers,
    "transactions": transactions,
    "transaction_items": transaction_items,
    "online_item_financials": online_item_financials,
    "stock_movements": stock_movements,
    "stock_opname": stock_opname,
    "marketplace_metrics": marketplace_metrics,
    "product_price_history": product_price_history,
    "data_status": data_status,
    "penjualan_ambigu": penjualan_ambigu,
    "product_mapping": product_mapping,
    "laporan_ditolak": laporan_ditolak_semua,
    "duplikat_per_berkas": duplikat_per_berkas,
    "change_log": change_log,
}
for nama, df in tabel_akhir.items():
    df.to_csv(PROCESSED / f"{nama}.csv", index=False)
pd.DataFrame([{"tabel": k, "baris": len(v), "kolom": v.shape[1]} for k, v in tabel_akhir.items()])

penjualan_ambigu: 331 baris | per kanal: {'Offline': 331}


,tabel,baris,kolom
0,channels,4,6
1,products,190,22
2,bundle_components,13,3
3,suppliers,7,4
4,transactions,35436,7
5,transaction_items,59946,11
6,online_item_financials,5053,22
7,stock_movements,66542,8
8,stock_opname,1062,10
9,marketplace_metrics,4130,15


**Cek hasil simpan.** CSV dibaca ulang seperti yang akan dilakukan notebook EDA, lalu dicek join antartabel. Nomor pesanan online harus dibaca sebagai teks (`dtype={"no_order": str}`), karena nomor TikTok Shop 18 digit rusak jika dibaca sebagai angka.

In [167]:
items = pd.read_csv(PROCESSED / "transaction_items.csv", parse_dates=["tanggal"])
kanal = pd.read_csv(PROCESSED / "channels.csv")
print(items.merge(kanal, on="kanal", how="inner").groupby("kanal").agg(baris=("item_id", "size"), revenue=("revenue", "sum")))
print("Baris item yang kanalnya tidak ada di channels:", (~items["kanal"].isin(kanal["kanal"])).sum())

metrik_csv = pd.read_csv(PROCESSED / "marketplace_metrics.csv")
produk_csv = pd.read_csv(PROCESSED / "products.csv")
print("Baris metrik yang SKU-nya tidak ada di products:", (~metrik_csv["sku"].isin(produk_csv["sku"])).sum())

online_csv = pd.read_csv(INTERIM / "online_clean.csv", dtype={"no_order": str})
print("Panjang nomor pesanan setelah dibaca ulang:", online_csv.groupby("kanal")["no_order"].apply(lambda s: sorted(s.str.len().unique().tolist())).to_dict())

             baris       revenue
kanal                           
Offline      54893  3.237620e+09
Shopee        2293  1.169944e+08
TikTok Shop   1876  9.358527e+07
Tokopedia      884  4.656427e+07
Baris item yang kanalnya tidak ada di channels: 0
Baris metrik yang SKU-nya tidak ada di products: 0
Panjang nomor pesanan setelah dibaca ulang: {'Shopee': [14], 'TikTok Shop': [18], 'Tokopedia': [26]}


## 8. Ringkasan Akhir

In [168]:
tahap_tabel = {"offline": (n_masuk_offline, offline_clean, offline_ditolak),
               "online": (n_masuk_online, online_clean, online_ditolak),
               "stok_masuk_keluar": (n_masuk_stok_mk, stok_mk_clean, stok_mk_ditolak),
               "pemecahan": (n_masuk_pemecahan, pemecahan_clean, pemecahan_ditolak),
               "rakit_paket": (n_masuk_rakit, rakit_clean, rakit_ditolak),
               "opname": (n_masuk_opname, opname_clean, opname_ditolak),
               "metrik_marketplace": (n_masuk_metrik, marketplace_metrics, metrik_ditolak)}
rekap = []
for tahap, (n_masuk, bersih, ditolak) in tahap_tabel.items():
    log = change_log[change_log["tahap"] == tahap]
    diperbaiki = log[log["kategori"] == "diperbaiki"]
    rekap.append({"tahap": tahap, "baris_masuk": n_masuk, "baris_bersih": len(bersih), "ditolak": len(ditolak),
                  "dibuang": (log["kategori"] == "dibuang").sum(),
                  "baris_diperbaiki": diperbaiki[["berkas", "baris_excel"]].drop_duplicates().shape[0],
                  "persen_bersih": round(len(bersih) / n_masuk * 100, 1)})
pd.DataFrame(rekap)

,tahap,baris_masuk,baris_bersih,ditolak,dibuang,baris_diperbaiki,persen_bersih
0,offline,55776,54893,425,458,20471,98.4
1,online,5220,5053,1,166,2108,96.8
2,stok_masuk_keluar,4234,4190,14,30,4205,99.0
3,pemecahan,708,708,0,0,233,100.0
4,rakit_paket,41,41,0,0,14,100.0
5,opname,1063,1062,1,0,0,99.9
6,metrik_marketplace,4130,4130,0,0,0,100.0


**Keterbatasan data** yang perlu diingat saat analisis (tidak menghentikan proses, tetapi memengaruhi cara membaca hasil):

In [169]:
keterbatasan = pd.DataFrame([
    {"keterbatasan": "Teks produk tidak dikenal atau ambigu (ditolak, K1/K3)",
     "jumlah": (laporan_ditolak_semua["jenis_alasan"] == "SKU tidak dikenal").sum(), "dampak": "omzet/qty sedikit lebih rendah dari aslinya"},
    {"keterbatasan": "Nama beberapa ukuran yang harganya tidak bisa membedakan (ditolak, K2)",
     "jumlah": (laporan_ditolak_semua["jenis_alasan"] == "nama dipakai beberapa SKU").sum(),
     "dampak": f"offline {total_persen:.2f}% omzet; qty AGT+AGB dan PLYBG15-50 sekitar {max(b['persen_qty_hilang'] for b in baris_banding):.0f}% lebih rendah (forecast berisiko terlalu rendah)"},
    {"keterbatasan": "Baris offline hasil mapping salah ketik yang harganya tidak sesuai SKU (ditolak)",
     "jumlah": (laporan_ditolak_semua["jenis_alasan"] == "harga tidak sesuai SKU hasil mapping salah ketik").sum(), "dampak": "sangat kecil"},
    {"keterbatasan": "Baris offline tanpa nota (ID TANPA-NOTA per baris, K11)",
     "jumlah": int(offline_clean["order_no_generated"].sum()), "dampak": "jumlah transaksi offline dan rata-rata nilai belanja tidak boleh dipakai"},
    {"keterbatasan": "Komisi platform dihitung balik (K10)",
     "jumlah": int(online_clean["komisi_dihitung_balik"].sum()), "dampak": "nilai pasti dari rumus laba kotor, tidak ada asumsi tarif"},
    {"keterbatasan": "Stok fisik berbeda dengan stok sistem saat opname (K12)",
     "jumlah": int((stock_movements["jenis"] == "ADJUSTMENT").sum()), "dampak": "stok sistem disesuaikan ke stok fisik; penyebab selisih dibahas di Q6"},
    {"keterbatasan": "Rating marketplace kosong (K16)",
     "jumlah": int(marketplace_metrics["rating"].isna().sum()), "dampak": "analisis rating hanya memakai produk yang sudah punya rating"},
    {"keterbatasan": "SKU tanpa supplier utama (K14)",
     "jumlah": int(products_final["supplier_utama"].isna().sum()), "dampak": "lead time Q5/Q8 untuk SKU ini memakai asumsi"},
])
keterbatasan

,keterbatasan,jumlah,dampak
0,"Teks produk tidak dikenal atau ambigu (ditolak, K1/K3)",88,omzet/qty sedikit lebih rendah dari aslinya
1,Nama beberapa ukuran yang harganya tidak bisa membedakan...,345,offline 0.36% omzet; qty AGT+AGB dan PLYBG15-50 sekitar ...
2,Baris offline hasil mapping salah ketik yang harganya ti...,1,sangat kecil
3,"Baris offline tanpa nota (ID TANPA-NOTA per baris, K11)",2366,jumlah transaksi offline dan rata-rata nilai belanja tid...
4,Komisi platform dihitung balik (K10),52,"nilai pasti dari rumus laba kotor, tidak ada asumsi tarif"
5,Stok fisik berbeda dengan stok sistem saat opname (K12),640,stok sistem disesuaikan ke stok fisik; penyebab selisih ...
6,Rating marketplace kosong (K16),1269,analisis rating hanya memakai produk yang sudah punya ra...
7,SKU tanpa supplier utama (K14),23,lead time Q5/Q8 untuk SKU ini memakai asumsi


**Ringkasan akhir**

1. Semua tabel **seimbang**: baris masuk = bersih + ditolak + dibuang. Total **441 baris ditolak** (0,6% dari semua baris) dan dilaporkan per berkas dan baris di `laporan_ditolak.csv`. Tidak ada SKU baru yang dibuat.
2. **Aturan pemetaan**: 911 dari 1.010 teks historis masuk mapping tersimpan, 99 ambigu. Cek "satu langkah" berlaku untuk semua teks. Harga baris hasil mapping dicek (±10%) di offline (1 dari 1.008 ditolak), online (0 dari 96), dan restok (0 dari 40 yang punya harga).
3. **Nama ambigu** (`pupuk amazing`, `polybag tanaman`) tetap ditolak dan disimpan di `penjualan_ambigu.csv`. Dampaknya **0,36% omzet offline** (tidak ada bulan ≥ 1%), tetapi qty **AGT, AGB, dan PLYBG15–50** tercatat sekitar 19–20% lebih rendah dan stok sistemnya lebih tinggi dari fisik (perlu diingat di Q6). Harga polybag terbukti tumpang tindih antar ukuran. UMKM disarankan menulis varian dan ukuran lengkap di rekap.
4. Semua validasi akhir lolos: SKU ada di master, kolom inti tidak kosong, rumus netto dan laba online tepat, panjang nomor pesanan 14/26/18, dan nama kanal baku di semua tabel.
5. Opname pertama lengkap (189 SKU) dan dipakai sebagai saldo awal. Setelah `ADJUSTMENT` di tiap opname berikutnya (640 penyesuaian), stok sistem sama dengan stok fisik di setiap tanggal opname. Tanpa penyesuaian, 21 SKU akan berstok negatif per 31 Okt 2026; dengan penyesuaian tinggal **8**. Kedelapannya mencatat barang keluar setelah opname terakhirnya yang lebih banyak daripada stok dan barang masuk yang tercatat (mis. `DLMT`, `EXTRAPCKG`). Ini perlu dicek di EDA, bukan diperbaiki di wrangling.
6. Supplier utama per kelompok produk induk mengurangi SKU tanpa supplier dari 27 menjadi **23** (5 paket dan 18 SKU yang restoknya tidak pernah mencantumkan supplier).
7. **ID `TANPA-NOTA`** menyumbang 7,6% transaksi offline tetapi hanya 4,4% omzet. Q1–Q12 tidak memakai jumlah transaksi, jadi tidak ada jawaban yang bias. Jika nanti dibutuhkan jumlah transaksi offline, angkanya harus diberi catatan.
8. Riwayat harga jual dibentuk dari transaksi yang SKU-nya pasti (bukan hasil mapping salah ketik), karena harga jual historis tidak ada di data mentah.
9. `data_complete_until` semua kanal = **31 Okt 2026**. Penjualan online mengurangi stok pada hari terakhir bulan laporan (K8), jadi stok sistem tengah bulan bisa lebih tinggi dari fisik. Analisis penjualan memakai `order_date`.

10. Semua aturan perbaikan hanya memakai isi file itu sendiri, misalnya qty yang tidak cocok dengan subtotal (104 baris) atau baris yang baru sama setelah dibersihkan (114 baris offline). Karena itu aturan yang sama tetap berlaku saat admin meng-import file baru.